# SBF-2: чистый пайплайн для NGC 1380

Компактная рабочая версия `sbf-1.ipynb`: оставлен принятый путь измерения SBF для NGC 1380, тяжёлые исследовательские диагностики и systematic scans убраны.

Входы намеренно закреплены на `data/NGC 1380/...f150w_i2d.fits` и `data/NGC 1380/...f090w_i2d.fits`.

### Метки сопоставления с Jensen

- Критические расхождения были переписаны; активных code-cell с тегом `jensen-fix` не осталось.
- **⚠️ АДАПТАЦИЯ JWST — ПРОВЕРИТЬ**: буквальное повторение HST-процедуры невозможно, но отличие требует численного контроля; тег `jensen-check`.
- **✅ ОСОЗНАННОЕ ОТЛИЧИЕ — ОСТАВЛЯЕМ**: наше заранее принятое улучшение; тег `jensen-keep`.

Метки являются аудитом текущей реализации. Они пока не меняют вычислительную логику.


In [1]:
import sys
print(sys.executable)


/Users/zuha/Desktop/FKI/4 курс 2025-2026/course_work-SBF/astro_env/bin/python3


In [2]:
import argparse, numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
import builtins
from pathlib import Path
from astropy.io import fits
from astropy.wcs import WCS
from astropy.stats import sigma_clipped_stats
from scipy.ndimage import gaussian_filter
from scipy import ndimage
from scipy.signal import fftconvolve
from astropy.convolution import Gaussian2DKernel, interpolate_replace_nans
from photutils.isophote import Ellipse, EllipseGeometry, build_ellipse_model
from photutils.segmentation import detect_sources, deblend_sources, SegmentationImage, SourceCatalog
import stpsf
from scipy.fft import fft2, fftfreq, fftshift, set_workers
from photutils.background import Background2D, MedianBackground
from astropy.stats import SigmaClip
from astropy.modeling.models import Sersic2D
from astropy.modeling.fitting import LevMarLSQFitter

_orig_print = builtins.print

def print(*args, **kwargs):
    _orig_print(f"[{time.strftime('%H:%M:%S')}]", *args, **kwargs)


**WARNING**: LOCAL JWST PRD VERSION PRDOPSSOC-072 DOESN'T MATCH THE CURRENT ONLINE VERSION PRDOPSSOC-073
Please consider updating pysiaf, e.g. pip install --upgrade pysiaf or conda update pysiaf


# Конфигурация

Эта ячейка задаёт все фиксированные входы и численные настройки для референсного запуска NGC 1380: пути к кадрам, директорию результатов, фотометрические константы, пороги масок, пределы изофот, параметры PSF/SBF, дополнительную residual-mask и цветовые проверки. Code-cell ниже должен быть единственным источником параметров пайплайна; объяснения вынесены сюда, а не в inline-комментарии.


In [3]:
f150w_path = Path("data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d.fits")
f090w_path = Path("data/NGC 1380/jw03055-o001_t001_nircam_clear-f090w_i2d.fits")

out_dir = f150w_path.parent
stem = f150w_path.stem

ARCSEC2_PER_SR = 2.350443e-11
MJY_SR_TO_JY_PER_ARCSEC2 = 2.350443e-5
AB_ZEROPOINT_JY = 3631.0

USE_ISOPHOTE = True
DUMP_ISOPHOTES = True
USE_BKG = True
DO_DEBLEND = True
DRY = False

FIXED_CENTER = None
CENTER_GUESS_DOWN = 4
CENTER_GUESS_SMOOTH_SIGMA = 3.0
CENTER_GUESS_Q = 99.5
CENTER_GUESS_MIN_PIXELS = 50
CENTER_GUESS_VALID_FLOOR = 1e-6
CENTER_GUESS_WEIGHT_FLOOR = 1e-12

BKG_BOX0 = 256
BKG_BOX2D = 256
BKG_FILTER = 5
SIGMA_STAT = 3.0
SIGMA_DET = 2.5
SIGMA_MAXIT = 5
MASK_NPIXELS = 4
SOURCE_DETECT_CONNECTIVITY = 8
DEBLEND_NLEVELS = 8
DEBLEND_CONTRAST = 0.001
DEBLEND_NPROC = 8
PREMASK_MAX_COMPACT_AREA = 3000
BKG_CHECK_CORNER_FRAC = 0.10
BKG_CHECK_MIN_PIXELS = 100

SERSIC_FIT_SAMPLE_STEP = 16
SERSIC_INIT_PERCENTILE = 95
SERSIC_INIT_REFF_DIV = 8.0
SERSIC_MIN_AMPLITUDE = 1e-6
SERSIC_MIN_REFF = 10.0
SERSIC_INIT_N = 4.0
SERSIC_INIT_ELLIP = 0.2
SERSIC_INIT_THETA = 0.0
SERSIC_REFF_BOUND_MIN = 5.0
SERSIC_N_BOUNDS = (0.5, 8.0)
SERSIC_ELLIP_BOUND_MAX = 0.9

HALF_SIZE = 3000
ISO_START_SMA = 50.0
ISO_START_EPS = 0.2
ISO_START_PA = 0.0
ISO_MAXSMA_FIT = 2000.0
ISO_MINSMA = 15.0
ISO_STEP_MAIN = 10.0
ISO_STEP_COARSE = 20.0
ISO_CHECK_SMA_RANGE = (100.0, 300.0)
ISO_FIT_USE_REAL_PIXELS = True
ISO_FIT_ALLOW_FILLED_FALLBACK = True

SBF_LIT_INNER_ARCSEC = (8.2, 16.4)
SBF_LIT_OUTER_ARCSEC = (16.4, 32.8)
SBF_MODEL_MARGIN = 1.03
MODEL_FULL_CHUNK_ROWS = 256
EXTRAP_GEOM_N = 20
EXTRAP_PROFILE_FIT_N = 20
EXTRAP_PROFILE_FIT_MIN_N = 5
MODEL_GEOM_EPS_MAX = 0.95
MODEL_FULL_EPS_CLIP_MAX = 0.90
MODEL_FULL_Q_MIN = 0.05
MODEL_FULL_BLEND_WIDTH_PX = 80.0
MODEL_FULL_BLEND_MIN_PX = 30.0
MODEL_FULL_BLEND_MAX_PX = 120.0
ISO_REAL_COVERAGE_MIN = 0.50
MODEL_SYNTH_SMA_STEP_PX = 10.0
MODEL_SYNTH_RELAX_SCALE_PX = 250.0
MODEL_CENTER_SLOPE_CLIP = 1.0e-2
MODEL_EPS_SLOPE_CLIP = 3.0e-4
MODEL_PA_SLOPE_CLIP = 3.0e-4

ROBUST_SCALE_FLOOR = 1e-12
GEOM_Q_FLOOR = 1e-3
MIN_PIXELS_ISO_CROP = 5000
MIN_PIXELS_SBF = 5000
MIN_PIXELS_SIGMA_CLIP = 100
MIN_POINTS_PK_BIN = 10
MIN_POINTS_FIT = 10
MIN_COLOR_PIXELS = 100
MIN_WORKING_ISOPHOTES = 10
MIN_ISO_CHECK_POINTS = 5
MIN_ISOPHOTES_MODEL_PROFILE = 8
MIN_ISOPHOTES_MODEL_GEOM = 5

CLIP_SIGMA_QC = 3.5
CLIP_MAXIT_QC = 5
CLIP_TAG_QC = f"{CLIP_SIGMA_QC:g}".replace(".", "p")

WIN_LEN = 8
WIN_STEP = 2
QUALITY_MAD_SCALE = 1.5
PLATEAU_MIN_LEN = 5
PLATEAU_MAX_LEN = None
DM_RANGE_MAX = 0.15
PLATEAU_ERR_WEIGHT = 0.7
PLATEAU_LEN_WEIGHT = 0.5
WINDOW_LEVEL_WEIGHT = 1.0
WINDOW_DYN_WEIGHT = 0.03
WINDOW_STD_WEIGHT = 0.03
WINDOW_MASK_WEIGHT = 0.03
PLATEAU_SPEC_MIN_SUCCESS = 3
SINGLE_WINDOW_SPEC_MIN_SUCCESS = 1

FFT_WORKERS = -1
PSF_SIZE = 129
PSF_NLAMBDA = 7
PSFREF = None
FFT_E_REALIZATIONS_MAIN = 64
FFT_E_REALIZATIONS_DIAG = 64
FFT_KBINS_N = 80
SBF_FFT_CROP_PAD_PX = PSF_SIZE
FFT_K_RANGE_MAIN = (0.04, 0.25)
SBF_REGION_K_WINDOWS = [
    (0.01, 0.25),
    (0.03, 0.25),
    FFT_K_RANGE_MAIN,
]
FIT_CORR_WARN = 0.3
FIT_P1_WARN_FACTOR = 5.0

SBF_PR_ENABLE = True
SBF_PR_MAG_BIN = 0.25
SBF_PR_MLIM_OVERRIDE = None
SBF_PR_MLIM_OFFSET = 0.0
SBF_PR_FIT_WIDTH = 1.5
SBF_PR_MIN_SOURCES_REGION = 6
SBF_PR_MIN_SOURCES_GLOBAL = 20
SBF_PR_MIN_FIT_BINS = 3
SBF_PR_GAMMA_BOUNDS = (0.10, 0.70)
SBF_PR_DEFAULT_GAMMA = 0.30
SBF_PR_FALLBACK_QUANTILE = 0.80
SBF_PR_CATALOG_MIN_FLUX_JY = 0.0
SBF_PR_MAX_CORRECTION_WARN = 0.50
SBF_PR_DET_SIGMA = SIGMA_DET
SBF_PR_DET_NPIXELS = MASK_NPIXELS
SBF_PR_MAX_COMPACT_AREA = PREMASK_MAX_COMPACT_AREA
SBF_PR_MASK_DILATE_ITERATIONS = 3
SBF_PR_DO_DEBLEND = DO_DEBLEND
SBF_PR_SOURCE_IMAGE = "img_minus_model_full_unmasked"

COLOR_CLIP_SIGMA = 3.0
COLOR_CLIP_MAXIT = 5
# Заполнить из принятой карты Galactic extinction; None запрещает молчаливую коррекцию.
A_F090W = None
A_F150W = None

UNROLL_N_BINS = 360
UNROLL_N_SUBANNULI = 6
UNROLL_PAD_EXTRA = 3
UNROLL_MIN_PIXELS = 10
UNROLL_SPLINE_POINTS = 1000
UNROLL_SPLINE_SMOOTH_BINS = 21

FFT_RNG_SEED = 1489


## Загрузка JWST-кадров

Загружаем F150W science frame для SBF и, если файл доступен, F090W frame для цветовых проверок. Здесь же читаются WCS и фотометрические метаданные, которые дальше нужны для pixel scale и AB-калибровки.

> **⚠️ АДАПТАЦИЯ JWST — ПРОВЕРИТЬ КОРРЕЛИРОВАННЫЙ ШУМ.** Jensen складывает отдельные HST `flt`-экспозиции целыми сдвигами и избегает ресэмплинга. Здесь используются готовые JWST `i2d`, где интерполяция может сделать $P_1(k)$ неплоским. До статьи нужны спектр пустого неба и проверка устойчивости результата к диапазону $k$.


In [4]:
print("loading i2d files...")

with fits.open(f150w_path, memmap=False) as hdul:
    img_f150 = hdul["SCI"].data.astype(float)
    hdr150 = hdul["SCI"].header
    valid150 = np.isfinite(img_f150)

    if "WHT" in hdul:
        try:
            wht150 = hdul["WHT"].data
            valid150 &= np.isfinite(wht150) & (wht150 > 0)
        except Exception:
            pass

wcs150 = WCS(hdr150)
pixar_sr = float(hdr150["PIXAR_SR"])
pix_area = pixar_sr / ARCSEC2_PER_SR
print(f"[WCS] PIXAR_SR={pixar_sr:.8e} sr → pix_area={pix_area:.8e} arcsec^2")

img_f090 = None
hdr090 = None
valid090 = None
wcs090 = None

if f090w_path.exists():
    with fits.open(f090w_path, memmap=False) as hdul:
        img_f090 = hdul["SCI"].data.astype(float)
        hdr090 = hdul["SCI"].header
        valid090 = np.isfinite(img_f090)

        if "WHT" in hdul:
            try:
                wht090 = hdul["WHT"].data
                valid090 &= np.isfinite(wht090) & (wht090 > 0)
            except Exception:
                pass

    wcs090 = WCS(hdr090)

print(f"F150W shape = {img_f150.shape}, valid = {valid150.sum()}")
if img_f090 is not None:
    print(f"F090W shape = {img_f090.shape}, valid = {valid090.sum()}")


[14:33:34] loading i2d files...
[14:33:35] [WCS] PIXAR_SR=2.29190411e-14 sr → pix_area=9.75094527e-04 arcsec^2


Set DATE-AVG to '2023-11-09T21:33:24.076' from MJD-AVG.
Set DATE-END to '2023-11-09T21:52:00.860' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    23.109100 from OBSGEO-[XYZ].
Set OBSGEO-H to 1523626142.965 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2023-11-09T19:55:21.653' from MJD-AVG.
Set DATE-END to '2023-11-09T21:10:19.167' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    23.148810 from OBSGEO-[XYZ].
Set OBSGEO-H to 1522915295.335 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


[14:33:35] F150W shape = (4377, 9876), valid = 33055333
[14:33:35] F090W shape = (4376, 9875), valid = 33051148


## Быстрая оценка центра

Определяем грубый центр галактики по валидной области кадра и ярким пикселям. Этот центр нужен как начальное приближение для Sérsic и изофот, но не является отдельной финальной геометрией SBF.


In [5]:
def guess_center_fast(img, valid, down=None, sigma=None, q=None, min_sel_pixels=None, wcs=None, log=True):
    if down is None:
        down = CENTER_GUESS_DOWN
    if sigma is None:
        sigma = CENTER_GUESS_SMOOTH_SIGMA
    if q is None:
        q = CENTER_GUESS_Q
    if min_sel_pixels is None:
        min_sel_pixels = CENTER_GUESS_MIN_PIXELS

    down = int(down)
    sigma = float(sigma)
    q = float(q)
    min_sel_pixels = int(min_sel_pixels)

    ny, nx = img.shape

    def _log_center(xc, yc, note=""):
        if not log:
            return
        msg = f"[CENTER-FAST] x={xc:.2f}, y={yc:.2f}"
        if note:
            msg += f" ({note})"
        if wcs is not None:
            ra_deg, dec_deg = wcs.pixel_to_world_values(xc, yc)
            msg += f" | RA={ra_deg:.8f} deg, Dec={dec_deg:.8f} deg"
        print(msg)

    img_d = img[::down, ::down]
    val_d = valid[::down, ::down] & np.isfinite(img_d)

    if not np.any(val_d):
        xc, yc = nx / 2.0, ny / 2.0
        _log_center(xc, yc, note="fallback: no valid downsampled pixels")
        return xc, yc

    data = np.where(val_d, img_d, 0.0).astype(np.float32)
    w = val_d.astype(np.float32)

    num = gaussian_filter(data, sigma=sigma)
    den = gaussian_filter(w, sigma=sigma)
    sm = np.divide(
        num,
        den,
        out=np.full_like(num, np.nan),
        where=den > CENTER_GUESS_VALID_FLOOR,
    )

    if not np.isfinite(sm).any():
        xc, yc = nx / 2.0, ny / 2.0
        _log_center(xc, yc, note="fallback: sm is all-NaN")
        return xc, yc

    thr = np.nanpercentile(sm, q)
    sel = np.isfinite(sm) & (sm >= thr)

    if sel.sum() < min_sel_pixels:
        y, x = np.unravel_index(np.nanargmax(sm), sm.shape)
        xc, yc = float(x * down), float(y * down)
        _log_center(xc, yc, note="fallback: argmax")
        return xc, yc

    ys, xs = np.nonzero(sel)
    ws = sm[sel] - np.nanmin(sm[sel])
    ws = np.nan_to_num(ws, nan=0.0) + CENTER_GUESS_WEIGHT_FLOOR

    x0 = float((xs * ws).sum() / ws.sum()) * down
    y0 = float((ys * ws).sum() / ws.sum()) * down
    _log_center(x0, y0, note=f"down={down}, sigma={sigma}, q={q}")
    return x0, y0


## Подготовка кадра

Готовим science image к моделированию: вычитаем крупномасштабный фон, проверяем остаточный фон и строим первичную маску компактных объектов перед фитированием света галактики.


## Вычитание фона

Оцениваем и вычитаем **один скалярный уровень неба** тремя способами, близкими к процедуре Jensen: по углам кадра, по закону $r^{1/4}$ и по остаточному фону после грубой гладкой модели. Принятым уровнем служит медиана независимых оценок, а их максимальное расхождение с принятой величиной сохраняется как систематика фона.

> **✅ ПЕРЕПИСАНО В ЛОГИКЕ JENSEN.** Двумерная карта `Background2D` больше не вычитается из science image: она используется только внутри независимой оценки остаточного скалярного уровня на внешних участках. Это не позволяет алгоритму принять протяжённый свет галактики за пространственно меняющийся фон.


In [6]:
print("estimating and subtracting Jensen-like scalar background...")

img0 = np.array(img_f150, copy=True)

# Все оценки выполняются на разреженной копии: это не меняет science image,
# но резко уменьшает память и время для большого NIRCam mosaic.
BKG_JENSEN_DOWNSAMPLE = 8
BKG_JENSEN_CORNER_FRAC = 0.10
BKG_JENSEN_OUTER_QUANTILE = 0.80
BKG_JENSEN_PROFILE_BINS = 64
BKG_JENSEN_SKY_GRID = 500
BKG_JENSEN_MODEL_ITERS = 3

def _background_clipped_stats(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size < BKG_CHECK_MIN_PIXELS:
        return np.nan, np.nan, np.nan, int(values.size)
    mean_value, median_value, std_value = sigma_clipped_stats(
        values,
        sigma=SIGMA_STAT,
        maxiters=SIGMA_MAXIT,
    )
    return float(mean_value), float(median_value), float(std_value), int(values.size)

if USE_BKG:
    # Для оценки радиального профиля сначала принимаем каталоговый центр.
    # Если координаты цели недоступны, используем уже определённый быстрый поиск центра.
    try:
        primary_header = fits.getheader(f150w_path, 0)
        target_ra = float(primary_header["TARG_RA"])
        target_dec = float(primary_header["TARG_DEC"])
        center_xy = wcs150.celestial.all_world2pix([[target_ra, target_dec]], 0)[0]
        x_bg_center, y_bg_center = map(float, center_xy)
        if not (np.isfinite(x_bg_center) and np.isfinite(y_bg_center)):
            raise ValueError("catalog center is not finite")
        bg_center_source = "TARG_RA/TARG_DEC"
    except Exception as error:
        x_bg_center, y_bg_center = guess_center_fast(
            img0,
            valid150,
            down=max(CENTER_GUESS_DOWN, BKG_JENSEN_DOWNSAMPLE),
            sigma=CENTER_GUESS_SMOOTH_SIGMA,
            q=CENTER_GUESS_Q,
            min_sel_pixels=CENTER_GUESS_MIN_PIXELS,
            wcs=None,
            log=False,
        )
        bg_center_source = f"automatic fallback: {error}"

    step_bg = int(BKG_JENSEN_DOWNSAMPLE)
    img_bg = np.asarray(img0[::step_bg, ::step_bg], dtype=float)
    valid_bg = np.asarray(valid150[::step_bg, ::step_bg], dtype=bool) & np.isfinite(img_bg)
    yy_bg, xx_bg = np.indices(img_bg.shape, dtype=float)
    radius_bg = np.hypot(
        xx_bg * step_bg - x_bg_center,
        yy_bg * step_bg - y_bg_center,
    )

    print(
        f"[BKG] profile center=({x_bg_center:.2f}, {y_bg_center:.2f}) "
        f"from {bg_center_source}; downsample={step_bg}"
    )

    # 1. Jensen-like оценка по четырём углам кадра.
    ny_bg, nx_bg = img_bg.shape
    corner_ny = max(1, int(ny_bg * BKG_JENSEN_CORNER_FRAC))
    corner_nx = max(1, int(nx_bg * BKG_JENSEN_CORNER_FRAC))
    corner_values = np.concatenate([
        img_bg[:corner_ny, :corner_nx][valid_bg[:corner_ny, :corner_nx]],
        img_bg[:corner_ny, -corner_nx:][valid_bg[:corner_ny, -corner_nx:]],
        img_bg[-corner_ny:, :corner_nx][valid_bg[-corner_ny:, :corner_nx]],
        img_bg[-corner_ny:, -corner_nx:][valid_bg[-corner_ny:, -corner_nx:]],
    ])
    corner_mean, sky_corners, corner_std, corner_count = _background_clipped_stats(corner_values)

    # Внешние пиксели нужны как опора для диапазона поиска и гладкой residual-оценки.
    outer_radius_cut = float(np.nanquantile(radius_bg[valid_bg], BKG_JENSEN_OUTER_QUANTILE))
    outer_mask = valid_bg & (radius_bg >= outer_radius_cut)
    outer_mean, sky_outer, outer_std, outer_count = _background_clipped_stats(img_bg[outer_mask])

    # 2. Jensen-like оценка из линейности log(I - sky) относительно r^(1/4).
    radius_max = float(np.nanpercentile(radius_bg[valid_bg], 98.0))
    radius_min_profile = max(300.0, 0.06 * radius_max)
    radius_max_profile = 0.90 * radius_max
    radial_edges = np.linspace(
        radius_min_profile,
        radius_max_profile,
        BKG_JENSEN_PROFILE_BINS + 1,
    )

    profile_radius = []
    profile_intensity = []
    profile_scatter = []
    profile_count = []
    for radius_lo, radius_hi in zip(radial_edges[:-1], radial_edges[1:]):
        annulus = valid_bg & (radius_bg >= radius_lo) & (radius_bg < radius_hi)
        if int(annulus.sum()) < BKG_CHECK_MIN_PIXELS:
            continue
        ann_mean, ann_median, ann_std, ann_count = _background_clipped_stats(img_bg[annulus])
        if np.isfinite(ann_median):
            profile_radius.append(0.5 * (radius_lo + radius_hi))
            profile_intensity.append(ann_median)
            profile_scatter.append(ann_std)
            profile_count.append(ann_count)

    profile_radius = np.asarray(profile_radius, dtype=float)
    profile_intensity = np.asarray(profile_intensity, dtype=float)
    profile_scatter = np.asarray(profile_scatter, dtype=float)
    profile_count = np.asarray(profile_count, dtype=int)

    sky_r14 = np.nan
    r14_log_scatter = np.nan
    r14_slope = np.nan
    r14_intercept = np.nan
    if profile_radius.size >= 12 and np.isfinite(sky_outer) and np.isfinite(outer_std):
        positive_scale = max(abs(sky_outer), 1.0)
        epsilon_sky = max(1.0e-8, positive_scale * 1.0e-6)
        sky_high = float(np.nanmin(profile_intensity) - epsilon_sky)
        data_low = float(np.nanpercentile(img_bg[valid_bg], 0.5))
        sky_low = max(data_low, float(sky_outer - 8.0 * max(outer_std, epsilon_sky)))
        if not np.isfinite(sky_low) or sky_low >= sky_high:
            sky_low = sky_high - max(abs(sky_high) * 0.5, 10.0 * epsilon_sky)

        r_quarter = profile_radius ** 0.25
        best_r14 = None
        for sky_try in np.linspace(sky_low, sky_high, BKG_JENSEN_SKY_GRID):
            galaxy_profile = profile_intensity - sky_try
            if np.any(galaxy_profile <= 0.0):
                continue
            log_profile = np.log(galaxy_profile)
            coefficients = np.polyfit(r_quarter, log_profile, 1)
            log_residual = log_profile - np.polyval(coefficients, r_quarter)
            log_scatter = float(
                1.4826 * np.nanmedian(np.abs(log_residual - np.nanmedian(log_residual)))
            )
            if best_r14 is None or log_scatter < best_r14[0]:
                best_r14 = (log_scatter, float(sky_try), coefficients)

        if best_r14 is not None:
            r14_log_scatter, sky_r14, r14_coefficients = best_r14
            r14_slope = float(r14_coefficients[0])
            r14_intercept = float(r14_coefficients[1])

    # 3. Итеративная оценка остатка после грубой гладкой модели внешнего поля.
    seed_values = [value for value in (sky_corners, sky_r14, sky_outer) if np.isfinite(value)]
    if not seed_values:
        raise RuntimeError("[BKG] all Jensen-like background estimates failed")
    sky_model_residual = float(np.nanmedian(seed_values))
    model_outer_radius = max(
        1.5 * SBF_LIT_OUTER_ARCSEC[1] / np.sqrt(pix_area),
        0.45 * radius_max,
    )

    model_iteration_log = []
    for iteration in range(BKG_JENSEN_MODEL_ITERS):
        residual_bg = img_bg - sky_model_residual
        fit_outer = valid_bg & (radius_bg >= model_outer_radius)
        res_mean, res_median, res_std, res_count = _background_clipped_stats(residual_bg[fit_outer])
        if not (np.isfinite(res_median) and np.isfinite(res_std)):
            break

        source_like = np.abs(residual_bg - res_median) > SIGMA_STAT * max(res_std, ROBUST_SCALE_FLOOR)
        background_mask = (~valid_bg) | (radius_bg < model_outer_radius) | source_like
        try:
            residual_map = Background2D(
                residual_bg,
                box_size=max(8, BKG_BOX2D // step_bg),
                filter_size=(3, 3),
                sigma_clip=SigmaClip(sigma=SIGMA_STAT, maxiters=SIGMA_MAXIT),
                bkg_estimator=MedianBackground(),
                mask=background_mask,
                exclude_percentile=95.0,
            )
            correction_values = residual_map.background[fit_outer & (~source_like)]
            _, correction, correction_std, correction_count = _background_clipped_stats(correction_values)
        except Exception as error:
            correction = res_median
            correction_std = res_std
            correction_count = res_count
            print(f"[BKG][WARN] smooth residual map failed at iteration {iteration + 1}: {error}")

        if not np.isfinite(correction):
            break
        sky_model_residual += float(correction)
        model_iteration_log.append({
            "iteration": iteration + 1,
            "correction": float(correction),
            "sky_after": float(sky_model_residual),
            "scatter": float(correction_std),
            "n_pixels": int(correction_count),
        })
        if abs(correction) <= max(1.0e-8, 1.0e-4 * max(abs(sky_model_residual), 1.0)):
            break

    background_rows = [
        {
            "method": "four_corners",
            "sky": sky_corners,
            "scatter": corner_std,
            "n_pixels": corner_count,
            "used_for_adopted": True,
        },
        {
            "method": "r_quarter_profile",
            "sky": sky_r14,
            "scatter": r14_log_scatter,
            "n_pixels": int(profile_count.sum()) if profile_count.size else 0,
            "used_for_adopted": np.isfinite(sky_r14),
        },
        {
            "method": "iterative_smooth_residual",
            "sky": sky_model_residual,
            "scatter": model_iteration_log[-1]["scatter"] if model_iteration_log else outer_std,
            "n_pixels": model_iteration_log[-1]["n_pixels"] if model_iteration_log else outer_count,
            "used_for_adopted": True,
        },
        {
            "method": "outer_pixels_diagnostic",
            "sky": sky_outer,
            "scatter": outer_std,
            "n_pixels": outer_count,
            "used_for_adopted": False,
        },
    ]
    df_background_estimates = pd.DataFrame(background_rows)

    adopted_values = df_background_estimates.loc[
        df_background_estimates["used_for_adopted"] & np.isfinite(df_background_estimates["sky"]),
        "sky",
    ].to_numpy(dtype=float)
    if adopted_values.size < 2:
        raise RuntimeError(f"[BKG] fewer than two usable background estimates: {adopted_values}")

    bg_scalar = float(np.nanmedian(adopted_values))
    bg_uncertainty = float(np.nanmax(np.abs(adopted_values - bg_scalar)))
    df_background_estimates["delta_from_adopted"] = df_background_estimates["sky"] - bg_scalar

    # Science image получает только скалярное вычитание. Никакая 2D background map
    # не вычитается, чтобы не удалить протяжённый свет галактики.
    bg_map = np.full(img0.shape, bg_scalar, dtype=np.float32)
    img = img0 - bg_scalar
    background_method = "Jensen-like median: corners + r^(1/4) + iterative smooth residual"

    print("[BKG] independent estimates:")
    print(df_background_estimates.to_string(index=False))
    print(
        f"[BKG] adopted scalar={bg_scalar:.8g} MJy/sr; "
        f"systematic={bg_uncertainty:.3g} MJy/sr; method={background_method}"
    )
    print(
        f"[BKG] r^(1/4) fit: slope={r14_slope:.6g}, intercept={r14_intercept:.6g}, "
        f"robust log-scatter={r14_log_scatter:.6g}"
    )

else:
    bg_scalar = 0.0
    bg_uncertainty = 0.0
    bg_map = np.zeros_like(img0, dtype=np.float32)
    img = img0.copy()
    background_method = "disabled"
    df_background_estimates = pd.DataFrame([
        {
            "method": "disabled",
            "sky": 0.0,
            "scatter": 0.0,
            "n_pixels": 0,
            "used_for_adopted": False,
            "delta_from_adopted": 0.0,
        }
    ])
    print("[BKG] skipped")


[14:33:35] estimating and subtracting Jensen-like scalar background...
[14:33:35] [BKG] profile center=(6616.07, 1140.53) from TARG_RA/TARG_DEC; downsample=8
[14:33:35] [BKG] independent estimates:
[14:33:35]                    method      sky  scatter  n_pixels  used_for_adopted  delta_from_adopted
             four_corners 0.207482 0.046663     20088              True            0.000000
        r_quarter_profile 0.174969 0.083540    455238              True           -0.032514
iterative_smooth_residual 0.218459 0.041993    267741              True            0.010977
  outer_pixels_diagnostic 0.196518 0.012880    103336             False           -0.010965
[14:33:35] [BKG] adopted scalar=0.20748239 MJy/sr; systematic=0.0325 MJy/sr; method=Jensen-like median: corners + r^(1/4) + iterative smooth residual
[14:33:35] [BKG] r^(1/4) fit: slope=-1.58569, intercept=10.0099, robust log-scatter=0.0835403


## Проверка фона

Измеряем остаточный фон после вычитания во внешней валидной области. Это быстрая проверка, которая должна поймать грубую ошибку sky-level до дорогих ячеек модели.


In [7]:
print("checking residual background after subtraction...")

check = np.array(img, copy=True)
check[~valid150] = np.nan

ny, nx = check.shape
dx = max(1, int(nx * BKG_CHECK_CORNER_FRAC))
dy = max(1, int(ny * BKG_CHECK_CORNER_FRAC))

corners = np.concatenate([
    check[:dy, :dx].ravel(),
    check[:dy, -dx:].ravel(),
    check[-dy:, :dx].ravel(),
    check[-dy:, -dx:].ravel(),
])

corners = corners[np.isfinite(corners)]

if corners.size > BKG_CHECK_MIN_PIXELS:
    mean_c, med_c, std_c = sigma_clipped_stats(corners, sigma=SIGMA_STAT, maxiters=SIGMA_MAXIT)
    print(
        f"[BKG-CHECK] corners: "
        f"med={med_c:.3e}, mean={mean_c:.3e}, std={std_c:.3e}, N={corners.size}"
    )
else:
    print("[BKG-CHECK] too few valid corner pixels")


[14:33:35] checking residual background after subtraction...
[14:33:35] [BKG-CHECK] corners: med=7.780e-04, mean=2.159e-02, std=4.926e-02, N=1322679


## Первичная маска компактных источников

Сначала строим **предварительную** гладкую модель галактики, затем ищем компактные источники на остатках. Порог задаётся не одной глобальной дисперсией, а эмпирической моделью шума как функцией локальной яркости галактики: это позволяет включить собственные SBF в noise model и не принимать их за шаровые скопления.

> **✅ ПЕРЕПИСАНО В ПОРЯДКЕ JENSEN:** preliminary galaxy model → model-subtracted residual → galaxy-dependent noise model → source detection → deblending → compact-source mask. Отдельная маска пыли и нерегулярных структур всё ещё потребуется позднее: этот блок отвечает только за положительные компактные источники.


In [8]:
print("building Jensen-like primary compact-source mask...")

PREMASK_MODEL_BOX = 128
PREMASK_MODEL_FILTER = 5
PREMASK_NOISE_SAMPLE_STEP = 8
PREMASK_NOISE_BINS = 20
PREMASK_DET_SIGMA = max(3.5, float(SIGMA_DET))
PREMASK_DET_NPIXELS = max(9, int(MASK_NPIXELS))
PREMASK_DILATE_ITERATIONS = 2
PREMASK_RADIUS_MARGIN = 1.15

use_det = valid150 & np.isfinite(img)
if int(use_det.sum()) < MIN_PIXELS_SBF:
    raise RuntimeError(f"[PREMASK] too few valid pixels: N={int(use_det.sum())}")

# Центр берём из уже принятой фоновой ячейки; при автономном запуске используем
# тот же безопасный автоматический fallback.
if "x_bg_center" in globals() and "y_bg_center" in globals():
    x_premask_center = float(x_bg_center)
    y_premask_center = float(y_bg_center)
    premask_center_source = "background/TARG center"
else:
    x_premask_center, y_premask_center = guess_center_fast(
        img,
        use_det,
        down=max(CENTER_GUESS_DOWN, PREMASK_NOISE_SAMPLE_STEP),
        sigma=CENTER_GUESS_SMOOTH_SIGMA,
        q=CENTER_GUESS_Q,
        min_sel_pixels=CENTER_GUESS_MIN_PIXELS,
        wcs=None,
        log=False,
    )
    premask_center_source = "automatic fallback"

rout_sbf_pixels = SBF_LIT_OUTER_ARCSEC[1] / np.sqrt(pix_area)
premask_work_radius = PREMASK_RADIUS_MARGIN * max(float(ISO_MAXSMA_FIT), float(rout_sbf_pixels))
yy_premask, xx_premask = np.ogrid[:img.shape[0], :img.shape[1]]
premask_work_region = (
    use_det
    & ((xx_premask - x_premask_center) ** 2 + (yy_premask - y_premask_center) ** 2 <= premask_work_radius ** 2)
)

print(
    f"[PREMASK] center=({x_premask_center:.2f}, {y_premask_center:.2f}) "
    f"from {premask_center_source}; work radius={premask_work_radius:.1f} px; "
    f"Nwork={int(premask_work_region.sum())}"
)

# 1. Предварительная гладкая модель. Это только detection model, а не финальная
# изофотная модель, используемая для измерения SBF.
preliminary_input = np.asarray(img, dtype=np.float32).copy()
preliminary_input[~use_det] = np.nan
preliminary_background = Background2D(
    preliminary_input,
    box_size=PREMASK_MODEL_BOX,
    filter_size=(PREMASK_MODEL_FILTER, PREMASK_MODEL_FILTER),
    sigma_clip=SigmaClip(sigma=SIGMA_STAT, maxiters=SIGMA_MAXIT),
    bkg_estimator=MedianBackground(),
    mask=~use_det,
    exclude_percentile=90.0,
)
premask_preliminary_model = np.clip(
    np.asarray(preliminary_background.background, dtype=np.float32),
    0.0,
    None,
)
premask_preliminary_residual = np.asarray(img - premask_preliminary_model, dtype=np.float32)
premask_preliminary_residual[~use_det] = np.nan

# 2. Эмпирическая noise model. В каждом диапазоне яркости предварительной модели
# измеряется sigma остатков; тем самым в порог входят photon noise и сами SBF.
sample_step = int(PREMASK_NOISE_SAMPLE_STEP)
sample_region = premask_work_region[::sample_step, ::sample_step]
sample_model = premask_preliminary_model[::sample_step, ::sample_step][sample_region]
sample_residual = premask_preliminary_residual[::sample_step, ::sample_step][sample_region]

finite_sample = np.isfinite(sample_model) & np.isfinite(sample_residual)
sample_model = sample_model[finite_sample]
sample_residual = sample_residual[finite_sample]
if sample_model.size < MIN_PIXELS_SIGMA_CLIP:
    raise RuntimeError(f"[PREMASK] too few pixels for the noise model: N={sample_model.size}")

model_quantile_edges = np.unique(
    np.nanquantile(sample_model, np.linspace(0.0, 1.0, PREMASK_NOISE_BINS + 1))
)
noise_rows = []
for model_lo, model_hi in zip(model_quantile_edges[:-1], model_quantile_edges[1:]):
    if model_hi == model_quantile_edges[-1]:
        in_bin = (sample_model >= model_lo) & (sample_model <= model_hi)
    else:
        in_bin = (sample_model >= model_lo) & (sample_model < model_hi)
    if int(in_bin.sum()) < BKG_CHECK_MIN_PIXELS:
        continue
    bin_mean, bin_median, bin_sigma = sigma_clipped_stats(
        sample_residual[in_bin],
        sigma=SIGMA_STAT,
        maxiters=SIGMA_MAXIT,
    )
    if np.isfinite(bin_sigma) and bin_sigma > 0.0:
        noise_rows.append({
            "model_intensity": float(np.nanmedian(sample_model[in_bin])),
            "residual_median": float(bin_median),
            "residual_sigma": float(bin_sigma),
            "n_pixels": int(in_bin.sum()),
        })

df_premask_noise_model = pd.DataFrame(noise_rows).sort_values("model_intensity").reset_index(drop=True)
if len(df_premask_noise_model) < 5:
    raise RuntimeError(f"[PREMASK] noise model has too few valid bins: {len(df_premask_noise_model)}")

noise_model_x = df_premask_noise_model["model_intensity"].to_numpy(dtype=float)
noise_model_sigma = df_premask_noise_model["residual_sigma"].to_numpy(dtype=float)
# Реальная sigma не должна уменьшаться к более яркому свету галактики.
noise_model_sigma = np.maximum.accumulate(noise_model_sigma)
df_premask_noise_model["residual_sigma_monotonic"] = noise_model_sigma

premask_noise_sigma = np.interp(
    premask_preliminary_model,
    noise_model_x,
    noise_model_sigma,
    left=noise_model_sigma[0],
    right=noise_model_sigma[-1],
).astype(np.float32)
premask_noise_sigma = np.maximum(premask_noise_sigma, ROBUST_SCALE_FLOOR)

# 3. Детектирование проводится только на model-subtracted residual внутри области,
# покрывающей финальный изофотный fit и оба кольца Jensen.
detect_image = np.zeros_like(premask_preliminary_residual, dtype=np.float32)
detect_image[premask_work_region] = premask_preliminary_residual[premask_work_region]
threshold_map = np.full_like(premask_noise_sigma, np.inf, dtype=np.float32)
threshold_map[premask_work_region] = (
    PREMASK_DET_SIGMA * premask_noise_sigma[premask_work_region]
)

segm = detect_sources(
    detect_image,
    threshold=threshold_map,
    npixels=PREMASK_DET_NPIXELS,
    connectivity=SOURCE_DETECT_CONNECTIVITY,
)
n_detected_raw = int(segm.nlabels) if segm is not None else 0
print(
    f"[PREMASK] detected raw={n_detected_raw}; threshold={PREMASK_DET_SIGMA:.2f} sigma; "
    f"npixels={PREMASK_DET_NPIXELS}"
)

compact_labels = []
premask_segm = segm

if segm is None:
    premask_src_raw = np.zeros_like(img, dtype=bool)
else:
    if DO_DEBLEND:
        print("[PREMASK] deblending residual sources...")
        try:
            segm = deblend_sources(
                detect_image,
                segm,
                npixels=PREMASK_DET_NPIXELS,
                nlevels=DEBLEND_NLEVELS,
                contrast=DEBLEND_CONTRAST,
                nproc=DEBLEND_NPROC,
                progress_bar=False,
            )
        except Exception as parallel_error:
            if DEBLEND_NPROC == 1:
                raise
            print(
                f"[PREMASK][WARN] parallel deblend failed: {parallel_error}; "
                "retrying sequentially"
            )
            segm = deblend_sources(
                detect_image,
                segm,
                npixels=PREMASK_DET_NPIXELS,
                nlevels=DEBLEND_NLEVELS,
                contrast=DEBLEND_CONTRAST,
                nproc=1,
                progress_bar=False,
            )

    premask_segm = segm
    labels = segm.labels
    counts = np.bincount(segm.data.ravel(), minlength=int(labels.max()) + 1)
    counts[0] = 0
    compact_labels = [
        int(label)
        for label in labels
        if 0 < counts[label] <= PREMASK_MAX_COMPACT_AREA
    ]

    label_lookup = np.zeros(int(labels.max()) + 1, dtype=bool)
    if compact_labels:
        label_lookup[np.asarray(compact_labels, dtype=int)] = True
    premask_src_raw = label_lookup[segm.data]

premask_src = ndimage.binary_dilation(
    premask_src_raw,
    iterations=PREMASK_DILATE_ITERATIONS,
) & premask_work_region
premask_compact_labels = np.asarray(compact_labels, dtype=int)
premask = (~valid150) | premask_src

premask_mask_path = out_dir / f"{stem}_sbf_premask_initial_compact_mask.fits"
premask_masked_image_path = out_dir / f"{stem}_sbf_premask_initial_masked_image.fits"
premask_noise_table_path = out_dir / f"{stem}_sbf_premask_noise_model.csv"

premask_masked_image = np.asarray(img, dtype=np.float32).copy()
premask_masked_image[premask] = np.nan
fits.writeto(premask_mask_path, np.asarray(premask_src, dtype=np.uint8), hdr150, overwrite=True)
fits.writeto(premask_masked_image_path, premask_masked_image, hdr150, overwrite=True)
df_premask_noise_model.to_csv(premask_noise_table_path, index=False)

n_deblended = int(segm.nlabels) if segm is not None else 0
n_mask_raw = int(premask_src_raw.sum())
n_mask_dilated = int(premask_src.sum())
work_pixels = int(premask_work_region.sum())
print(
    f"[PREMASK] deblended={n_deblended}; compact kept={len(premask_compact_labels)}; "
    f"raw mask={n_mask_raw} px; dilated mask={n_mask_dilated} px"
)
print(
    f"[PREMASK] compact-mask coverage in work region="
    f"{100.0 * n_mask_dilated / max(work_pixels, 1):.3f}%"
)
print(f"[OUT] initial compact mask   -> {premask_mask_path}")
print(f"[OUT] initial masked image  -> {premask_masked_image_path}")
print(f"[OUT] primary noise model   -> {premask_noise_table_path}")


[14:33:35] building Jensen-like primary compact-source mask...
[14:33:36] [PREMASK] center=(6616.07, 1140.53) from background/TARG center; work radius=2300.0 px; Nwork=8817809
[14:33:40] [PREMASK] detected raw=3300; threshold=3.50 sigma; npixels=9
[14:33:40] [PREMASK] deblending residual sources...
[14:33:43] [PREMASK] deblended=3375; compact kept=3371; raw mask=74255 px; dilated mask=201636 px
[14:33:43] [PREMASK] compact-mask coverage in work region=2.287%
[14:33:43] [OUT] initial compact mask   -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_premask_initial_compact_mask.fits
[14:33:43] [OUT] initial masked image  -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_premask_initial_masked_image.fits
[14:33:43] [OUT] primary noise model   -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_premask_noise_model.csv


## Принятый центр галактики

Выбираем рабочий центр для инициализации гладкой модели. Используется быстрая оценка, если в конфиге явно не задан ручной центр.


In [9]:
print("choosing galaxy center...")

if FIXED_CENTER is None:
    x0_center, y0_center = guess_center_fast(
        img,
        valid150 & (~premask),
        down=CENTER_GUESS_DOWN,
        sigma=CENTER_GUESS_SMOOTH_SIGMA,
        q=CENTER_GUESS_Q,
        min_sel_pixels=CENTER_GUESS_MIN_PIXELS,
        wcs=wcs150,
        log=True,
    )
    center_src = "auto-fast"
else:
    x0_center, y0_center = FIXED_CENTER
    ra_deg, dec_deg = wcs150.pixel_to_world_values(x0_center, y0_center)
    print(f"[CENTER-FIXED] x={x0_center:.2f}, y={y0_center:.2f} | RA={ra_deg:.8f} deg, Dec={dec_deg:.8f} deg")
    center_src = "fixed"

print(f"[CENTER] using ({x0_center:.2f}, {y0_center:.2f}) [{center_src}]")


[14:33:43] choosing galaxy center...
[14:33:43] [CENTER-FAST] x=6617.63, y=1167.29 (down=4, sigma=3.0, q=99.5) | RA=54.11488223 deg, Dec=-34.97607597 deg
[14:33:43] [CENTER] using (6617.63, 1167.29) [auto-fast]


## Sérsic fallback-модель

Фитируем простую гладкую Sérsic-компоненту по замаскированному кадру. В текущем science path это в основном fallback/fill для невалидных пикселей, а не финальная модель для вычитания SBF.

> **⚠️ АДАПТАЦИЯ JWST — КОНТРОЛИРОВАТЬ ДОЛЮ СИНТЕТИЧЕСКИХ ДАННЫХ.** У Jensen не требовалось заполнять широкие щели между детекторами. Sérsic здесь допустим как численная опора для прохождения изофот, но заполненные пиксели не должны попадать в FFT или цветовую фотометрию, а доля их влияния на каждую изофоту должна сохраняться.


In [10]:
yy_full, xx_full = np.indices(img.shape, dtype=float)

use_fit = (~premask) & np.isfinite(img)

sample_step = SERSIC_FIT_SAMPLE_STEP

y_fit = yy_full[use_fit][::sample_step]
x_fit = xx_full[use_fit][::sample_step]
z_fit = img[use_fit][::sample_step]

print(f"[SERSIC] fit sample size = {z_fit.size}")

amp0 = float(np.nanpercentile(z_fit, SERSIC_INIT_PERCENTILE))

r_eff0 = float(min(img.shape) / SERSIC_INIT_REFF_DIV)

sersic_init = Sersic2D(

    amplitude=max(amp0, SERSIC_MIN_AMPLITUDE),

    r_eff=max(r_eff0, SERSIC_MIN_REFF),

    n=SERSIC_INIT_N,

    x_0=float(x0_center),
    y_0=float(y0_center),

    ellip=SERSIC_INIT_ELLIP,

    theta=SERSIC_INIT_THETA,
)

sersic_init.amplitude.bounds = (0.0, None)

sersic_init.r_eff.bounds = (SERSIC_REFF_BOUND_MIN, float(max(img.shape)))

sersic_init.n.bounds = SERSIC_N_BOUNDS

sersic_init.x_0.bounds = (0.0, float(img.shape[1] - 1))
sersic_init.y_0.bounds = (0.0, float(img.shape[0] - 1))

sersic_init.ellip.bounds = (0.0, SERSIC_ELLIP_BOUND_MAX)

sersic_init.theta.bounds = (-np.pi / 2.0, np.pi / 2.0)

fitter = LevMarLSQFitter()

try:
    sersic_fit = fitter(sersic_init, x_fit, y_fit, z_fit)
    print(
        "[SERSIC] fitted:",
        f"amp={float(sersic_fit.amplitude.value):.3e},",
        f"r_eff={float(sersic_fit.r_eff.value):.2f},",
        f"n={float(sersic_fit.n.value):.2f},",
        f"x0={float(sersic_fit.x_0.value):.2f},",
        f"y0={float(sersic_fit.y_0.value):.2f},",
        f"ellip={float(sersic_fit.ellip.value):.3f},",
        f"theta={float(sersic_fit.theta.value):.3f}",
    )
except Exception as e:
    print(f"[SERSIC] fit failed, fallback to initial model: {e}")
    sersic_fit = sersic_init

sersic_model = sersic_fit(xx_full, yy_full)

sm = np.array(img, copy=True)

bad = (~valid150) | (~np.isfinite(sm))

sm[bad] = sersic_model[bad]


[14:33:43] [SERSIC] fit sample size = 2053357
[14:33:53] [SERSIC] fitted: amp=2.836e+00, r_eff=1076.92, n=3.69, x0=6609.95, y0=1166.48, ellip=0.000, theta=1.571


## Кроп вокруг галактики

Строим cutout вокруг NGC 1380, чтобы изофоты считались быстрее и устойчивее. Координаты crop сохраняются, чтобы затем перенести модель обратно на полный detector frame.


In [11]:
print("building cutout...")

ny, nx = sm.shape
x1 = max(0, int(x0_center - HALF_SIZE))
x2 = min(nx, int(x0_center + HALF_SIZE))
y1 = max(0, int(y0_center - HALF_SIZE))
y2 = min(ny, int(y0_center + HALF_SIZE))

img_real_c = np.asarray(img[y1:y2, x1:x2], dtype=float)
img_fill_c = np.asarray(sm[y1:y2, x1:x2], dtype=float)
img_c = np.array(img_real_c, copy=True)
valid_c = valid150[y1:y2, x1:x2]
mask_c = premask_src[y1:y2, x1:x2]

x0c = x0_center - x1
y0c = y0_center - y1

print(f"[CUTOUT] bounds = x[{x1}:{x2}], y[{y1}:{y2}]")
print(f"[CUTOUT] center in crop = ({x0c:.2f}, {y0c:.2f})")
print(f"[CUTOUT] shape = {img_c.shape}")
print(f"[CUTOUT] real finite = {int(np.isfinite(img_real_c).sum())}, filled finite = {int(np.isfinite(img_fill_c).sum())}")


[14:33:54] building cutout...
[14:33:54] [CUTOUT] bounds = x[3617:9617], y[0:4167]
[14:33:54] [CUTOUT] center in crop = (3000.63, 1167.29)
[14:33:54] [CUTOUT] shape = (4167, 6000)
[14:33:54] [CUTOUT] real finite = 17480214, filled finite = 25002000


## Входные данные для изофот

Готовим cutout и маски для `photutils.isophote`. Реальные пиксели детектора остаются основной областью фита; заполненные значения нужны только там, где численному алгоритму требуется конечное изображение.


In [12]:
print("preparing data for isophote fit...")

data_real = np.asarray(img_real_c, dtype=float).copy()
data_real[(~valid_c) | mask_c] = np.nan
ok_real = np.isfinite(data_real)
data_real_ma = np.ma.array(data_real, mask=~ok_real)

data_fill = np.asarray(img_fill_c, dtype=float).copy()
data_fill[mask_c] = np.nan
ok_fill = np.isfinite(data_fill)
data_fill_ma = np.ma.array(data_fill, mask=~ok_fill)

iso_fit_primary_mode = "real_only" if ISO_FIT_USE_REAL_PIXELS else "filled_primary"
iso_fit_fallback_enabled = bool(ISO_FIT_ALLOW_FILLED_FALLBACK)

print(f"[ISO] real finite pixels in crop = {int(ok_real.sum())}")
print(f"[ISO] filled finite pixels in crop = {int(ok_fill.sum())}")
print(f"[ISO] masked by source mask = {mask_c.sum()}, old invalid in crop = {(~valid_c).sum()}")
print(f"[ISO] primary fit mode = {iso_fit_primary_mode}, filled fallback enabled = {iso_fit_fallback_enabled}")

if ISO_FIT_USE_REAL_PIXELS and ok_real.sum() < MIN_PIXELS_ISO_CROP and not ISO_FIT_ALLOW_FILLED_FALLBACK:
    raise RuntimeError(f"[ISO] too few valid real-data pixels in crop for real-only fit: N={int(ok_real.sum())}")
if (not ISO_FIT_USE_REAL_PIXELS) and ok_fill.sum() < MIN_PIXELS_ISO_CROP:
    raise RuntimeError(f"[ISO] too few valid filled pixels in crop for isophote fit: N={int(ok_fill.sum())}")


[14:33:54] preparing data for isophote fit...
[14:33:55] [ISO] real finite pixels in crop = 17278578
[14:33:55] [ISO] filled finite pixels in crop = 24800364
[14:33:55] [ISO] masked by source mask = 201636, old invalid in crop = 7521786
[14:33:55] [ISO] primary fit mode = real_only, filled fallback enabled = True


## Фит изофот

Фитируем галактику свободными эллиптическими изофотами с плавающим центром и геометрией до заданного предела сходимости. Это основной smooth-light model для SBF residual.


In [13]:
print("fitting isophotes...")

geom = EllipseGeometry(
    x0=x0c,
    y0=y0c,
    sma=ISO_START_SMA,
    eps=ISO_START_EPS,
    pa=ISO_START_PA,
)

minsma = ISO_MINSMA
fit_maxsma = ISO_MAXSMA_FIT

def _fit_isolist_for_dataset(data_ma_try):
    ell = Ellipse(data_ma_try, geom)
    last_err_local = None
    fit_signature_local = None
    isolist_local = None

    try:
        isolist_local = ell.fit_image(
            minsma=minsma,
            maxsma=fit_maxsma,
            step=ISO_STEP_MAIN,
            linear=True,
        )
        fit_signature_local = "linear=True"
    except TypeError as e:
        last_err_local = e
        print(f"[ISO] fit_image(linear=True) signature mismatch: {e}")
    except Exception as e:
        last_err_local = e
        print(f"[ISO] fit_image(linear=True) failed: {e}")

    if isolist_local is None:
        try:
            isolist_local = ell.fit_image(
                minsma=minsma,
                maxsma=fit_maxsma,
            )
            fit_signature_local = "no linear"
        except Exception as e:
            last_err_local = e
            print(f"[ISO] fit_image(no linear) failed: {e}")

    if isolist_local is None:
        try:
            isolist_local = ell.fit_image(
                minsma=minsma,
                maxsma=fit_maxsma,
                step=ISO_STEP_COARSE,
            )
            fit_signature_local = "coarse step=20"
        except Exception as e:
            last_err_local = e
            print(f"[ISO] fit_image(coarse) failed: {e}")

    return isolist_local, fit_signature_local, last_err_local

fit_datasets = []
if ISO_FIT_USE_REAL_PIXELS:
    fit_datasets.append(("real_only", data_real_ma, int(ok_real.sum())))
    if ISO_FIT_ALLOW_FILLED_FALLBACK:
        fit_datasets.append(("filled_fallback", data_fill_ma, int(ok_fill.sum())))
else:
    fit_datasets.append(("filled_primary", data_fill_ma, int(ok_fill.sum())))

isolist = None
last_err = None
iso_fit_mode_used = None
iso_fit_signature_used = None
fit_attempt_summaries = []

for mode_label, data_ma_try, n_valid_try in fit_datasets:
    print(f"[ISO] trying dataset={mode_label}, N_finite={n_valid_try}")
    if n_valid_try < MIN_PIXELS_ISO_CROP:
        msg = f"[ISO] skipping dataset={mode_label}: too few finite pixels N={n_valid_try}"
        print(msg)
        fit_attempt_summaries.append(msg)
        continue

    isolist_try, fit_signature_try, last_err_try = _fit_isolist_for_dataset(data_ma_try)
    if isolist_try is not None and len(isolist_try) >= MIN_WORKING_ISOPHOTES:
        isolist = isolist_try
        last_err = last_err_try
        iso_fit_mode_used = mode_label
        iso_fit_signature_used = fit_signature_try
        break

    msg = (
        f"[ISO] dataset={mode_label} did not yield a working isolist: "
        f"N={0 if isolist_try is None else len(isolist_try)}, last_err={last_err_try}"
    )
    print(msg)
    fit_attempt_summaries.append(msg)
    last_err = last_err_try

if isolist is None or len(isolist) < MIN_WORKING_ISOPHOTES:
    summary_text = " | ".join(fit_attempt_summaries) if fit_attempt_summaries else "no fit attempts recorded"
    raise RuntimeError(
        f"[ISO] isolist too short after all datasets: {0 if isolist is None else len(isolist)}; "
        f"last_err={last_err}; attempts={summary_text}"
    )

print(f"[ISO] fit_image maxsma = {fit_maxsma}")
print(f"[ISO] fit dataset used = {iso_fit_mode_used}, solver path = {iso_fit_signature_used}")
print(f"[ISO] isolist N={len(isolist)}, maxsma≈{float(isolist[-1].sma):.1f} px")

x0_fit = float(np.nanmedian([iso.x0 for iso in isolist]))
y0_fit = float(np.nanmedian([iso.y0 for iso in isolist]))
eps_fit = float(np.nanmedian([iso.eps for iso in isolist]))
pa_fit = float(np.nanmedian([iso.pa for iso in isolist]))
print(f"[ISO] fitted center≈({x0_fit:.1f}, {y0_fit:.1f}) in crop, eps≈{eps_fit:.3f}, pa≈{pa_fit:.3f} rad")


[14:33:55] fitting isophotes...
[14:33:55] [ISO] trying dataset=real_only, N_finite=17278578
[14:52:39] [ISO] fit_image maxsma = 2000.0
[14:52:39] [ISO] fit dataset used = real_only, solver path = linear=True
[14:52:39] [ISO] isolist N=198, maxsma≈1990.0 px
[14:52:39] [ISO] fitted center≈(2997.1, 1168.8) in crop, eps≈0.414, pa≈1.889 rad


## Изофотная модель на crop

Строим модель по сходящимся изофотам на crop и локальный residual. Это прямой результат фита до переноса и продолжения модели на полный кадр.


In [14]:
print("building isophote model...")

model_c = build_ellipse_model(img_real_c.shape, isolist)
model_c = np.where(np.isfinite(model_c) & (model_c > 0.0), model_c, np.nan)

model = np.full_like(img, np.nan)

resid = np.full_like(img, np.nan)

model[y1:y2, x1:x2] = model_c

resid[y1:y2, x1:x2] = img_real_c - model_c

resid[premask] = np.nan

finite_resid = np.isfinite(resid)
print(
    f"[CHK] resid: finite={finite_resid.sum()}, "
    f"min={np.nanmin(resid):.3e}, med={np.nanmedian(resid):.3e}, max={np.nanmax(resid):.3e}"
)

finite_model_c = np.isfinite(model_c)
print(
    f"[CHK] model_c: finite={finite_model_c.sum()}, "
    f"min={np.nanmin(model_c):.3e}, med={np.nanmedian(model_c):.3e}, max={np.nanmax(model_c):.3e}"
)


[14:52:39] building isophote model...
[14:52:43] [CHK] resid: finite=3541163, min=-1.356e+02, med=-4.184e-02, max=6.838e+02
[14:52:43] [CHK] model_c: finite=4014133, min=1.771e+00, med=5.121e+00, max=2.811e+02


## Full-frame изофотная модель

Переносим на полный кадр модель, уже построенную на crop непосредственно по
измеренному `isolist`. За последнюю измеренную изофоту яркость и геометрия не
продолжаются.

> **✅ ИСПРАВЛЕНО В ПОРЯДКЕ JENSEN:** непокрытая моделью часть рабочего кольца
> становится частью оконной маски. Неполное покрытие само по себе не является
> ошибкой: выполнение останавливается только тогда, когда в кольце остаётся
> недостаточно реальных пикселей для измерения SBF. Покрытие ниже 95% явно
> выводится как предупреждение и должно попасть в таблицу контроля качества.


In [15]:
print("building measured full-frame isophotal model for Jensen/TRGB-SBF annuli...")

pix_scale = float(np.sqrt(pix_area))
r1_in, r1_out = SBF_LIT_INNER_ARCSEC[0] / pix_scale, SBF_LIT_INNER_ARCSEC[1] / pix_scale
r2_in, r2_out = SBF_LIT_OUTER_ARCSEC[0] / pix_scale, SBF_LIT_OUTER_ARCSEC[1] / pix_scale

# `model` уже имеет размер полного кадра: в предыдущей ячейке в него помещена
# модель crop, построенная непосредственно из измеренного isolist.
model_full = np.array(model, dtype=np.float32, copy=True)
model_full[
    (~valid150)
    | (~np.isfinite(img))
    | (~np.isfinite(model_full))
    | (model_full <= 0.0)
] = np.nan

resid_full = np.array(img - model_full, dtype=np.float32, copy=True)
resid_full[premask | (~np.isfinite(model_full)) | (~np.isfinite(img))] = np.nan

fitted_support_mask = np.isfinite(model_full) & (model_full > 0.0)
synthetic_support_mask = np.zeros_like(fitted_support_mask, dtype=bool)

fitted_sma_values = np.asarray([float(iso.sma) for iso in isolist], dtype=float)
fitted_sma_values = fitted_sma_values[np.isfinite(fitted_sma_values) & (fitted_sma_values > 0.0)]
if fitted_sma_values.size < MIN_ISOPHOTES_MODEL_PROFILE:
    raise RuntimeError(
        f"[MODEL-FULL] too few measured isophotes: N={fitted_sma_values.size}"
    )

fitted_sma_min_px = float(np.nanmin(fitted_sma_values))
fitted_sma_max_px = float(np.nanmax(fitted_sma_values))
x0_sbf_circ = float(x1 + np.nanmedian([float(iso.x0) for iso in isolist]))
y0_sbf_circ = float(y1 + np.nanmedian([float(iso.y0) for iso in isolist]))

def circular_annulus_model_coverage(r_in, r_out):
    yy_full, xx_full = np.indices(img.shape, dtype=float)
    rr = np.hypot(xx_full - x0_sbf_circ, yy_full - y0_sbf_circ)
    annulus = (rr >= float(r_in)) & (rr <= float(r_out))
    valid_annulus = annulus & valid150 & np.isfinite(img)
    model_annulus = valid_annulus & fitted_support_mask
    usable_annulus = model_annulus & (~premask)

    total_valid = int(valid_annulus.sum())
    model_valid = int(model_annulus.sum())
    usable_valid = int(usable_annulus.sum())
    frac_model = float(model_valid / total_valid) if total_valid > 0 else np.nan
    frac_usable = float(usable_valid / total_valid) if total_valid > 0 else np.nan
    return total_valid, model_valid, usable_valid, frac_model, frac_usable

inner_cov = circular_annulus_model_coverage(r1_in, r1_out)
outer_cov = circular_annulus_model_coverage(r2_in, r2_out)

print(
    f"[MODEL-FULL] measured sma range = "
    f"{fitted_sma_min_px:.1f}..{fitted_sma_max_px:.1f} px "
    f"({fitted_sma_min_px * pix_scale:.2f}..{fitted_sma_max_px * pix_scale:.2f} arcsec)"
)
print(
    f"[MODEL-FULL] Jensen/TRGB-SBF circular annuli = "
    f"{SBF_LIT_INNER_ARCSEC[0]:.1f}-{SBF_LIT_INNER_ARCSEC[1]:.1f} arcsec and "
    f"{SBF_LIT_OUTER_ARCSEC[0]:.1f}-{SBF_LIT_OUTER_ARCSEC[1]:.1f} arcsec "
    f"= {r1_in:.1f}-{r1_out:.1f} px and {r2_in:.1f}-{r2_out:.1f} px"
)
print(
    f"[MODEL-FULL] measured center: x0={x0_sbf_circ:.2f}, y0={y0_sbf_circ:.2f}"
)
print("[MODEL-FULL] outer extrapolation: DISABLED")

for annulus_name, coverage in (("inner", inner_cov), ("outer", outer_cov)):
    print(
        f"[MODEL-FULL] {annulus_name} annulus coverage: "
        f"valid={coverage[0]}, model={coverage[1]} ({100.0 * coverage[3]:.2f}%), "
        f"usable after premask={coverage[2]} ({100.0 * coverage[4]:.2f}%)"
    )

MODEL_FULL_WARN_ANNULUS_COVERAGE = 0.95
for annulus_name, coverage in (("inner", inner_cov), ("outer", outer_cov)):
    if coverage[0] < MIN_PIXELS_SBF:
        raise RuntimeError(
            f"[MODEL-FULL] too few valid detector pixels in the {annulus_name} "
            f"SBF annulus: N={coverage[0]}"
        )
    if coverage[2] < MIN_PIXELS_SBF:
        raise RuntimeError(
            f"[MODEL-FULL] too few measured-model pixels remain in the {annulus_name} "
            f"SBF annulus after masking: N={coverage[2]}"
        )
    if (not np.isfinite(coverage[3])) or coverage[3] < MODEL_FULL_WARN_ANNULUS_COVERAGE:
        print(
            f"[MODEL-FULL][WARN] measured isophotes cover "
            f"{100.0 * coverage[3]:.2f}% of valid pixels in the {annulus_name} "
            "SBF annulus; the uncovered pixels will enter the Fourier window mask"
        )

print(
    f"[MODEL-FULL] resid_full finite={int(np.isfinite(resid_full).sum())}, "
    f"model_full finite={int(np.isfinite(model_full).sum())}"
)


[14:52:43] building measured full-frame isophotal model for Jensen/TRGB-SBF annuli...
[14:52:45] [MODEL-FULL] measured sma range = 20.0..1990.0 px (0.62..62.14 arcsec)
[14:52:45] [MODEL-FULL] Jensen/TRGB-SBF circular annuli = 8.2-16.4 arcsec and 16.4-32.8 arcsec = 262.6-525.2 px and 525.2-1050.4 px
[14:52:45] [MODEL-FULL] measured center: x0=6614.14, y0=1168.76
[14:52:45] [MODEL-FULL] outer extrapolation: DISABLED
[14:52:45] [MODEL-FULL] inner annulus coverage: valid=649906, model=649906 (100.00%), usable after premask=648558 (99.79%)
[14:52:45] [MODEL-FULL] outer annulus coverage: valid=2560924, model=2058281 (80.37%), usable after premask=2042903 (79.77%)
[14:52:45] [MODEL-FULL][WARN] measured isophotes cover 80.37% of valid pixels in the outer SBF annulus; the uncovered pixels will enter the Fourier window mask
[14:52:45] [MODEL-FULL] resid_full finite=3541163, model_full finite=3574962


## Clipping science residual

Вычитаем `model_full` из science image и ограничиваем экстремальные residual pixels для SBF-измерения. Цель: подавить сильные non-SBF выбросы, сохранив поле пиксельных флуктуаций.

> **✅ ОСОЗНАННОЕ ОТЛИЧИЕ — ОСТАВЛЯЕМ.** В описанном алгоритме Jensen такого ограничения амплитуды нет. Здесь значения за пределами медиана ±3.5σ не удаляются, а прижимаются к границе: это симметричное винзорирование. Для статьи обязателен параллельный результат без ограничения и тест отсутствия смещения $P_0$.


In [16]:
print("building sigma-capped full-frame science residual for SBF annuli...")

if "resid_full" not in globals() or "model_full" not in globals():
    raise RuntimeError("[RESID-CLIP] run the extrapolated full-frame model cell before sigma clipping")

science_clip_mask = (~premask) & np.isfinite(resid_full) & np.isfinite(model_full) & (model_full > 0.0)
n_science_clip = int(science_clip_mask.sum())
if n_science_clip < MIN_PIXELS_SBF:
    raise RuntimeError(f"[RESID-CLIP] too few valid science pixels: N={n_science_clip}")

vals_science_clip = resid_full[science_clip_mask]
mean_clip_full, med_clip_full, std_clip_full = sigma_clipped_stats(
    vals_science_clip,
    sigma=CLIP_SIGMA_QC,
    maxiters=CLIP_MAXIT_QC,
)

if (not np.isfinite(std_clip_full)) or std_clip_full <= 0.0:
    raise RuntimeError(f"[RESID-CLIP] bad clipped std={std_clip_full}")

clip_lo_full = float(med_clip_full - CLIP_SIGMA_QC * std_clip_full)
clip_hi_full = float(med_clip_full + CLIP_SIGMA_QC * std_clip_full)

resid_full_clip = np.array(resid_full, dtype=np.float32, copy=True)
resid_full_clip[science_clip_mask] = np.clip(
    resid_full_clip[science_clip_mask],
    clip_lo_full,
    clip_hi_full,
)
resid_full_clip[~science_clip_mask] = np.nan

resid_full_clip_delta = np.full_like(resid_full_clip, np.nan, dtype=np.float32)
resid_full_clip_delta[science_clip_mask] = (
    resid_full_clip[science_clip_mask] - resid_full[science_clip_mask]
)

n_clip_changed = int(np.count_nonzero(np.abs(resid_full_clip_delta[science_clip_mask]) > 0.0))
clip_tag = CLIP_TAG_QC

print(
    f"[RESID-CLIP] sigma={CLIP_SIGMA_QC}, valid={n_science_clip}, "
    f"med={med_clip_full:.3e}, std={std_clip_full:.3e}"
)
print(
    f"[RESID-CLIP] cap range = [{clip_lo_full:.3e}, {clip_hi_full:.3e}], "
    f"changed={n_clip_changed} px ({100.0 * n_clip_changed / n_science_clip:.2f}%)"
)


[14:52:45] building sigma-capped full-frame science residual for SBF annuli...
[14:52:45] [RESID-CLIP] sigma=3.5, valid=3541163, med=-4.560e-02, std=5.949e-01
[14:52:45] [RESID-CLIP] cap range = [-2.128e+00, 2.036e+00], changed=67701 px (1.91%)


In [17]:
print("saving sigma-capped science residual FITS right after clipping...")

resid_science_clip_path = out_dir / f"{stem}_sbf_resid_full_science_clip_{CLIP_TAG_QC}sigma.fits"
resid_science_delta_path = out_dir / f"{stem}_sbf_resid_full_science_clip_{CLIP_TAG_QC}sigma_delta.fits"
resid_science_path = out_dir / f"{stem}_sbf_resid_full_science.fits"

fits.writeto(resid_science_clip_path, np.array(resid_full_clip, dtype=np.float32), hdr150, overwrite=True)
fits.writeto(resid_science_delta_path, np.array(resid_full_clip_delta, dtype=np.float32), hdr150, overwrite=True)
fits.writeto(resid_science_path, np.array(resid_full_clip, dtype=np.float32), hdr150, overwrite=True)

print(f"[OUT] science residual clipped  -> {resid_science_clip_path}")
print(f"[OUT] clip delta                -> {resid_science_delta_path}")
print(f"[OUT] science residual used     -> {resid_science_path}")


[14:52:45] saving sigma-capped science residual FITS right after clipping...
[14:52:46] [OUT] science residual clipped  -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_resid_full_science_clip_3p5sigma.fits
[14:52:46] [OUT] clip delta                -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_resid_full_science_clip_3p5sigma_delta.fits
[14:52:46] [OUT] science residual used     -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_resid_full_science.fits


In [18]:
print("saving raw and sigma-capped full-frame residual FITS...")

model_for_view = model_full if "model_full" in globals() else model
model_support = np.isfinite(model_for_view) & (model_for_view > 0.0)
model_zero = np.where(model_support, model_for_view, 0.0)
resid_full_frame = np.array(img - model_zero, dtype=np.float32, copy=True)
resid_full_frame[premask | ~np.isfinite(resid_full_frame)] = np.nan

resid_full_frame_path = out_dir / f"{stem}_sbf_resid_full_frame.fits"
resid_full_masked_path = out_dir / f"{stem}_sbf_resid_full_masked.fits"
model_support_path = out_dir / f"{stem}_sbf_model_support_mask.fits"
model_full_path = out_dir / f"{stem}_sbf_model_full.fits"
resid_science_raw_path = out_dir / f"{stem}_sbf_resid_full_science_raw.fits"
resid_science_clip_path = out_dir / f"{stem}_sbf_resid_full_science_clip_{CLIP_TAG_QC}sigma.fits"
resid_science_delta_path = out_dir / f"{stem}_sbf_resid_full_science_clip_{CLIP_TAG_QC}sigma_delta.fits"
resid_science_path = out_dir / f"{stem}_sbf_resid_full_science.fits"

hdr_resid_full = hdr150.copy()
hdr_resid_full["BUNIT"] = hdr150.get("BUNIT", "")
hdr_resid_full.add_history("Full-frame residual view for inspection")
hdr_resid_full.add_history("Where model is finite: background-subtracted SCI - smooth model")
hdr_resid_full.add_history("Where model is not finite: background-subtracted SCI is kept unchanged")
hdr_resid_full.add_history("Pixels in premask or non-finite output are stored as NaN")

fits.writeto(resid_full_frame_path, resid_full_frame, hdr_resid_full, overwrite=True)
fits.writeto(resid_full_masked_path, resid_full_frame, hdr_resid_full, overwrite=True)
fits.writeto(model_support_path, model_support.astype(np.uint8), hdr150, overwrite=True)
fits.writeto(model_full_path, np.array(model_for_view, dtype=np.float32), hdr150, overwrite=True)

if "resid_full" in globals():
    fits.writeto(resid_science_raw_path, np.array(resid_full, dtype=np.float32), hdr150, overwrite=True)

if "resid_full_clip" in globals():
    fits.writeto(resid_science_clip_path, np.array(resid_full_clip, dtype=np.float32), hdr150, overwrite=True)
    fits.writeto(resid_science_delta_path, np.array(resid_full_clip_delta, dtype=np.float32), hdr150, overwrite=True)
    fits.writeto(resid_science_path, np.array(resid_full_clip, dtype=np.float32), hdr150, overwrite=True)
elif "resid_full" in globals():
    fits.writeto(resid_science_path, np.array(resid_full, dtype=np.float32), hdr150, overwrite=True)

finite_full = np.isfinite(resid_full_frame)
print(f"[OUT] full-frame residual view -> {resid_full_frame_path}")
print(f"[OUT] compatibility copy      -> {resid_full_masked_path}")
print(f"[OUT] model support mask       -> {model_support_path}")
print(f"[OUT] smooth model             -> {model_full_path}")
if "resid_full" in globals():
    print(f"[OUT] science residual raw      -> {resid_science_raw_path}")
if "resid_full_clip" in globals():
    print(f"[OUT] science residual clipped  -> {resid_science_clip_path}")
    print(f"[OUT] clip delta                -> {resid_science_delta_path}")
print(f"[OUT] science residual used     -> {resid_science_path}")
print(
    f"[RESID-FULL] finite={int(finite_full.sum())} px, "
    f"model_support={int(model_support.sum())} px "
    f"({100.0 * model_support.sum() / model_support.size:.2f}%)"
)
print(
    f"[RESID-FULL] med={np.nanmedian(resid_full_frame):.3e}, "
    f"std={np.nanstd(resid_full_frame):.3e}"
)


[14:52:46] saving raw and sigma-capped full-frame residual FITS...
[14:52:46] [OUT] full-frame residual view -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_resid_full_frame.fits
[14:52:46] [OUT] compatibility copy      -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_resid_full_masked.fits
[14:52:46] [OUT] model support mask       -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_model_support_mask.fits
[14:52:46] [OUT] smooth model             -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_model_full.fits
[14:52:46] [OUT] science residual raw      -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_resid_full_science_raw.fits
[14:52:46] [OUT] science residual clipped  -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_resid_full_science_clip_3p5sigma.fits
[14:52:46] [OUT] clip delta                -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_resid_full_science_clip_3p5sigma_delta.fits
[14:52:4

## Единый science residual path

Выбираем единую пару residual/model, которую обязаны использовать все дальнейшие SBF-измерения. Это защищает от случайного использования устаревших промежуточных residual.


In [19]:
print("selecting unified science residual/model path for all spectral SBF measurements...")

if "model_full" not in globals() or "resid_full_clip" not in globals():
    raise RuntimeError("[SCIENCE-PATH] run the full-frame model and sigma-capped residual cells before spectral SBF")

science_model = model_full
science_resid = resid_full_clip
science_model_name = "model_full"
science_resid_name = f"resid_full_clip_{CLIP_TAG_QC}sigma"

if science_model.shape != science_resid.shape:
    raise RuntimeError(
        f"[SCIENCE-PATH] shape mismatch: model={science_model.shape}, resid={science_resid.shape}"
    )

def _science_array_stats(arr):
    good = np.isfinite(arr)
    if not np.any(good):
        return {
            "shape": tuple(arr.shape),
            "n_finite": 0,
            "min": np.nan,
            "median": np.nan,
            "max": np.nan,
            "std": np.nan,
        }
    vals = arr[good]
    return {
        "shape": tuple(arr.shape),
        "n_finite": int(good.sum()),
        "min": float(np.nanmin(vals)),
        "median": float(np.nanmedian(vals)),
        "max": float(np.nanmax(vals)),
        "std": float(np.nanstd(vals)),
    }

science_model_stats = _science_array_stats(science_model)
science_resid_stats = _science_array_stats(science_resid)

print(f"[SCIENCE-PATH] science_model = {science_model_name}")
print(f"[SCIENCE-PATH] science_resid = {science_resid_name}")
print(
    f"[SCIENCE-PATH][model] shape={science_model_stats['shape']}, finite={science_model_stats['n_finite']}, "
    f"min={science_model_stats['min']:.3e}, med={science_model_stats['median']:.3e}, "
    f"max={science_model_stats['max']:.3e}, std={science_model_stats['std']:.3e}"
)
print(
    f"[SCIENCE-PATH][resid] shape={science_resid_stats['shape']}, finite={science_resid_stats['n_finite']}, "
    f"min={science_resid_stats['min']:.3e}, med={science_resid_stats['median']:.3e}, "
    f"max={science_resid_stats['max']:.3e}, std={science_resid_stats['std']:.3e}"
)


[14:52:47] selecting unified science residual/model path for all spectral SBF measurements...
[14:52:47] [SCIENCE-PATH] science_model = model_full
[14:52:47] [SCIENCE-PATH] science_resid = resid_full_clip_3p5sigma
[14:52:47] [SCIENCE-PATH][model] shape=(4377, 9876), finite=3574962, min=1.771e+00, med=5.705e+00, max=2.811e+02, std=1.586e+01
[14:52:47] [SCIENCE-PATH][resid] shape=(4377, 9876), finite=3541163, min=-2.128e+00, med=-4.184e-02, max=2.036e+00, std=6.542e-01


## Диагностика пропущенных компактных источников

Ищем на остатках кандидатов, пропущенных первичным детектированием. Эта ячейка является только диагностикой: она не меняет научную маску. Окончательная маска ниже строится из единого каталога по общему пределу $m_{cut}$.

> **✅ ИСПРАВЛЕНО В ПОРЯДКЕ JENSEN:** поздняя эвристическая сегментация сохраняется только как контроль качества. Ни один найденный здесь пиксель не попадает в рабочую маску без прохождения через общий фотометрический каталог и предел $m_{cut}$.


In [20]:
print("building diagnostic catalog of compact residual leftovers...")

if "science_resid" not in globals() or "science_model" not in globals():
    raise RuntimeError("[RESID-MASK] unified science residual/model are unavailable; rerun the science-path cell")
if "img" not in globals() or "model_full" not in globals():
    raise RuntimeError("[RESID-MASK] img/model_full are unavailable; rerun the model cell")
if "premask" not in globals():
    raise RuntimeError("[RESID-MASK] premask is unavailable; rerun the premask cell")

RESID_EXTRA_MASK_SIGMA = 3
RESID_EXTRA_MASK_NPIX = 9
RESID_EXTRA_MASK_KERNEL_SIGMA = 1.0
RESID_EXTRA_MASK_DILATE = 2
RESID_EXTRA_MASK_MAX_AREA = 3000
RESID_EXTRA_MASK_POSITIVE_ONLY = True
RESID_EXTRA_MASK_BRANCH = "compactresidmask"

if "premask_base" not in globals():
    premask_base = np.array(premask, dtype=bool, copy=True)
if "premask_src_base" not in globals() and "premask_src" in globals():
    premask_src_base = np.array(premask_src, dtype=bool, copy=True)

detect_support = (~premask_base) & np.isfinite(science_resid) & np.isfinite(science_model) & (science_model > 0.0)
n_detect_support = int(detect_support.sum())
if n_detect_support < MIN_PIXELS_SBF:
    raise RuntimeError(f"[RESID-MASK] too few science pixels for residual-mask detection: N={n_detect_support}")

vals_det = np.asarray(science_resid[detect_support], dtype=float)
_, med_det, std_det = sigma_clipped_stats(vals_det, sigma=CLIP_SIGMA_QC, maxiters=CLIP_MAXIT_QC)
if (not np.isfinite(std_det)) or std_det <= 0.0:
    raise RuntimeError(f"[RESID-MASK] bad residual std for compact-mask detection: std={std_det}")

resid_det = np.zeros(science_resid.shape, dtype=np.float32)
resid_centered = np.asarray(science_resid - med_det, dtype=np.float32)
if RESID_EXTRA_MASK_POSITIVE_ONLY:
    resid_det[detect_support] = np.maximum(resid_centered[detect_support], 0.0)
else:
    resid_det[detect_support] = np.abs(resid_centered[detect_support])

if RESID_EXTRA_MASK_KERNEL_SIGMA > 0.0:
    resid_det_smooth = gaussian_filter(resid_det, sigma=RESID_EXTRA_MASK_KERNEL_SIGMA, mode="nearest")
else:
    resid_det_smooth = resid_det

threshold_det = float(RESID_EXTRA_MASK_SIGMA * std_det)
segm_extra = detect_sources(resid_det_smooth, threshold_det, npixels=RESID_EXTRA_MASK_NPIX)

extra_resid_mask_raw = np.zeros_like(premask_base, dtype=bool)
extra_resid_mask = np.zeros_like(premask_base, dtype=bool)
df_resid_extra_sources = pd.DataFrame(columns=["label", "area_pix", "peak_resid", "peak_snr"])

if segm_extra is not None:
    seg_data = np.asarray(segm_extra.data, dtype=int)
    labels = np.unique(seg_data)
    labels = labels[labels > 0]
    if labels.size > 0:
        areas_all = np.bincount(seg_data.ravel())
        peak_vals = ndimage.maximum(resid_det, labels=seg_data, index=labels)
        peak_snr = peak_vals / std_det
        keep_rows = []
        keep_labels = []
        for lab, peak_val, peak_sig in zip(labels, peak_vals, peak_snr):
            area_pix = int(areas_all[int(lab)])
            if area_pix > RESID_EXTRA_MASK_MAX_AREA:
                continue
            if not np.isfinite(peak_val) or peak_val <= threshold_det:
                continue
            keep_labels.append(int(lab))
            keep_rows.append({
                "label": int(lab),
                "area_pix": area_pix,
                "peak_resid": float(peak_val),
                "peak_snr": float(peak_sig),
            })
        if keep_labels:
            extra_resid_mask_raw = np.isin(seg_data, np.asarray(keep_labels, dtype=int)) & detect_support
            if RESID_EXTRA_MASK_DILATE > 0:
                extra_resid_mask = ndimage.binary_dilation(extra_resid_mask_raw, iterations=RESID_EXTRA_MASK_DILATE)
            else:
                extra_resid_mask = extra_resid_mask_raw.copy()
            extra_resid_mask &= detect_support
            df_resid_extra_sources = pd.DataFrame(keep_rows).sort_values(["peak_snr", "area_pix"], ascending=[False, False]).reset_index(drop=True)

# Важно: диагностические кандидаты не имеют права менять science mask.
premask_extra_resid_candidate = np.array(extra_resid_mask, dtype=bool, copy=True)
premask_extra_resid = np.zeros_like(premask_base, dtype=bool)
premask = np.array(premask_base, dtype=bool, copy=True)

resid_full = np.array(img - model_full, dtype=np.float32, copy=True)
resid_full[premask | (~np.isfinite(model_full)) | (~np.isfinite(img))] = np.nan

science_clip_mask = (~premask) & np.isfinite(resid_full) & np.isfinite(model_full) & (model_full > 0.0)
n_science_clip = int(science_clip_mask.sum())
if n_science_clip < MIN_PIXELS_SBF:
    raise RuntimeError(f"[RESID-MASK] too few valid science pixels after extra mask: N={n_science_clip}")

vals_science_clip = np.asarray(resid_full[science_clip_mask], dtype=float)
mean_clip_full, med_clip_full, std_clip_full = sigma_clipped_stats(
    vals_science_clip,
    sigma=CLIP_SIGMA_QC,
    maxiters=CLIP_MAXIT_QC,
)
if (not np.isfinite(std_clip_full)) or std_clip_full <= 0.0:
    raise RuntimeError(f"[RESID-MASK] bad clipped std after extra mask: std={std_clip_full}")

clip_lo_full = float(med_clip_full - CLIP_SIGMA_QC * std_clip_full)
clip_hi_full = float(med_clip_full + CLIP_SIGMA_QC * std_clip_full)
resid_full_clip = np.array(resid_full, dtype=np.float32, copy=True)
resid_full_clip[science_clip_mask] = np.clip(resid_full_clip[science_clip_mask], clip_lo_full, clip_hi_full)
resid_full_clip[~science_clip_mask] = np.nan

resid_full_clip_delta = np.full_like(resid_full_clip, np.nan, dtype=np.float32)
resid_full_clip_delta[science_clip_mask] = resid_full_clip[science_clip_mask] - resid_full[science_clip_mask]

science_model = model_full
science_resid = resid_full_clip
science_model_name = "model_full"
science_resid_name = f"resid_full_clip_{CLIP_TAG_QC}sigma"

n_raw_extra = int(extra_resid_mask_raw.sum())
n_final_extra = int(extra_resid_mask.sum())
n_clip_changed = int(np.count_nonzero(np.abs(resid_full_clip_delta[science_clip_mask]) > 0.0))

extra_mask_path = out_dir / f"{stem}_sbf_resid_extra_candidates_{RESID_EXTRA_MASK_BRANCH}.fits"
premask_path = out_dir / f"{stem}_sbf_premask_before_mcut.fits"
resid_science_clip_path_branch = out_dir / f"{stem}_sbf_resid_full_science_clip_{CLIP_TAG_QC}sigma_before_mcut.fits"
resid_science_delta_path_branch = out_dir / f"{stem}_sbf_resid_full_science_clip_{CLIP_TAG_QC}sigma_delta_before_mcut.fits"

fits.writeto(extra_mask_path, np.asarray(premask_extra_resid_candidate, dtype=np.uint8), hdr150, overwrite=True)
fits.writeto(premask_path, np.asarray(premask, dtype=np.uint8), hdr150, overwrite=True)
fits.writeto(resid_science_clip_path_branch, np.asarray(resid_full_clip, dtype=np.float32), hdr150, overwrite=True)
fits.writeto(resid_science_delta_path_branch, np.asarray(resid_full_clip_delta, dtype=np.float32), hdr150, overwrite=True)

print(
    f"[RESID-MASK] residual sigma for compact leftover detection: med={med_det:.3e}, std={std_det:.3e}, "
    f"threshold={threshold_det:.3e} ({RESID_EXTRA_MASK_SIGMA:.1f} sigma)"
)
print(
    f"[RESID-MASK] diagnostic candidates={len(df_resid_extra_sources)}, raw={n_raw_extra} px, "
    f"dilated={n_final_extra} px; science premask unchanged ({100.0 * premask.sum() / premask.size:.2f}%)"
)
print(
    f"[RESID-MASK] updated science clip: valid={n_science_clip}, med={med_clip_full:.3e}, std={std_clip_full:.3e}, "
    f"cap=[{clip_lo_full:.3e}, {clip_hi_full:.3e}], changed={n_clip_changed} px ({100.0 * n_clip_changed / n_science_clip:.2f}%)"
)
print(f"[OUT] diagnostic candidates -> {extra_mask_path}")
print(f"[OUT] premask before m_cut  -> {premask_path}")
print(f"[OUT] residual before m_cut -> {resid_science_clip_path_branch}")
print(f"[OUT] clip delta            -> {resid_science_delta_path_branch}")
print("[RESID-MASK] candidates are diagnostic only; the catalog cell below defines the final mask.")
display(df_resid_extra_sources.head(20))


[14:52:47] building diagnostic catalog of compact residual leftovers...
[14:52:51] [RESID-MASK] residual sigma for compact leftover detection: med=-4.184e-02, std=6.542e-01, threshold=1.963e+00 (3.0 sigma)
[14:52:51] [RESID-MASK] diagnostic candidates=45, raw=3222 px, dilated=6268 px; science premask unchanged (24.00%)
[14:52:51] [RESID-MASK] updated science clip: valid=3541163, med=-4.560e-02, std=5.949e-01, cap=[-2.128e+00, 2.036e+00], changed=67701 px (1.91%)
[14:52:51] [OUT] diagnostic candidates -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_resid_extra_candidates_compactresidmask.fits
[14:52:51] [OUT] premask before m_cut  -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_premask_before_mcut.fits
[14:52:51] [OUT] residual before m_cut -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_resid_full_science_clip_3p5sigma_before_mcut.fits
[14:52:51] [OUT] clip delta            -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_resid_ful

,label,area_pix,peak_resid,peak_snr
0,24,1406,2.078241,3.176675
1,23,413,2.078241,3.176675
2,20,378,2.078241,3.176675
3,13,128,2.078241,3.176675
4,6,78,2.078241,3.176675
5,28,70,2.078241,3.176675
6,30,61,2.078241,3.176675
7,18,42,2.078241,3.176675
8,17,40,2.078241,3.176675
9,4,39,2.078241,3.176675


## Единый каталог источников и окончательная маска

Строим единый фотометрический каталог на остатках модели. Один предел $m_{cut}$ разделяет источники на две группы: более яркие входят в окончательную маску, более слабые остаются в кадре и учитываются статистически через $P_r$.

> **✅ ИСПРАВЛЕНО В ПОРЯДКЕ JENSEN:** каталог, $m_{cut}$, окончательная маска и последующая модель $P_r$ используют один и тот же набор источников. Если известен предел полноты из искусственных звёзд, его следует задать через `SBF_PR_MLIM_OVERRIDE`; автоматическая оценка явно помечается как приближение по S/N каталога.


In [21]:
print("building one compact-source catalog, completeness cut, and final science mask...")

jy_per_pix = MJY_SR_TO_JY_PER_ARCSEC2 * pix_area
source_catalog_ref_mask = np.isfinite(model_full) & (model_full > 0.0)
source_catalog_columns = [
    "label", "xcentroid", "ycentroid", "area_pix",
    "flux_img_signed", "flux_img_positive", "flux_img_adopted",
    "flux_err_img", "snr", "flux_jy", "mag_ab", "mask_brighter_than_mcut",
]

if not np.any(source_catalog_ref_mask):
    raise RuntimeError("[SRC-CAT] model_full has no positive pixels for reference mask")

source_catalog_seg_full = np.zeros(model_full.shape, dtype=np.int32)
source_catalog_build_info = {}
compact_source_catalog = None

yy_ref, xx_ref = np.where(source_catalog_ref_mask)
y0_src, y1_src = int(yy_ref.min()), int(yy_ref.max()) + 1
x0_src, x1_src = int(xx_ref.min()), int(xx_ref.max()) + 1
source_catalog_bbox = (y0_src, y1_src, x0_src, x1_src)

ref_mask_crop = np.asarray(source_catalog_ref_mask[y0_src:y1_src, x0_src:x1_src], dtype=bool)
catalog_image = np.array(
    img[y0_src:y1_src, x0_src:x1_src] - model_full[y0_src:y1_src, x0_src:x1_src],
    dtype=float,
    copy=True,
)
catalog_image[~np.isfinite(catalog_image)] = np.nan
catalog_image[~ref_mask_crop] = np.nan

valid_detect = ref_mask_crop & np.isfinite(catalog_image)
detect_values = catalog_image[valid_detect]
if detect_values.size < MIN_PIXELS_SIGMA_CLIP:
    raise RuntimeError(f"[SRC-CAT] too few valid residual pixels in model-support crop: N={detect_values.size}")

mean_det, med_det, std_det = sigma_clipped_stats(
    detect_values,
    sigma=SIGMA_STAT,
    maxiters=SIGMA_MAXIT,
)
if "premask_noise_sigma" in globals() and np.shape(premask_noise_sigma) == np.shape(img):
    catalog_noise_sigma = np.asarray(
        premask_noise_sigma[y0_src:y1_src, x0_src:x1_src], dtype=float
    )
else:
    catalog_noise_sigma = np.full_like(
        catalog_image, max(std_det, ROBUST_SCALE_FLOOR), dtype=float
    )
catalog_noise_sigma[~valid_detect] = np.nan
catalog_threshold_map = np.full_like(catalog_image, np.inf, dtype=float)
catalog_threshold_map[valid_detect] = (
    med_det + SBF_PR_DET_SIGMA * catalog_noise_sigma[valid_detect]
)
threshold_det = float(np.nanmedian(catalog_threshold_map[valid_detect]))

detect_image = np.zeros_like(catalog_image, dtype=float)
detect_image[valid_detect] = catalog_image[valid_detect]

segm_detect = detect_sources(
    detect_image,
    threshold=catalog_threshold_map,
    npixels=SBF_PR_DET_NPIXELS,
    connectivity=SOURCE_DETECT_CONNECTIVITY,
)
n_detect_raw = int(segm_detect.nlabels) if segm_detect is not None else 0

if segm_detect is not None and SBF_PR_DO_DEBLEND:
    try:
        segm_detect = deblend_sources(
            detect_image,
            segm_detect,
            npixels=SBF_PR_DET_NPIXELS,
            nlevels=DEBLEND_NLEVELS,
            contrast=DEBLEND_CONTRAST,
            nproc=DEBLEND_NPROC,
            progress_bar=False,
        )
    except Exception as parallel_error:
        if DEBLEND_NPROC == 1:
            raise
        print(f"[SRC-CAT] parallel deblend failed: {parallel_error}; retrying sequentially")
        segm_detect = deblend_sources(
            detect_image,
            segm_detect,
            npixels=SBF_PR_DET_NPIXELS,
            nlevels=DEBLEND_NLEVELS,
            contrast=DEBLEND_CONTRAST,
            nproc=1,
            progress_bar=False,
        )

n_detect_deblend = int(segm_detect.nlabels) if segm_detect is not None else 0

premask_overlap_count = np.nan
if "premask_segm" in globals() and premask_segm is not None and "premask_compact_labels" in globals():
    premask_crop = np.asarray(premask_segm.data[y0_src:y1_src, x0_src:x1_src], dtype=int)
    premask_overlap_labels = np.intersect1d(
        np.unique(premask_crop[ref_mask_crop & (premask_crop > 0)]),
        np.asarray(premask_compact_labels, dtype=int),
        assume_unique=False,
    )
    premask_overlap_count = int(premask_overlap_labels.size)

source_catalog_df = pd.DataFrame(columns=source_catalog_columns)
source_catalog_seg_crop = np.zeros_like(detect_image, dtype=np.int32)
dropped_large_area = 0
dropped_nonpositive_signed_flux = 0
dropped_nonfinite_mag = 0
n_compact_area = 0
n_final = 0
source_pixels_final = 0

if segm_detect is not None and segm_detect.nlabels > 0:
    seg_data = np.asarray(segm_detect.data, dtype=np.int32)
    seg_data[~ref_mask_crop] = 0

    labels_all = np.asarray(segm_detect.labels, dtype=int)
    max_label = int(seg_data.max()) if seg_data.size else 0
    counts = np.bincount(seg_data.ravel(), minlength=max_label + 1) if max_label > 0 else np.zeros(1, dtype=int)
    counts[0] = 0

    compact_labels = labels_all[(counts[labels_all] > 0) & (counts[labels_all] <= SBF_PR_MAX_COMPACT_AREA)]
    dropped_large_area = int(labels_all.size - compact_labels.size)
    n_compact_area = int(compact_labels.size)

    label_lut = np.zeros(max_label + 1, dtype=np.int32)
    if compact_labels.size > 0:
        label_lut[compact_labels] = compact_labels.astype(np.int32)
    source_catalog_seg_crop = label_lut[seg_data]

    ys_src, xs_src = np.nonzero(source_catalog_seg_crop > 0)
    if ys_src.size > 0:
        labels_src = source_catalog_seg_crop[ys_src, xs_src].astype(np.int64, copy=False)
        labels_present = np.unique(labels_src)
        values_src = np.nan_to_num(catalog_image[ys_src, xs_src], nan=0.0)
        positive_values_src = np.clip(values_src, 0.0, None)

        signed_sum = np.bincount(labels_src, weights=values_src, minlength=max_label + 1)
        positive_sum = np.bincount(labels_src, weights=positive_values_src, minlength=max_label + 1)
        noise_variance_src = np.nan_to_num(
            catalog_noise_sigma[ys_src, xs_src] ** 2,
            nan=max(std_det, ROBUST_SCALE_FLOOR) ** 2,
            posinf=max(std_det, ROBUST_SCALE_FLOOR) ** 2,
        )
        variance_sum = np.bincount(
            labels_src, weights=noise_variance_src, minlength=max_label + 1
        )
        area_sum = np.bincount(labels_src, minlength=max_label + 1)
        x_sum = np.bincount(labels_src, weights=xs_src.astype(float), minlength=max_label + 1)
        y_sum = np.bincount(labels_src, weights=ys_src.astype(float), minlength=max_label + 1)

        rows = []
        kept_labels = []
        for label in labels_present.astype(int):
            area_pix = int(area_sum[label])
            if area_pix <= 0:
                continue

            flux_img_signed = float(signed_sum[label])
            flux_img_positive = float(positive_sum[label])
            flux_img_adopted = flux_img_signed

            if not (np.isfinite(flux_img_adopted) and flux_img_adopted > 0.0):
                dropped_nonpositive_signed_flux += 1
                continue

            flux_err_img = float(np.sqrt(max(variance_sum[label], ROBUST_SCALE_FLOOR ** 2)))
            snr = float(flux_img_adopted / flux_err_img) if flux_err_img > 0.0 else np.nan
            flux_jy = float(flux_img_adopted * jy_per_pix)
            mag_ab = float(-2.5 * np.log10(flux_jy / AB_ZEROPOINT_JY)) if flux_jy > 0.0 else np.nan
            if not np.isfinite(mag_ab):
                dropped_nonfinite_mag += 1
                continue

            kept_labels.append(label)
            rows.append({
                "label": label,
                "xcentroid": float(x0_src + x_sum[label] / area_pix),
                "ycentroid": float(y0_src + y_sum[label] / area_pix),
                "area_pix": area_pix,
                "flux_img_signed": flux_img_signed,
                "flux_img_positive": flux_img_positive,
                "flux_img_adopted": flux_img_adopted,
                "flux_err_img": flux_err_img,
                "snr": snr,
                "flux_jy": flux_jy,
                "mag_ab": mag_ab,
                "mask_brighter_than_mcut": False,
            })

        source_catalog_df = pd.DataFrame(rows, columns=source_catalog_columns)
        if not source_catalog_df.empty:
            source_catalog_df = source_catalog_df.sort_values(["mag_ab", "label"], na_position="last").reset_index(drop=True)

        final_lut = np.zeros(max_label + 1, dtype=np.int32)
        if kept_labels:
            final_lut[np.asarray(kept_labels, dtype=int)] = np.asarray(kept_labels, dtype=np.int32)
        source_catalog_seg_crop = final_lut[source_catalog_seg_crop]
        source_catalog_seg_full[y0_src:y1_src, x0_src:x1_src] = source_catalog_seg_crop
        n_final = int(len(kept_labels))
        source_pixels_final = int(np.count_nonzero(source_catalog_seg_crop))

source_catalog_build_info.update({
    "bbox": source_catalog_bbox,
    "support_pixels": int(source_catalog_ref_mask.sum()),
    "n_detect_raw": int(n_detect_raw),
    "n_detect_deblend": int(n_detect_deblend),
    "n_compact_area": int(n_compact_area),
    "dropped_large_area": int(dropped_large_area),
    "dropped_nonpositive_signed_flux": int(dropped_nonpositive_signed_flux),
    "dropped_nonfinite_mag": int(dropped_nonfinite_mag),
    "n_final": int(n_final),
    "source_pixels_final": int(source_pixels_final),
    "premask_overlap_count": premask_overlap_count,
    "detect_threshold": float(threshold_det),
    "detect_med": float(med_det),
    "detect_std": float(std_det),
    "detect_image": SBF_PR_SOURCE_IMAGE,
})

# Один и тот же m_cut используется и для маски, и для интеграла P_r.
try:
    source_mask_m_cut_override = float(SBF_PR_MLIM_OVERRIDE)
except (TypeError, ValueError):
    source_mask_m_cut_override = np.nan

SOURCE_CATALOG_RELIABLE_SNR = 5.0
SOURCE_CATALOG_AUTO_CUT_QUANTILE = 0.95
if np.isfinite(source_mask_m_cut_override):
    source_mask_m_cut = source_mask_m_cut_override
    source_mask_m_cut_method = "manual_completeness_override"
else:
    reliable_catalog = source_catalog_df[
        np.isfinite(source_catalog_df["mag_ab"])
        & np.isfinite(source_catalog_df["snr"])
        & (source_catalog_df["snr"] >= SOURCE_CATALOG_RELIABLE_SNR)
    ].copy()
    if len(reliable_catalog) >= SBF_PR_MIN_SOURCES_GLOBAL:
        source_mask_m_cut = float(
            np.nanquantile(reliable_catalog["mag_ab"], SOURCE_CATALOG_AUTO_CUT_QUANTILE)
            + SBF_PR_MLIM_OFFSET
        )
        source_mask_m_cut_method = "catalog_snr5_q95_approximation"
    elif not source_catalog_df.empty:
        source_mask_m_cut = float(
            np.nanquantile(source_catalog_df["mag_ab"], SBF_PR_FALLBACK_QUANTILE)
            + SBF_PR_MLIM_OFFSET
        )
        source_mask_m_cut_method = "catalog_quantile_fallback"
    else:
        source_mask_m_cut = np.nan
        source_mask_m_cut_method = "empty_catalog"

if not np.isfinite(source_mask_m_cut):
    raise RuntimeError(
        "[SRC-CAT] cannot define one m_cut; set SBF_PR_MLIM_OVERRIDE or improve source detection"
    )

source_catalog_df["mask_brighter_than_mcut"] = (
    np.isfinite(source_catalog_df["mag_ab"])
    & (source_catalog_df["mag_ab"] <= source_mask_m_cut)
)
bright_catalog_labels = source_catalog_df.loc[
    source_catalog_df["mask_brighter_than_mcut"], "label"
].to_numpy(dtype=int)
final_catalog_source_mask_raw = np.isin(source_catalog_seg_full, bright_catalog_labels)
final_catalog_source_mask = ndimage.binary_dilation(
    final_catalog_source_mask_raw,
    iterations=SBF_PR_MASK_DILATE_ITERATIONS,
) & source_catalog_ref_mask

# Первичная маска нужна для устойчивого фита изофот. В science residual её
# заменяет catalog mask: слабые источники возвращаются в кадр и входят в P_r.
premask_isophote_fit = np.array(globals().get("premask_base", premask), dtype=bool, copy=True)
premask = np.array((~valid150) | final_catalog_source_mask, dtype=bool)
premask_src = np.array(final_catalog_source_mask, dtype=bool, copy=True)

resid_full = np.array(img - model_full, dtype=np.float32, copy=True)
resid_full[premask | (~np.isfinite(model_full)) | (~np.isfinite(img))] = np.nan
science_clip_mask = (
    (~premask) & np.isfinite(resid_full) & np.isfinite(model_full) & (model_full > 0.0)
)
if int(science_clip_mask.sum()) < MIN_PIXELS_SBF:
    raise RuntimeError("[SRC-CAT] too few science pixels after the catalog mask")
_, med_clip_catalog, std_clip_catalog = sigma_clipped_stats(
    resid_full[science_clip_mask], sigma=CLIP_SIGMA_QC, maxiters=CLIP_MAXIT_QC
)
if (not np.isfinite(std_clip_catalog)) or std_clip_catalog <= 0.0:
    raise RuntimeError(f"[SRC-CAT] invalid residual sigma after catalog mask: {std_clip_catalog}")
clip_lo_catalog = float(med_clip_catalog - CLIP_SIGMA_QC * std_clip_catalog)
clip_hi_catalog = float(med_clip_catalog + CLIP_SIGMA_QC * std_clip_catalog)
resid_full_clip = np.array(resid_full, dtype=np.float32, copy=True)
resid_full_clip[science_clip_mask] = np.clip(
    resid_full_clip[science_clip_mask], clip_lo_catalog, clip_hi_catalog
)
resid_full_clip[~science_clip_mask] = np.nan
science_model = model_full
science_resid = resid_full_clip
science_model_name = "model_full_measured_isophotes"
science_resid_name = f"resid_full_clip_{CLIP_TAG_QC}sigma_catalog_mcut"

source_catalog_build_info.update({
    "m_cut": float(source_mask_m_cut),
    "m_cut_method": source_mask_m_cut_method,
    "n_masked_sources": int(len(bright_catalog_labels)),
    "final_mask_pixels": int(final_catalog_source_mask.sum()),
})

source_catalog_csv_path = out_dir / f"{stem}_sbf_compact_source_catalog.csv"
source_catalog_mask_path = out_dir / f"{stem}_sbf_catalog_mask_mcut.fits"
source_catalog_resid_path = out_dir / f"{stem}_sbf_resid_catalog_mask_clip_{CLIP_TAG_QC}sigma.fits"
source_catalog_df.to_csv(source_catalog_csv_path, index=False)
fits.writeto(source_catalog_mask_path, np.asarray(premask, dtype=np.uint8), hdr150, overwrite=True)
fits.writeto(source_catalog_resid_path, np.asarray(science_resid, dtype=np.float32), hdr150, overwrite=True)

finite_mag = np.isfinite(source_catalog_df.get("mag_ab", pd.Series(dtype=float))).sum() if not source_catalog_df.empty else 0
print(
    f"[SRC-CAT] detect image={SBF_PR_SOURCE_IMAGE}, crop={y0_src}:{y1_src}, {x0_src}:{x1_src}, "
    f"support pixels={int(source_catalog_ref_mask.sum())}"
)
print(
    f"[SRC-CAT] residual stats in support: med={med_det:.3e}, std={std_det:.3e}, "
    f"threshold={threshold_det:.3e}"
)
if np.isfinite(premask_overlap_count):
    print(f"[SRC-CAT] old premask-overlap inside model support = {int(premask_overlap_count)} compact labels")
print(
    f"[SRC-CAT] segmentation: raw={n_detect_raw}, deblended={n_detect_deblend}, "
    f"compact-area pass={n_compact_area}, dropped_large={dropped_large_area}"
)
print(
    f"[SRC-CAT] photometry filter: kept={n_final}, dropped_nonpositive_signed_flux={dropped_nonpositive_signed_flux}, "
    f"dropped_nonfinite_mag={dropped_nonfinite_mag}, source pixels kept={source_pixels_final}"
)
print(
    f"[SRC-CAT] compact catalog size={len(source_catalog_df)}, finite mags={finite_mag}, "
    f"reference pixels={int(source_catalog_ref_mask.sum())}"
)
print(
    f"[SRC-CAT] one m_cut={source_mask_m_cut:.3f} ({source_mask_m_cut_method}); "
    f"masked catalog sources={len(bright_catalog_labels)}, "
    f"mask pixels={int(final_catalog_source_mask.sum())}"
)
print(f"[OUT] compact source catalog -> {source_catalog_csv_path}")
print(f"[OUT] final catalog mask     -> {source_catalog_mask_path}")
print(f"[OUT] final science residual -> {source_catalog_resid_path}")


[14:52:52] building one compact-source catalog, completeness cut, and final science mask...
[14:52:56] [SRC-CAT] detect image=img_minus_model_full_unmasked, crop=112:3113, 5604:7377, support pixels=3574962
[14:52:56] [SRC-CAT] residual stats in support: med=-4.621e-02, std=5.599e-01, threshold=1.489e+00
[14:52:56] [SRC-CAT] old premask-overlap inside model support = 387 compact labels
[14:52:56] [SRC-CAT] segmentation: raw=5368, deblended=5572, compact-area pass=5572, dropped_large=0
[14:52:56] [SRC-CAT] photometry filter: kept=5572, dropped_nonpositive_signed_flux=0, dropped_nonfinite_mag=0, source pixels kept=49893
[14:52:56] [SRC-CAT] compact catalog size=5572, finite mags=5572, reference pixels=3574962
[14:52:56] [SRC-CAT] one m_cut=26.608 (catalog_snr5_q95_approximation); masked catalog sources=5285, mask pixels=265897
[14:52:56] [OUT] compact source catalog -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_compact_source_catalog.csv
[14:52:56] [OUT] final catalog mask

## Поправка за неразрешённые источники $P_r$

Подгоняем две физически разные компоненты функции светимости: гауссовскую GCLF шаровых скоплений фиксированной ширины 1.2 mag и степенной закон фоновых галактик. Обе компоненты интегрируются слабее того же $m_{cut}$, которым построена маска.

> **✅ ИСПРАВЛЕНО В ПОРЯДКЕ JENSEN:** $P_r=P_{r,GC}+P_{r,bg}$, где первая компонента получена из GCLF, а вторая — из степенной функции фоновых галактик. После нормировки residual на $\sqrt{I_{gal}}$ поправка дополнительно взвешивается локальным $1/I_{gal}$.


In [ ]:
print("preparing Jensen GCLF + background-galaxy variance model...")

from scipy.optimize import curve_fit

def sbf_ab_zeropoint_from_pix_area(pix_area):
    jy_per_pix_local = MJY_SR_TO_JY_PER_ARCSEC2 * pix_area
    return float(-2.5 * np.log10(jy_per_pix_local / AB_ZEROPOINT_JY))

def sbf_flux_image_units_from_mag(mag_ab, pix_area):
    zp_ab_local = sbf_ab_zeropoint_from_pix_area(pix_area)
    return float(10.0 ** (-0.4 * (float(mag_ab) - zp_ab_local)))

def optional_finite_float(value):
    try:
        value_f = float(value)
    except (TypeError, ValueError):
        return np.nan
    return value_f if np.isfinite(value_f) else np.nan

def select_sources_in_region_mask(region_mask, region_origin_x=0, region_origin_y=0):
    if source_catalog_df.empty:
        return source_catalog_df.copy()

    region_mask = np.asarray(region_mask, dtype=bool)
    ny_mask, nx_mask = region_mask.shape

    seg_full = globals().get("source_catalog_seg_full", None)
    if isinstance(seg_full, np.ndarray) and seg_full.ndim == 2:
        y0 = int(region_origin_y)
        x0 = int(region_origin_x)
        y1 = y0 + ny_mask
        x1 = x0 + nx_mask
        if 0 <= y0 < y1 <= seg_full.shape[0] and 0 <= x0 < x1 <= seg_full.shape[1]:
            seg_view = np.asarray(seg_full[y0:y1, x0:x1], dtype=int)
            labels_in_region = np.unique(seg_view[region_mask & (seg_view > 0)])
            if labels_in_region.size == 0:
                return source_catalog_df.iloc[0:0].copy()
            keep = source_catalog_df["label"].isin(labels_in_region.astype(int))
            return source_catalog_df.loc[keep].copy().reset_index(drop=True)

    keep = []
    for idx, row in source_catalog_df.iterrows():
        xcen = row.get("xcentroid", np.nan)
        ycen = row.get("ycentroid", np.nan)
        if not (np.isfinite(xcen) and np.isfinite(ycen)):
            continue

        ix = int(np.rint(float(xcen))) - int(region_origin_x)
        iy = int(np.rint(float(ycen))) - int(region_origin_y)
        if 0 <= ix < nx_mask and 0 <= iy < ny_mask and bool(region_mask[iy, ix]):
            keep.append(idx)

    if not keep:
        return source_catalog_df.iloc[0:0].copy()
    return source_catalog_df.loc[keep].copy().reset_index(drop=True)

def summarize_sources_in_region_mask(region_mask, region_origin_x=0, region_origin_y=0):
    reg_sources = select_sources_in_region_mask(
        region_mask,
        region_origin_x=region_origin_x,
        region_origin_y=region_origin_y,
    )
    if reg_sources.empty:
        return {
            "n_overlap": 0,
            "n_valid": 0,
            "n_nonpositive_flux": 0,
            "mag_min": np.nan,
            "mag_med": np.nan,
            "mag_max": np.nan,
        }

    flux = reg_sources["flux_jy"].to_numpy(dtype=float) if "flux_jy" in reg_sources else np.full(len(reg_sources), np.nan)
    mags = reg_sources["mag_ab"].to_numpy(dtype=float) if "mag_ab" in reg_sources else np.full(len(reg_sources), np.nan)
    valid = np.isfinite(flux) & (flux > SBF_PR_CATALOG_MIN_FLUX_JY) & np.isfinite(mags)
    mags_valid = mags[valid]

    return {
        "n_overlap": int(len(reg_sources)),
        "n_valid": int(np.count_nonzero(valid)),
        "n_nonpositive_flux": int(np.count_nonzero(~(np.isfinite(flux) & (flux > SBF_PR_CATALOG_MIN_FLUX_JY)))),
        "mag_min": float(np.nanmin(mags_valid)) if mags_valid.size > 0 else np.nan,
        "mag_med": float(np.nanmedian(mags_valid)) if mags_valid.size > 0 else np.nan,
        "mag_max": float(np.nanmax(mags_valid)) if mags_valid.size > 0 else np.nan,
    }

def estimate_turnover_magnitude(mags, mag_bin=None):
    if mag_bin is None:
        mag_bin = SBF_PR_MAG_BIN
    arr = np.asarray(mags, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size < 3:
        return np.nan, {"method": "too_few_sources", "n_sources": int(arr.size)}

    lo = float(np.floor(np.nanmin(arr) / mag_bin) * mag_bin)
    hi = float(np.ceil(np.nanmax(arr) / mag_bin) * mag_bin + mag_bin)
    edges = np.arange(lo, hi + 0.5 * mag_bin, mag_bin)
    if edges.size < 3:
        return np.nan, {"method": "bad_edges", "n_sources": int(arr.size)}

    counts, edges = np.histogram(arr, bins=edges)
    if not np.any(counts > 0):
        return np.nan, {"method": "empty_hist", "n_sources": int(arr.size)}

    idx_peak = int(np.argmax(counts))
    m_turn = float(0.5 * (edges[idx_peak] + edges[idx_peak + 1]))
    return m_turn, {
        "method": "turnover_hist",
        "n_sources": int(arr.size),
        "peak_count": int(counts[idx_peak]),
        "n_bins": int(counts.size),
    }



GCLF_SIGMA_MAG = 1.2
PR_FAINT_INTEGRATION_SPAN_MAG = 12.0

def fit_jensen_contaminant_lf(mags, area_pix, m_lim, mag_bin=None):
    """Fit Gaussian GCLF plus a background-galaxy power law."""
    if mag_bin is None:
        mag_bin = SBF_PR_MAG_BIN
    arr = np.asarray(mags, dtype=float)
    arr = arr[np.isfinite(arr)]
    area_pix = float(area_pix)
    out = {
        "lf_model": "gaussian_gclf_plus_background_powerlaw",
        "lf_gamma": np.nan,
        "lf_phi_mlim": np.nan,
        "lf_fit_method": "unavailable",
        "lf_fit_rmse": np.nan,
        "lf_n_fit_bins": 0,
        "lf_n_fit_sources": 0,
        "lf_fit_mbright": np.nan,
        "lf_fit_mfaint": float(m_lim) if np.isfinite(m_lim) else np.nan,
        "gclf_sigma_mag": float(GCLF_SIGMA_MAG),
        "gclf_turnover_mag": np.nan,
        "gclf_phi_peak": np.nan,
        "background_phi_mcut": np.nan,
    }
    if arr.size < 5 or area_pix <= 0.0 or not np.isfinite(m_lim):
        out["lf_fit_method"] = "too_few_sources_or_bad_area"
        return out

    fit_mbright = float(max(np.nanmin(arr), m_lim - 5.0))
    fit_arr = arr[(arr >= fit_mbright) & (arr <= m_lim)]
    out["lf_fit_mbright"] = fit_mbright
    out["lf_n_fit_sources"] = int(fit_arr.size)
    edges = np.arange(fit_mbright, m_lim + 1.01 * mag_bin, mag_bin)
    if fit_arr.size < 5 or edges.size < 4:
        out["lf_fit_method"] = "too_few_complete_sources"
        return out

    counts, edges = np.histogram(fit_arr, bins=edges)
    centers = 0.5 * (edges[:-1] + edges[1:])
    positive = counts > 0
    x = centers[positive]
    y = counts[positive] / (area_pix * mag_bin)
    yerr = np.sqrt(counts[positive]) / (area_pix * mag_bin)
    out["lf_n_fit_bins"] = int(x.size)
    if x.size < max(4, SBF_PR_MIN_FIT_BINS):
        out["lf_fit_method"] = "too_few_nonempty_bins"
        return out

    gamma_min, gamma_max = map(float, SBF_PR_GAMMA_BOUNDS)
    gamma0 = float(np.clip(SBF_PR_DEFAULT_GAMMA, gamma_min, gamma_max))
    m_gc0 = float(np.nanmedian(fit_arr))
    phi0 = float(max(np.nanmax(y), ROBUST_SCALE_FLOOR))
    phi_bg0 = float(max(y[-1] * 0.5, ROBUST_SCALE_FLOOR))

    def lf_model(mag, phi_gc, m_gc, phi_bg_mcut, gamma_bg):
        gclf = phi_gc * np.exp(-0.5 * ((mag - m_gc) / GCLF_SIGMA_MAG) ** 2)
        background = phi_bg_mcut * 10.0 ** (gamma_bg * (mag - m_lim))
        return gclf + background

    try:
        popt, _ = curve_fit(
            lf_model,
            x,
            y,
            p0=(phi0, m_gc0, phi_bg0, gamma0),
            sigma=np.maximum(yerr, ROBUST_SCALE_FLOOR),
            absolute_sigma=True,
            bounds=(
                (0.0, fit_mbright - 2.0, 0.0, gamma_min),
                (np.inf, m_lim + 3.0, np.inf, gamma_max),
            ),
            maxfev=50000,
        )
        phi_gc, m_gc, phi_bg_mcut, gamma_bg = map(float, popt)
        fit_method = "poisson_weighted_gclf_plus_background"
    except Exception:
        # Раздельные формы сохраняются даже в fallback: фиксируем m_GC и gamma,
        # затем решаем только две неотрицательные нормировки.
        m_gc = m_gc0
        gamma_bg = gamma0
        design = np.column_stack([
            np.exp(-0.5 * ((x - m_gc) / GCLF_SIGMA_MAG) ** 2),
            10.0 ** (gamma_bg * (x - m_lim)),
        ])
        coeff = np.linalg.lstsq(design, y, rcond=None)[0]
        phi_gc, phi_bg_mcut = np.maximum(coeff, 0.0)
        phi_gc = float(phi_gc)
        phi_bg_mcut = float(phi_bg_mcut)
        fit_method = "fixed_shape_nonnegative_fallback"

    y_fit = lf_model(x, phi_gc, m_gc, phi_bg_mcut, gamma_bg)
    rmse = float(np.sqrt(np.nanmean((y - y_fit) ** 2)))
    out.update({
        "lf_gamma": gamma_bg,
        "lf_phi_mlim": phi_bg_mcut,
        "lf_fit_method": fit_method,
        "lf_fit_rmse": rmse,
        "gclf_turnover_mag": m_gc,
        "gclf_phi_peak": phi_gc,
        "background_phi_mcut": phi_bg_mcut,
    })
    return out

def integrate_pr_from_jensen_lf(fit, m_lim, pix_area, mean_inverse_model):
    required = (
        fit.get("gclf_phi_peak", np.nan),
        fit.get("gclf_turnover_mag", np.nan),
        fit.get("background_phi_mcut", np.nan),
        fit.get("lf_gamma", np.nan),
        m_lim,
        mean_inverse_model,
    )
    if not np.all(np.isfinite(required)) or mean_inverse_model <= 0.0:
        return {"Pr": np.nan, "Pr_gc": np.nan, "Pr_background": np.nan}

    mag_grid = np.linspace(float(m_lim), float(m_lim + PR_FAINT_INTEGRATION_SPAN_MAG), 4097)
    phi_gc = fit["gclf_phi_peak"] * np.exp(
        -0.5 * ((mag_grid - fit["gclf_turnover_mag"]) / GCLF_SIGMA_MAG) ** 2
    )
    phi_bg = fit["background_phi_mcut"] * 10.0 ** (
        fit["lf_gamma"] * (mag_grid - m_lim)
    )
    zp_ab_local = sbf_ab_zeropoint_from_pix_area(pix_area)
    flux_img = 10.0 ** (-0.4 * (mag_grid - zp_ab_local))
    pr_gc = float(np.trapezoid(phi_gc * flux_img ** 2, mag_grid) * mean_inverse_model)
    pr_bg = float(np.trapezoid(phi_bg * flux_img ** 2, mag_grid) * mean_inverse_model)
    return {"Pr": pr_gc + pr_bg, "Pr_gc": pr_gc, "Pr_background": pr_bg}

def build_global_pr_reference():
    m_lim = float(source_mask_m_cut)
    ref_sources = select_sources_in_region_mask(source_catalog_ref_mask)
    valid_sources = ref_sources[
        np.isfinite(ref_sources["mag_ab"])
        & np.isfinite(ref_sources["flux_jy"])
        & (ref_sources["flux_jy"] > SBF_PR_CATALOG_MIN_FLUX_JY)
    ].copy()
    fit = fit_jensen_contaminant_lf(
        valid_sources["mag_ab"].to_numpy(dtype=float),
        area_pix=float(source_catalog_ref_mask.sum()),
        m_lim=m_lim,
    )
    norm_mask = source_catalog_ref_mask & (~premask) & np.isfinite(model_full) & (model_full > 0.0)
    mean_inverse_model = float(np.nanmean(1.0 / model_full[norm_mask])) if np.any(norm_mask) else np.nan
    pr_parts = integrate_pr_from_jensen_lf(fit, m_lim, pix_area, mean_inverse_model)
    return {
        "region": "global_science_reference",
        "n_detected_sources": int(len(valid_sources)),
        "area_pix": int(source_catalog_ref_mask.sum()),
        "m_lim": m_lim,
        "m_lim_method": source_mask_m_cut_method,
        "mean_inverse_model": mean_inverse_model,
        **fit,
        **pr_parts,
    }

def estimate_pr_for_region(region_mask, model, region_name="region", region_origin_x=0, region_origin_y=0):
    if not SBF_PR_ENABLE:
        return {
            "Pr": 0.0, "Pr_gc": 0.0, "Pr_background": 0.0,
            "Pr_over_P0": np.nan, "n_detected_sources": 0,
            "n_detected_sources_total": 0, "m_lim": source_mask_m_cut,
            "m_lim_method": "disabled", "lf_model": "disabled",
            "lf_gamma": np.nan, "lf_phi_mlim": np.nan,
            "lf_fit_method": "disabled", "lf_fit_rmse": np.nan,
            "lf_n_fit_bins": 0, "lf_n_fit_sources": 0,
            "lf_fit_mbright": np.nan, "lf_fit_mfaint": np.nan,
            "pr_note": "Pr correction disabled",
        }

    catalog_region = np.asarray(region_mask, dtype=bool) & np.isfinite(model) & (model > 0.0)
    analysis_region = catalog_region & (~premask)
    if int(analysis_region.sum()) < MIN_PIXELS_SBF:
        return {
            "Pr": np.nan, "Pr_gc": np.nan, "Pr_background": np.nan,
            "Pr_over_P0": np.nan, "n_detected_sources": 0,
            "n_detected_sources_total": 0, "m_lim": source_mask_m_cut,
            "m_lim_method": source_mask_m_cut_method, "lf_model": "unavailable",
            "lf_gamma": np.nan, "lf_phi_mlim": np.nan,
            "lf_fit_method": "too_few_analysis_pixels", "lf_fit_rmse": np.nan,
            "lf_n_fit_bins": 0, "lf_n_fit_sources": 0,
            "lf_fit_mbright": np.nan, "lf_fit_mfaint": np.nan,
            "pr_note": "too few analysis pixels",
        }

    reg_sources = select_sources_in_region_mask(
        catalog_region,
        region_origin_x=region_origin_x,
        region_origin_y=region_origin_y,
    )
    reg_valid = reg_sources[
        np.isfinite(reg_sources["mag_ab"])
        & np.isfinite(reg_sources["flux_jy"])
        & (reg_sources["flux_jy"] > SBF_PR_CATALOG_MIN_FLUX_JY)
    ].copy()
    fit = fit_jensen_contaminant_lf(
        reg_valid["mag_ab"].to_numpy(dtype=float),
        area_pix=float(catalog_region.sum()),
        m_lim=float(source_mask_m_cut),
    )
    pr_note = "local GCLF + background-galaxy LF"
    if not np.isfinite(fit.get("gclf_phi_peak", np.nan)):
        global_ref = globals().get("sbf_pr_global_reference", {})
        fit = {key: global_ref.get(key, value) for key, value in fit.items()}
        pr_note = "global GCLF + background-galaxy LF fallback"

    mean_inverse_model = float(np.nanmean(1.0 / model[analysis_region]))
    pr_parts = integrate_pr_from_jensen_lf(
        fit, float(source_mask_m_cut), pix_area, mean_inverse_model
    )
    return {
        "Pr_over_P0": np.nan,
        "n_detected_sources": int(len(reg_valid)),
        "n_detected_sources_total": int(len(reg_sources)),
        "m_lim": float(source_mask_m_cut),
        "m_lim_method": source_mask_m_cut_method,
        "mean_inverse_model": mean_inverse_model,
        **fit,
        **pr_parts,
        "pr_note": pr_note,
    }

def solve_sbf_power_budget(P0, Imean, Pr, pix_area, P0_fit_sigma=np.nan):
    out = {
        "measurement_ok": False,
        "failure_reason": "",
        "P0": float(P0) if np.isfinite(P0) else np.nan,
        "Pr": float(Pr) if np.isfinite(Pr) else np.nan,
        "P_fluc": np.nan,
        "Pr_over_P0": np.nan,
        "Imean": float(Imean) if np.isfinite(Imean) else np.nan,
        "Pf_spec_raw": np.nan,
        "Pf_spec": np.nan,
        "mbar_spec_raw": np.nan,
        "mbar_spec": np.nan,
        "Pf_spec_sigma_formal": np.nan,
        "Pf_spec_sigma_formal_raw": np.nan,
        "mbar_fit_sigma": np.nan,
        "mbar_fit_sigma_raw": np.nan,
    }

    if not (np.isfinite(P0) and P0 > 0.0):
        out["failure_reason"] = f"bad raw P0={P0}"
        return out
    if not (np.isfinite(Imean) and Imean > 0.0):
        out["failure_reason"] = f"bad Imean={Imean}"
        return out

    zp_ab_local = sbf_ab_zeropoint_from_pix_area(pix_area)
    # residual уже нормирован на sqrt(I_gal), поэтому P0 имеет единицы
    # флуктуационного потока; повторно делить его на Imean нельзя.
    Pf_spec_raw = float(P0)
    out["Pf_spec_raw"] = Pf_spec_raw
    if np.isfinite(Pf_spec_raw) and Pf_spec_raw > 0.0:
        out["mbar_spec_raw"] = float(-2.5 * np.log10(Pf_spec_raw) + zp_ab_local)
        if np.isfinite(P0_fit_sigma) and P0_fit_sigma > 0.0:
            out["Pf_spec_sigma_formal_raw"] = float(Pf_spec_raw * P0_fit_sigma / P0)
            out["mbar_fit_sigma_raw"] = float((2.5 / np.log(10.0)) * P0_fit_sigma / P0)

    if not np.isfinite(Pr):
        out["failure_reason"] = "Pr is not finite"
        return out

    P_fluc = float(P0 - Pr)
    out["P_fluc"] = P_fluc
    out["Pr_over_P0"] = float(Pr / P0) if np.isfinite(Pr) else np.nan
    if not (np.isfinite(P_fluc) and P_fluc > 0.0):
        out["failure_reason"] = f"non-positive corrected power: P0={P0:.3e}, Pr={Pr:.3e}, P_fluc={P_fluc:.3e}"
        return out

    Pf_spec = float(P_fluc)
    out["Pf_spec"] = Pf_spec
    if not (np.isfinite(Pf_spec) and Pf_spec > 0.0):
        out["failure_reason"] = f"bad corrected Pf_spec={Pf_spec}"
        return out

    out["mbar_spec"] = float(-2.5 * np.log10(Pf_spec) + zp_ab_local)
    if np.isfinite(P0_fit_sigma) and P0_fit_sigma > 0.0:
        out["Pf_spec_sigma_formal"] = float(Pf_spec * P0_fit_sigma / P_fluc)
        out["mbar_fit_sigma"] = float((2.5 / np.log(10.0)) * P0_fit_sigma / P_fluc)

    out["measurement_ok"] = True
    return out

sbf_pr_global_reference = build_global_pr_reference()
print(
    f"[Pr-GLOBAL] Nsrc={sbf_pr_global_reference.get('n_detected_sources', 0)}, "
    f"m_lim={sbf_pr_global_reference.get('m_lim', np.nan):.3f}, "
    f"gamma={sbf_pr_global_reference.get('lf_gamma', np.nan):.3f}, "
    f"Pr={sbf_pr_global_reference.get('Pr', np.nan):.3e}, "
    f"fit={sbf_pr_global_reference.get('lf_fit_method', 'n/a')}"
)


[14:52:56] preparing Jensen GCLF + background-galaxy variance model...
[14:52:56] [Pr-GLOBAL] Nsrc=5572, m_lim=26.608, gamma=0.100, Pr=1.352e-05, fit=poisson_weighted_gclf_plus_background


## Сохранение модели и residual

Сохраняем FITS-продукты гладкой модели и residual для визуальной проверки. Это основные файлы, которые надо смотреть глазами после запуска чистого пайплайна.


In [23]:
model_path = out_dir / f"{stem}_sbf_model.fits"
resid_path = out_dir / f"{stem}_sbf_resid.fits"

fits.writeto(model_path, model, hdr150, overwrite=True)
fits.writeto(resid_path, resid, hdr150, overwrite=True)

print(f"[OUT] model → {model_path}")
print(f"[OUT] resid → {resid_path}")


[14:52:56] [OUT] model → data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_model.fits
[14:52:56] [OUT] resid → data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_resid.fits


## Helper спектрального SBF-фита

Определяем FFT-based процедуру SBF-измерения. Она строит expectation spectrum из PSF и smooth model, фитит `P(k) = P0 E(k) + P1`, оценивает unresolved-source correction и возвращает raw/corrected fluctuation magnitudes.

> **✅ ИСПРАВЛЕНО В ПОРЯДКЕ JENSEN:** residual заранее нормируется на $\sqrt{I_{gal}}$, поэтому модель яркости входит в измерение до FFT. Радиальные мощности считаются средними, фит $P(k)=P_0E(k)+P_1$ взвешивается по ошибкам средних, моды $k<10$ всегда исключаются. Несколько дополнительных `kmin` остаются только проверкой устойчивости.


## PSF для expectation spectrum

Строим библиотеку detector-sampled PSF через локальный `stpsf`. WSS OPD выбираются по близости времени измерения волнового фронта к `DATE-AVG/DATE-OBS` научного кадра; при наличии `PSFREF` в библиотеку также добавляются эмпирические PSF. Каждый вариант отдельно нормируется и ниже даёт отдельное SBF-измерение.

> **✅ ИСПРАВЛЕНО В ПОРЯДКЕ JENSEN:** используется несколько ближайших по дате модельных PSF и, если они заданы, эмпирические PSF. Разброс результатов между PSF включается в ошибку; время изменения файла больше не имеет физического смысла и нигде не используется.


In [24]:
print("loading/building PSF...")

import os

science_psf_file = f150w_path
psf_library_fits_path = out_dir / f"{stem}_psf_{PSF_SIZE}.fits"
print(f"[PSF] library output target: {psf_library_fits_path}")

stpsf_data_dir = Path(os.environ.get("STPSF_PATH", Path.home() / "data" / "stpsf-data")).expanduser().resolve()
if not stpsf_data_dir.exists():
    raise RuntimeError(f"[PSF] local STPSF data dir not found: {stpsf_data_dir}")
os.environ["STPSF_PATH"] = str(stpsf_data_dir)
print(f"[PSF] STPSF_PATH={stpsf_data_dir}")

from astropy.time import Time

science_hdr = fits.getheader(str(science_psf_file), 0)
science_hdr_sci = fits.getheader(str(science_psf_file), "SCI")
expected_signal_filter = str(globals().get("SIGNAL_FILTER", "F150W")).strip().upper()
actual_signal_filter = str(science_hdr.get("FILTER", "")).strip().upper()
if actual_signal_filter != expected_signal_filter:
    raise RuntimeError(
        f"[PSF] signal FITS filter mismatch: expected {expected_signal_filter}, "
        f"header has {actual_signal_filter}"
    )
if str(science_hdr.get("INSTRUME", "")).strip().upper() != "NIRCAM":
    raise RuntimeError(f"[PSF] unsupported signal instrument: {science_hdr.get('INSTRUME')}")

def _physical_time_from_header(header):
    for key in ("MJD-AVG", "MJD-BEG", "MJD-OBS"):
        value = header.get(key)
        if value is not None:
            try:
                return Time(float(value), format="mjd", scale="utc")
            except Exception:
                pass
    for key in ("DATE-AVG", "DATE-BEG", "DATE-OBS", "DATE"):
        value = header.get(key)
        if value:
            text_value = str(value)
            if key == "DATE-OBS" and "T" not in text_value and header.get("TIME-OBS"):
                text_value = f"{text_value}T{header['TIME-OBS']}"
            try:
                return Time(text_value, scale="utc")
            except Exception:
                pass
    return None

science_time = _physical_time_from_header(science_hdr_sci)
if science_time is None:
    science_time = _physical_time_from_header(science_hdr)
if science_time is None:
    raise RuntimeError("[PSF] science observation time is absent from FITS headers")

local_wss_dir = Path(
    os.environ.get(
        "SBF_WSS_OPD_DIR",
        f150w_path.resolve().parent.parent / "wss_opd",
    )
).expanduser().resolve()
PSF_MAX_OPD_DELTA_DAYS = float(os.environ.get("SBF_PSF_MAX_OPD_DELTA_DAYS", "7.0"))
print(f"[PSF] project WSS OPD directory={local_wss_dir}")
print(f"[PSF] maximum OPD date separation={PSF_MAX_OPD_DELTA_DAYS:.1f} d")

def _stpsf_ascii_opd_argument(opd_path):
    """Give STPSF an ASCII path; it copies this string into a FITS HISTORY card."""
    absolute = Path(opd_path).resolve()
    relative = Path(os.path.relpath(absolute, Path.cwd()))
    for candidate in (relative, absolute):
        text = str(candidate)
        try:
            text.encode("ascii")
        except UnicodeEncodeError:
            continue
        if Path(text).is_file():
            return text
    raise RuntimeError(
        f"[PSF] STPSF cannot ingest OPD through a non-ASCII path: {absolute}; "
        "run the notebook from the project root or code directory"
    )
local_wss_files = sorted(local_wss_dir.glob("*.fits"))
if not local_wss_files:
    raise RuntimeError(f"[PSF] no local WSS OPD files found in {local_wss_dir}")
dated_wss = []
rejected_wss = []
for opd_path in local_wss_files:
    try:
        if opd_path.stat().st_size <= 0 or opd_path.stat().st_size % 2880:
            raise ValueError("size is not a complete FITS block")
        with fits.open(opd_path, mode="readonly", memmap=True) as opd_hdul:
            opd_hdul.verify("exception")
            if "RESULT_PHASE" not in opd_hdul or not opd_hdul["RESULT_PHASE"].shape:
                raise ValueError("RESULT_PHASE extension is absent or empty")
            opd_header = opd_hdul[0].header.copy()
        opd_time = _physical_time_from_header(opd_header)
        if opd_time is None:
            raise ValueError("physical observation time is absent")
        signed_delta_days = float((opd_time - science_time).to_value("day"))
        dated_wss.append((abs(signed_delta_days), signed_delta_days, opd_path, opd_time))
    except Exception as exc:
        rejected_wss.append((opd_path, repr(exc)))
for rejected_path, rejected_reason in rejected_wss:
    print(f"[PSF][WARN] rejected local OPD {rejected_path.name}: {rejected_reason}")
if not dated_wss:
    raise RuntimeError("[PSF] local OPD files have no physical observation timestamps")
dated_wss.sort(key=lambda row: row[0])
eligible_wss = [row for row in dated_wss if row[0] <= PSF_MAX_OPD_DELTA_DAYS]
if not eligible_wss:
    nearest_delta, _, nearest_path, _ = dated_wss[0]
    raise RuntimeError(
        f"[PSF] no time-matched WSS OPD for {TARGET_GALAXY}: nearest "
        f"{nearest_path.name} is {nearest_delta:.2f} d away, limit is "
        f"{PSF_MAX_OPD_DELTA_DAYS:.2f} d"
    )
PSF_NEAREST_OPD_COUNT = min(3, len(eligible_wss))
local_wss_opds = [row[2] for row in eligible_wss[:PSF_NEAREST_OPD_COUNT]]
local_wss_opd = local_wss_opds[0]
print(
    "[PSF] nearest physical OPDs: "
    + ", ".join(
        f"{path.name} (dt={delta_days:.2f} d)"
        for delta_days, _, path, _ in eligible_wss[:PSF_NEAREST_OPD_COUNT]
    )
)

print(
    f"[PSF] science meta: INSTRUME={science_hdr.get('INSTRUME')} DETECTOR={science_hdr.get('DETECTOR')} "
    f"FILTER={science_hdr.get('FILTER')} PUPIL={science_hdr.get('PUPIL')} APERNAME={science_hdr.get('APERNAME')}"
)
print(f"[PSF] science date: {science_hdr.get('DATE-OBS')}T{science_hdr.get('TIME-OBS')}")
opd_hdr = fits.getheader(str(local_wss_opd), 0)
opd_time_selected = _physical_time_from_header(opd_hdr)
opd_signed_delta_days = float((opd_time_selected - science_time).to_value("day"))
opd_delta_days = abs(opd_signed_delta_days)
opd_corr_id = str(opd_hdr.get("CORR_ID", local_wss_opd.stem))
print(
    f"[PSF] OPD meta: CORR_ID={opd_hdr.get('CORR_ID', 'NA')} DATE-OBS={opd_hdr.get('DATE-OBS', opd_hdr.get('DATE', 'NA'))} "
    f"CONTENTS={opd_hdr.get('CONTENTS', 'NA')}"
)

sim = stpsf.instrument(science_hdr['INSTRUME'])
if (sim.name == 'NIRCam') and (science_hdr['PUPIL'][0] == 'F') and (science_hdr['PUPIL'][-1] in ['N', 'M']):
    sim.filter = science_hdr['PUPIL']
else:
    sim.filter = science_hdr['FILTER']
sim.set_position_from_aperture_name(science_hdr['APERNAME'])
if (sim.name == 'NIRCam') and (science_hdr.get('PUPIL', 'CLEAR') != 'CLEAR') and (not science_hdr['PUPIL'].startswith('F')) and (not science_hdr['PUPIL'].startswith('MASK')):
    sim.pupil_mask = science_hdr['PUPIL']
print(
    f"[PSF] configured local sim: instrument={sim.name} filter={sim.filter} detector={sim.detector} "
    f"apername={sim.aperturename} det_pos={sim.detector_position}"
)

sim.load_wss_opd(_stpsf_ascii_opd_argument(local_wss_opd), verbose=True, plot=False)
sim.options['output_mode'] = 'both'
psf_hdul = sim.calc_psf(
    nlambda=PSF_NLAMBDA,
    fov_pixels=PSF_SIZE,
    fft_oversample=4,
    detector_oversample=1,
    add_distortion=True,
)

def _psf_array_from_hdu(hdu):
    arr = np.array(hdu.data, dtype=float)
    if arr.ndim == 3:
        arr = arr.sum(axis=0)
    return arr

def _select_detector_sampled_psf(psf_product):
    if isinstance(psf_product, fits.HDUList):
        for extname in ("DET_DIST", "DET_SAMP"):
            try:
                return _psf_array_from_hdu(psf_product[extname]), extname
            except Exception:
                pass
        for index, hdu in enumerate(psf_product):
            if getattr(hdu, "data", None) is None:
                continue
            candidate = _psf_array_from_hdu(hdu)
            if candidate.shape == (PSF_SIZE, PSF_SIZE):
                return candidate, hdu.header.get("EXTNAME", f"EXT{index}")
        return _psf_array_from_hdu(psf_product[0]), "PRIMARY_FALLBACK"
    if isinstance(psf_product, fits.PrimaryHDU):
        return _psf_array_from_hdu(psf_product), "PRIMARY"
    return np.asarray(psf_product, dtype=float), "ARRAY"

def _normalize_psf_array(array, label):
    array = np.nan_to_num(np.asarray(array, dtype=float), nan=0.0)
    total = float(array.sum())
    if not np.isfinite(total) or total <= 0.0:
        raise RuntimeError(f"[PSF] invalid normalization for {label}: sum={total}")
    return array / total

def _center_psf_to_size(array, size):
    array = np.asarray(array, dtype=float)
    output = np.zeros((size, size), dtype=float)
    src_y0 = max(0, (array.shape[0] - size) // 2)
    src_x0 = max(0, (array.shape[1] - size) // 2)
    src = array[src_y0:src_y0 + size, src_x0:src_x0 + size]
    dst_y0 = max(0, (size - src.shape[0]) // 2)
    dst_x0 = max(0, (size - src.shape[1]) // 2)
    output[dst_y0:dst_y0 + src.shape[0], dst_x0:dst_x0 + src.shape[1]] = src
    return output

psf_selected_ext = None
if isinstance(psf_hdul, fits.HDUList):
    ext_rows = []
    for i, hdu in enumerate(psf_hdul):
        if getattr(hdu, 'data', None) is None:
            continue
        arr = np.array(hdu.data)
        extname = hdu.header.get('EXTNAME', 'PRIMARY' if i == 0 else f'EXT{i}')
        ext_rows.append(
            (
                i,
                extname,
                arr.shape,
                hdu.header.get('OVERSAMP', 'NA'),
                hdu.header.get('DET_SAMP', 'NA'),
                hdu.header.get('PIXELSCL', 'NA'),
            )
        )
    print('[PSF] returned HDU summary:')
    for row in ext_rows:
        print(f"    idx={row[0]} ext={row[1]} shape={row[2]} OVERSAMP={row[3]} DET_SAMP={row[4]} PIXELSCL={row[5]}")

    for extname in ('DET_DIST', 'DET_SAMP'):
        try:
            psf = _psf_array_from_hdu(psf_hdul[extname])
            psf_selected_ext = extname
            break
        except Exception:
            pass
    else:
        detector_like = []
        for i, hdu in enumerate(psf_hdul):
            if getattr(hdu, 'data', None) is None:
                continue
            arr = _psf_array_from_hdu(hdu)
            if arr.shape == (PSF_SIZE, PSF_SIZE):
                detector_like.append((hdu.header.get('EXTNAME', f'EXT{i}'), arr))
        if detector_like:
            psf_selected_ext, psf = detector_like[0]
        else:
            psf_selected_ext = 'PRIMARY_FALLBACK'
            psf = _psf_array_from_hdu(psf_hdul[0])
elif isinstance(psf_hdul, fits.PrimaryHDU):
    psf_selected_ext = 'PRIMARY'
    psf = _psf_array_from_hdu(psf_hdul)
else:
    psf_selected_ext = 'ARRAY'
    psf = np.array(psf_hdul, dtype=float)

print(f"[PSF] selected extension = {psf_selected_ext}")
science_pixel_scale_arcsec = float(np.sqrt(pix_area))
psf_pixel_scale_arcsec = np.nan
if isinstance(psf_hdul, fits.HDUList):
    try:
        psf_pixel_scale_arcsec = float(psf_hdul[psf_selected_ext].header.get("PIXELSCL"))
    except Exception:
        pass
psf_scale_rel_error = (
    abs(psf_pixel_scale_arcsec / science_pixel_scale_arcsec - 1.0)
    if np.isfinite(psf_pixel_scale_arcsec) and science_pixel_scale_arcsec > 0.0
    else np.nan
)
psf_native_scale_rel_error = psf_scale_rel_error
if (not np.isfinite(psf_scale_rel_error)) or psf_scale_rel_error > 0.01:
    raise RuntimeError(
        f"[PSF] detector-scale mismatch: science={science_pixel_scale_arcsec:.8f} "
        f"arcsec/pix, PSF={psf_pixel_scale_arcsec}, rel={psf_scale_rel_error}"
    )
psf_method_id = "stpsf_local_wss_time_and_field_ensemble_v1"
psf_method_limitations = (
    "local STPSF model ensemble; nearest time-matched WSS OPDs plus four "
    "detector-position offsets; no empirical stellar PSF unless PSFREF is set"
)
psf_detector_set = str(sim.detector)
psf = _normalize_psf_array(psf, f"nearest OPD {local_wss_opd.name}")

if psf.shape != (PSF_SIZE, PSF_SIZE):
    print(
        f"[PSF][WARN] detector-scale selection still gave shape={psf.shape} for requested fov_pixels={PSF_SIZE}. "
        "E(k) may still be using the wrong sampling."
    )
print(f"[PSF] final shape={psf.shape}, sum={psf.sum():.6f}, min={psf.min():.3e}, max={psf.max():.3e}")

psf_library = [{
    "id": f"stpsf_{local_wss_opd.stem}",
    "kind": "model",
    "opd_path": str(local_wss_opd),
    "opd_corr_id": opd_corr_id,
    "opd_delta_days": opd_delta_days,
    "selected_extension": psf_selected_ext,
    "filter": actual_signal_filter,
    "aperture": str(science_hdr.get("APERNAME", "")),
    "detector_position": tuple(sim.detector_position),
    "array": psf,
}]

for extra_opd in local_wss_opds[1:]:
    sim.load_wss_opd(_stpsf_ascii_opd_argument(extra_opd), verbose=False, plot=False)
    extra_product = sim.calc_psf(
        nlambda=PSF_NLAMBDA,
        fov_pixels=PSF_SIZE,
        fft_oversample=4,
        detector_oversample=1,
        add_distortion=True,
    )
    extra_array, extra_ext = _select_detector_sampled_psf(extra_product)
    extra_array = _normalize_psf_array(extra_array, f"OPD {extra_opd.name}")
    if extra_array.shape != (PSF_SIZE, PSF_SIZE):
        raise RuntimeError(
            f"[PSF] wrong detector sampling for {extra_opd.name}: {extra_array.shape}"
        )
    extra_opd_hdr = fits.getheader(str(extra_opd), 0)
    extra_opd_time = _physical_time_from_header(extra_opd_hdr)
    psf_library.append({
        "id": f"stpsf_{extra_opd.stem}",
        "kind": "model",
        "opd_path": str(extra_opd),
        "opd_corr_id": str(extra_opd_hdr.get("CORR_ID", extra_opd.stem)),
        "opd_delta_days": abs(float((extra_opd_time - science_time).to_value("day"))),
        "selected_extension": extra_ext,
        "filter": actual_signal_filter,
        "aperture": str(science_hdr.get("APERNAME", "")),
        "detector_position": tuple(sim.detector_position),
        "array": extra_array,
    })

# В дополнение к временной выборке OPD оцениваем полевую вариацию ближайшей PSF.
nominal_detector_position = tuple(sim.detector_position)
PSF_FIELD_OFFSETS_PIX = ((256, 0), (-256, 0), (0, 256), (0, -256))
sim.load_wss_opd(_stpsf_ascii_opd_argument(local_wss_opd), verbose=False, plot=False)
for dx_psf, dy_psf in PSF_FIELD_OFFSETS_PIX:
    x_psf_field = int(np.clip(nominal_detector_position[0] + dx_psf, 0, 2047))
    y_psf_field = int(np.clip(nominal_detector_position[1] + dy_psf, 0, 2047))
    if (x_psf_field, y_psf_field) == nominal_detector_position:
        continue
    sim.detector_position = (x_psf_field, y_psf_field)
    field_product = sim.calc_psf(
        nlambda=PSF_NLAMBDA,
        fov_pixels=PSF_SIZE,
        fft_oversample=4,
        detector_oversample=1,
        add_distortion=True,
    )
    field_array, field_ext = _select_detector_sampled_psf(field_product)
    field_array = _normalize_psf_array(
        field_array, f"field position {x_psf_field},{y_psf_field}"
    )
    if field_array.shape != (PSF_SIZE, PSF_SIZE):
        raise RuntimeError(
            f"[PSF] wrong field PSF sampling at {(x_psf_field, y_psf_field)}: "
            f"{field_array.shape}"
        )
    psf_library.append({
        "id": f"stpsf_field_{x_psf_field}_{y_psf_field}",
        "kind": "model_field_variation",
        "opd_path": str(local_wss_opd),
        "opd_corr_id": opd_corr_id,
        "opd_delta_days": opd_delta_days,
        "selected_extension": field_ext,
        "filter": actual_signal_filter,
        "aperture": str(science_hdr.get("APERNAME", "")),
        "detector_position": (x_psf_field, y_psf_field),
        "array": field_array,
    })
sim.detector_position = nominal_detector_position

if PSFREF is not None:
    empirical_paths = PSFREF if isinstance(PSFREF, (list, tuple)) else [PSFREF]
    for empirical_path in empirical_paths:
        empirical_path = Path(empirical_path)
        empirical_array = fits.getdata(empirical_path)
        if np.ndim(empirical_array) == 3:
            empirical_array = np.nansum(empirical_array, axis=0)
        empirical_array = _center_psf_to_size(empirical_array, PSF_SIZE)
        empirical_array = _normalize_psf_array(empirical_array, empirical_path.name)
        psf_library.append({
            "id": f"empirical_{empirical_path.stem}",
            "kind": "empirical",
            "opd_path": "",
            "array": empirical_array,
        })

# Отдельная F090W PSF нужна только для согласования цветовой фотометрии.
color_psf_f150 = np.asarray(psf_library[0]["array"], dtype=float)
color_psf_f090 = None
if img_f090 is not None and hdr090 is not None:
    science_hdr090 = fits.getheader(str(f090w_path), 0)
    science_hdr090_sci = fits.getheader(str(f090w_path), "SCI")
    expected_color_filter = str(globals().get("COLOR_FILTER", "F090W")).strip().upper()
    actual_color_filter = str(science_hdr090.get("FILTER", "")).strip().upper()
    if actual_color_filter != expected_color_filter:
        raise RuntimeError(
            f"[PSF] color FITS filter mismatch: expected {expected_color_filter}, "
            f"header has {actual_color_filter}"
        )
    if science_hdr090.get("INSTRUME") != science_hdr.get("INSTRUME"):
        raise RuntimeError(
            f"[PSF] instrument mismatch: signal={science_hdr.get('INSTRUME')} "
            f"color={science_hdr090.get('INSTRUME')}"
        )
    f090_time = _physical_time_from_header(science_hdr090_sci)
    if f090_time is None:
        f090_time = _physical_time_from_header(science_hdr090)
    if f090_time is None:
        raise RuntimeError("[PSF] F090W observation time is absent from FITS headers")
    f090_wss = sorted(
        (
            abs(float((opd_time - f090_time).to_value("day"))),
            float((opd_time - f090_time).to_value("day")),
            opd_path,
        )
        for _, _, opd_path, opd_time in dated_wss
    )
    f090_delta_days, f090_signed_delta_days, nearest_f090 = f090_wss[0]
    if f090_delta_days > PSF_MAX_OPD_DELTA_DAYS:
        raise RuntimeError(
            f"[PSF] no time-matched F090W OPD for {TARGET_GALAXY}: nearest "
            f"{nearest_f090.name} is {f090_delta_days:.2f} d away, limit is "
            f"{PSF_MAX_OPD_DELTA_DAYS:.2f} d"
        )
    f090_opd_hdr = fits.getheader(str(nearest_f090), 0)
    f090_opd_corr_id = str(f090_opd_hdr.get("CORR_ID", nearest_f090.stem))
    f090_opd_path = str(nearest_f090.resolve())
    sim090 = stpsf.instrument(science_hdr090["INSTRUME"])
    if (sim090.name == "NIRCam") and (science_hdr090["PUPIL"][0] == "F") and (science_hdr090["PUPIL"][-1] in ["N", "M"]):
        sim090.filter = science_hdr090["PUPIL"]
    else:
        sim090.filter = science_hdr090["FILTER"]
    sim090.set_position_from_aperture_name(science_hdr090["APERNAME"])
    if (
        (sim090.name == "NIRCam")
        and (science_hdr090.get("PUPIL", "CLEAR") != "CLEAR")
        and (not science_hdr090["PUPIL"].startswith("F"))
        and (not science_hdr090["PUPIL"].startswith("MASK"))
    ):
        sim090.pupil_mask = science_hdr090["PUPIL"]
    sim090.load_wss_opd(_stpsf_ascii_opd_argument(nearest_f090), verbose=False, plot=False)
    sim090.options["output_mode"] = "both"
    psf090_product = sim090.calc_psf(
        nlambda=PSF_NLAMBDA,
        fov_pixels=PSF_SIZE,
        fft_oversample=4,
        detector_oversample=1,
        add_distortion=True,
    )
    color_psf_f090, color_psf_f090_ext = _select_detector_sampled_psf(psf090_product)
    color_psf_f090 = _normalize_psf_array(color_psf_f090, "F090W color PSF")
    if color_psf_f090.shape != color_psf_f150.shape:
        raise RuntimeError(
            f"[PSF] color PSF shape mismatch: F090W={color_psf_f090.shape}, "
            f"F150W={color_psf_f150.shape}"
        )

psf_library_arrays = [entry["array"] for entry in psf_library]
psf = psf_library_arrays[0]
psf_library_table = pd.DataFrame([
    {
        "psf_id": entry["id"],
        "kind": entry["kind"],
        "opd_path": entry.get("opd_path", ""),
        "opd_corr_id": entry.get("opd_corr_id", ""),
        "opd_delta_days": entry.get("opd_delta_days", np.nan),
        "selected_extension": entry.get("selected_extension", ""),
        "filter": entry.get("filter", actual_signal_filter),
        "aperture": entry.get("aperture", science_hdr.get("APERNAME", "")),
        "detector_position": str(entry.get("detector_position", "")),
        "science_pixel_scale_arcsec": science_pixel_scale_arcsec,
        "psf_pixel_scale_arcsec": psf_pixel_scale_arcsec,
        "shape": str(entry["array"].shape),
        "sum": float(entry["array"].sum()),
    }
    for entry in psf_library
])

cache_hdus = [fits.PrimaryHDU()]
cache_hdus[0].header["PSFMETH"] = psf_method_id
cache_hdus[0].header["FILTER"] = actual_signal_filter
cache_hdus[0].header["APERNAME"] = str(science_hdr.get("APERNAME", ""))
cache_hdus[0].header["OPDCORR"] = opd_corr_id
cache_hdus[0].header["OPDDT"] = float(opd_delta_days)
cache_hdus[0].header["SCI_PXS"] = float(science_pixel_scale_arcsec)
cache_hdus[0].header["PSF_PXS"] = float(psf_pixel_scale_arcsec)
for index, entry in enumerate(psf_library):
    hdu = fits.ImageHDU(np.asarray(entry["array"], dtype=np.float32), name=f"PSF{index:02d}")
    hdu.header["PSFID"] = entry["id"][:68]
    hdu.header["PSFKIND"] = entry["kind"]
    cache_hdus.append(hdu)
fits.HDUList(cache_hdus).writeto(psf_library_fits_path, overwrite=True)
psf_library_csv_path = out_dir / f"{stem}_psf_library.csv"
psf_library_table.to_csv(psf_library_csv_path, index=False)
print(f"[PSF] library size={len(psf_library)}; every PSF normalized separately")
print(f"[OUT] PSF library FITS -> {psf_library_fits_path}")
print(f"[OUT] PSF library table -> {psf_library_csv_path}")


[14:52:56] loading/building PSF...
[14:52:56] [PSF] library cache target: data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_psf_129.fits
[14:52:56] [PSF] STPSF_PATH=/Users/zuha/data/stpsf-data
[14:52:56] [PSF] nearest physical OPDs: R2023111002-NRCA3_FP1-1.fits (dt=0.16 d)
[14:52:56] [PSF] science meta: INSTRUME=NIRCAM DETECTOR=MULTIPLE FILTER=F150W PUPIL=CLEAR APERNAME=NRCA1_FULL
[14:52:56] [PSF] science date: 2023-11-09T21:14:47.240
[14:52:56] [PSF] OPD meta: CORR_ID=R2023111002 DATE-OBS=2023-11-10 CONTENTS=NA
[14:52:56] [PSF] configured local sim: instrument=NIRCam filter=F150W detector=NRCA1 apername=NRCA1_FULL det_pos=(1024, 1024)
Importing and format-converting OPD from /Users/zuha/data/stpsf-data/MAST_JWST_WSS_OPDs/R2023111002-NRCA3_FP1-1.fits
Backing out SI WFE and OTE field dependence at the WF sensing field point (NRCA3_FP1)
[14:53:01] [PSF] returned HDU summary:
[14:53:01]     idx=0 ext=OVERSAMP shape=(129, 129) OVERSAMP=4 DET_SAMP=1 PIXELSCL=0.03122585
[14:53:01]     i

In [ ]:
# Jensen-aligned replacement is defined below the preserved legacy implementation.


def _sbf_fft_crop_bounds(region_mask, shape, pad_px):
    """Границы минимального прямоугольника кольца с запасом под PSF."""
    ys, xs = np.where(np.asarray(region_mask, dtype=bool))
    if ys.size == 0:
        return None
    ny_full, nx_full = shape
    pad_px = max(0, int(pad_px))
    return (
        max(0, int(ys.min()) - pad_px),
        min(ny_full, int(ys.max()) + 1 + pad_px),
        max(0, int(xs.min()) - pad_px),
        min(nx_full, int(xs.max()) + 1 + pad_px),
    )


def _build_radial_bin_plan(shape, kbins_n):
    """Один раз сопоставляет каждому Fourier-пикселю радиальный k-бин."""
    ny, nx = shape
    ky = fftfreq(ny)
    kx = fftfreq(nx)
    kr_flat = np.hypot(ky[:, None], kx[None, :]).ravel()
    kbins = np.linspace(0.0, float(kr_flat.max()), int(kbins_n))
    kcenters = 0.5 * (kbins[:-1] + kbins[1:])
    bin_ids = np.searchsorted(kbins, kr_flat, side="right") - 1
    valid_ids = (bin_ids >= 0) & (bin_ids < kcenters.size)
    return {
        "kcenters": kcenters,
        "bin_ids": bin_ids,
        "valid_ids": valid_ids,
        "n_bins": int(kcenters.size),
    }


def _radial_mean_and_error_fast(power2d, radial_plan):
    """Радиальные средние через bincount без 79 проходов по всему кадру."""
    values_all = np.asarray(power2d, dtype=float).ravel()
    select = radial_plan["valid_ids"] & np.isfinite(values_all)
    ids = radial_plan["bin_ids"][select]
    values = values_all[select]
    n_bins = radial_plan["n_bins"]

    counts = np.bincount(ids, minlength=n_bins).astype(int)
    sums = np.bincount(ids, weights=values, minlength=n_bins)
    sums_sq = np.bincount(ids, weights=values * values, minlength=n_bins)
    means = np.full(n_bins, np.nan, dtype=float)
    errors = np.full(n_bins, np.nan, dtype=float)

    enough = counts >= MIN_POINTS_PK_BIN
    means[enough] = sums[enough] / counts[enough]
    variance_ok = enough & (counts > 1)
    sample_variance = np.zeros(n_bins, dtype=float)
    sample_variance[variance_ok] = (
        sums_sq[variance_ok]
        - sums[variance_ok] * sums[variance_ok] / counts[variance_ok]
    ) / (counts[variance_ok] - 1)
    sample_variance = np.maximum(sample_variance, 0.0)
    errors[variance_ok] = np.sqrt(
        sample_variance[variance_ok] / counts[variance_ok]
    )
    return means, errors, counts


def _prepare_sbf_spectral_region(
    region_mask,
    resid,
    model,
    premask,
    psf,
    pix_area,
    fft_workers=None,
    n_e_realizations=None,
    kbins_n=None,
    region_name="region",
    region_role="science",
    region_origin_x=0,
    region_origin_y=0,
):
    """Считает P(k) и библиотеку E(k) один раз для одного кольца."""
    if fft_workers is None:
        fft_workers = FFT_WORKERS
    if n_e_realizations is None:
        n_e_realizations = FFT_E_REALIZATIONS_DIAG
    if kbins_n is None:
        kbins_n = FFT_KBINS_N

    region_mask = np.asarray(region_mask, dtype=bool)
    crop_bounds = _sbf_fft_crop_bounds(
        region_mask,
        np.shape(resid),
        SBF_FFT_CROP_PAD_PX,
    )
    if crop_bounds is None:
        return None
    y0_fft, y1_fft, x0_fft, x1_fft = crop_bounds
    crop_slice = (slice(y0_fft, y1_fft), slice(x0_fft, x1_fft))

    region_crop = region_mask[crop_slice]
    resid_crop = np.asarray(resid[crop_slice], dtype=float)
    model_crop = np.asarray(model[crop_slice], dtype=float)
    premask_crop = np.asarray(premask[crop_slice], dtype=bool)
    window = (
        region_crop & (~premask_crop)
        & np.isfinite(resid_crop) & np.isfinite(model_crop) & (model_crop > 0.0)
    )
    n_use = int(window.sum())
    if n_use < MIN_PIXELS_SBF:
        return None

    Imean = float(np.nanmean(model_crop[window]))
    if not np.isfinite(Imean) or Imean <= 0.0:
        return None

    data_fft = np.zeros_like(resid_crop, dtype=float)
    data_fft[window] = resid_crop[window] / np.sqrt(model_crop[window])
    data_fft[window] -= float(np.nanmean(data_fft[window]))
    with set_workers(fft_workers):
        P2d = (np.abs(fft2(data_fft)) ** 2) / float(n_use)

    ny, nx = data_fft.shape
    radial_plan = _build_radial_bin_plan((ny, nx), kbins_n)
    Pk, Pk_err, Pk_counts = _radial_mean_and_error_fast(P2d, radial_plan)
    kcenters = radial_plan["kcenters"]
    valid_pk = (
        np.isfinite(Pk) & np.isfinite(Pk_err) & (Pk_err > 0.0)
        & np.isfinite(kcenters) & (kcenters > 0.0)
    )
    if int(valid_pk.sum()) < MIN_POINTS_FIT:
        return None

    library = globals().get("psf_library", None)
    if isinstance(library, list) and library:
        psf_entries = library
    else:
        psf_entries = [{"id": "single_input_psf", "kind": "input", "array": psf}]

    print(
        f"[SBF-FFT] {region_name}: full={np.shape(resid)}, crop={(ny, nx)}, "
        f"bounds=(y:{y0_fft}:{y1_fft}, x:{x0_fft}:{x1_fft}), Nuse={n_use}; "
        f"P(k) считается один раз для всех k-окон"
    )

    expectation_profiles = []
    n_realizations = max(1, int(n_e_realizations))
    for psf_index, psf_entry in enumerate(psf_entries):
        psf_array = np.asarray(psf_entry["array"], dtype=float)
        if psf_array.ndim != 2 or not np.all(np.isfinite(psf_array)):
            continue
        psf_sum_local = float(psf_array.sum())
        if psf_sum_local <= 0.0:
            continue
        psf_array = psf_array / psf_sum_local

        big_psf = np.zeros((ny, nx), dtype=float)
        py, px = psf_array.shape
        if py > ny or px > nx:
            continue
        y0_psf = ny // 2 - py // 2
        x0_psf = nx // 2 - px // 2
        big_psf[y0_psf:y0_psf + py, x0_psf:x0_psf + px] = psf_array
        with set_workers(fft_workers):
            F_psf = fft2(big_psf)

        rng = np.random.default_rng(FFT_RNG_SEED + psf_index)
        E2d_sum = np.zeros((ny, nx), dtype=float)
        for _ in range(n_realizations):
            white = rng.normal(0.0, 1.0, size=(ny, nx))
            with set_workers(fft_workers):
                convolved = np.real(np.fft.ifft2(fft2(white) * F_psf))
            simulated = np.zeros_like(convolved, dtype=float)
            simulated[window] = convolved[window]
            simulated[window] -= float(np.nanmean(simulated[window]))
            with set_workers(fft_workers):
                E2d_sum += (np.abs(fft2(simulated)) ** 2) / float(n_use)

        E_profile, _, _ = _radial_mean_and_error_fast(
            E2d_sum / float(n_realizations),
            radial_plan,
        )
        expectation_profiles.append({
            "psf_id": psf_entry.get("id", f"psf_{psf_index}"),
            "E_profile": E_profile,
        })
        print(
            f"[SBF-FFT] {region_name}: E(k) PSF {psf_index + 1}/{len(psf_entries)} "
            f"готов, реализаций={n_realizations}"
        )

    if not expectation_profiles:
        return None

    pr_info = estimate_pr_for_region(
        region_mask=region_mask,
        model=model,
        region_name=region_name,
        region_origin_x=region_origin_x,
        region_origin_y=region_origin_y,
    )
    return {
        "region_name": region_name,
        "region_role": region_role,
        "n_use": n_use,
        "Imean": Imean,
        "shape": (ny, nx),
        "crop_bounds": crop_bounds,
        "kcenters": kcenters,
        "Pk": Pk,
        "Pk_err": Pk_err,
        "Pk_counts": Pk_counts,
        "valid_pk": valid_pk,
        "expectation_profiles": expectation_profiles,
        "pr_info": pr_info,
        "pix_area": pix_area,
    }


def _fit_sbf_prepared_spectrum(prepared, kmin, kmax):
    """Подгоняет P0 и P1 в одном k-окне без повторного FFT и Monte Carlo."""
    if prepared is None:
        return None
    ny, nx = prepared["shape"]
    valid_pk = prepared["valid_pk"]
    kP = prepared["kcenters"][valid_pk]
    P = prepared["Pk"][valid_pk]
    P_error = prepared["Pk_err"][valid_pk]

    jensen_kmin_normalized = 10.0 / float(min(ny, nx))
    effective_kmin = max(float(kmin), jensen_kmin_normalized)
    fit_select = (kP >= effective_kmin) & (kP <= float(kmax))
    if int(fit_select.sum()) < MIN_POINTS_FIT:
        return None
    y = P[fit_select]
    yerr = P_error[fit_select]

    psf_fit_rows = []
    for psf_profile in prepared["expectation_profiles"]:
        E_fit = psf_profile["E_profile"][valid_pk][fit_select]
        good = (
            np.isfinite(E_fit) & (E_fit > 0.0)
            & np.isfinite(y) & np.isfinite(yerr) & (yerr > 0.0)
        )
        if int(good.sum()) < MIN_POINTS_FIT:
            continue

        x = E_fit[good]
        y_local = y[good]
        sigma_local = yerr[good]
        design = np.column_stack([x, np.ones_like(x)])
        weights = 1.0 / np.maximum(sigma_local, ROBUST_SCALE_FLOOR) ** 2
        normal = design.T @ (weights[:, None] * design)
        rhs = design.T @ (weights * y_local)
        try:
            beta = np.linalg.solve(normal, rhs)
        except np.linalg.LinAlgError:
            beta = np.linalg.lstsq(normal, rhs, rcond=None)[0]
        P0_local, P1_local = map(float, beta)
        if not np.isfinite(P0_local) or P0_local <= 0.0:
            continue

        fitted = P0_local * x + P1_local
        residual_fit = y_local - fitted
        dof = max(int(x.size) - 2, 1)
        chi2_red = float(np.sum((residual_fit / sigma_local) ** 2) / dof)
        covariance = np.linalg.pinv(normal) * max(chi2_red, 1.0)
        psf_fit_rows.append({
            "psf_id": psf_profile["psf_id"],
            "P0": P0_local,
            "P1": P1_local,
            "P0_sigma_formal": float(np.sqrt(max(covariance[0, 0], 0.0))),
            "corr": float(np.corrcoef(x, y_local)[0, 1]),
            "fit_resid_std": float(np.nanstd(residual_fit, ddof=1)) if x.size > 2 else np.nan,
            "n_fit": int(x.size),
        })

    if not psf_fit_rows:
        return None
    df_psf_region = pd.DataFrame(psf_fit_rows)
    P0_values = df_psf_region["P0"].to_numpy(dtype=float)
    P0 = float(np.nanmedian(P0_values))
    P1 = float(np.nanmedian(df_psf_region["P1"].to_numpy(dtype=float)))
    P0_formal_sigma = float(np.nanmedian(df_psf_region["P0_sigma_formal"]))
    if P0_values.size > 1:
        P0_psf_sigma = float(
            1.4826 * np.nanmedian(np.abs(P0_values - np.nanmedian(P0_values)))
        )
    else:
        P0_psf_sigma = 0.0
    P0_fit_sigma = float(np.hypot(P0_formal_sigma, P0_psf_sigma))
    corr = float(np.nanmedian(df_psf_region["corr"]))
    fit_resid_std = float(np.nanmedian(df_psf_region["fit_resid_std"]))
    n_fit = int(np.nanmin(df_psf_region["n_fit"]))
    den = P0 + P1
    frac = float(P0 / den) if np.isfinite(den) and den != 0.0 else np.nan

    pr_info = prepared["pr_info"]
    power_info = solve_sbf_power_budget(
        P0=P0,
        Imean=prepared["Imean"],
        Pr=pr_info.get("Pr", np.nan),
        pix_area=prepared["pix_area"],
        P0_fit_sigma=P0_fit_sigma,
    )
    zp_ab_local = sbf_ab_zeropoint_from_pix_area(prepared["pix_area"])
    psf_magnitudes = -2.5 * np.log10(P0_values) + zp_ab_local
    psf_scatter_mag = (
        float(1.4826 * np.nanmedian(np.abs(psf_magnitudes - np.nanmedian(psf_magnitudes))))
        if psf_magnitudes.size > 1 else 0.0
    )
    y0_fft, y1_fft, x0_fft, x1_fft = prepared["crop_bounds"]

    return {
        "measurement_ok": bool(power_info.get("measurement_ok", False)),
        "failure_reason": power_info.get("failure_reason", ""),
        "region_name": prepared["region_name"],
        "region_role": prepared["region_role"],
        "n_use": prepared["n_use"],
        "Imean": prepared["Imean"],
        "P0": P0,
        "P1": P1,
        "P_fluc": power_info.get("P_fluc", np.nan),
        "Pr": pr_info.get("Pr", np.nan),
        "Pr_gc": pr_info.get("Pr_gc", np.nan),
        "Pr_background": pr_info.get("Pr_background", np.nan),
        "Pr_over_P0": power_info.get("Pr_over_P0", np.nan),
        "frac": frac,
        "corr": corr,
        "n_fit": n_fit,
        "fit_resid_std": fit_resid_std,
        "P0_fit_sigma": P0_fit_sigma,
        "P0_formal_sigma": P0_formal_sigma,
        "P0_psf_sigma": P0_psf_sigma,
        "psf_n_used": int(len(df_psf_region)),
        "psf_ids": ";".join(df_psf_region["psf_id"].astype(str)),
        "psf_scatter_mag": psf_scatter_mag,
        "k_mode_min": float(effective_kmin * min(ny, nx)),
        "k_normalized_min": effective_kmin,
        "fft_crop_y0": y0_fft,
        "fft_crop_y1": y1_fft,
        "fft_crop_x0": x0_fft,
        "fft_crop_x1": x1_fft,
        "fft_crop_ny": ny,
        "fft_crop_nx": nx,
        "mbar_fit_sigma_raw": power_info.get("mbar_fit_sigma_raw", np.nan),
        "mbar_fit_sigma": power_info.get("mbar_fit_sigma", np.nan),
        "Pf_spec_raw": power_info.get("Pf_spec_raw", np.nan),
        "Pf_spec": power_info.get("Pf_spec", np.nan),
        "Pf_spec_sigma_formal_raw": power_info.get("Pf_spec_sigma_formal_raw", np.nan),
        "Pf_spec_sigma_formal": power_info.get("Pf_spec_sigma_formal", np.nan),
        "mbar_spec_raw": power_info.get("mbar_spec_raw", np.nan),
        "mbar_spec": power_info.get("mbar_spec", np.nan),
        "n_detected_sources": pr_info.get("n_detected_sources", 0),
        "n_detected_sources_total": pr_info.get("n_detected_sources_total", 0),
        "m_lim": pr_info.get("m_lim", np.nan),
        "m_lim_method": pr_info.get("m_lim_method", "unknown"),
        "lf_model": pr_info.get("lf_model", "gaussian_gclf_plus_background_powerlaw"),
        "lf_gamma": pr_info.get("lf_gamma", np.nan),
        "lf_phi_mlim": pr_info.get("lf_phi_mlim", np.nan),
        "lf_fit_method": pr_info.get("lf_fit_method", "unknown"),
        "lf_fit_rmse": pr_info.get("lf_fit_rmse", np.nan),
        "lf_n_fit_bins": pr_info.get("lf_n_fit_bins", 0),
        "lf_n_fit_sources": pr_info.get("lf_n_fit_sources", 0),
        "lf_fit_mbright": pr_info.get("lf_fit_mbright", np.nan),
        "lf_fit_mfaint": pr_info.get("lf_fit_mfaint", np.nan),
        "gclf_turnover_mag": pr_info.get("gclf_turnover_mag", np.nan),
        "gclf_sigma_mag": pr_info.get("gclf_sigma_mag", GCLF_SIGMA_MAG),
        "pr_note": pr_info.get("pr_note", ""),
    }


def measure_sbf_spectral_windows(
    region_mask,
    resid,
    model,
    premask,
    psf,
    pix_area,
    k_windows,
    fft_workers=None,
    n_e_realizations=None,
    kbins_n=None,
    region_name="region",
    region_role="science",
    region_origin_x=0,
    region_origin_y=0,
):
    """Один FFT/Monte Carlo на кольцо и дешёвая подгонка всех k-окон."""
    prepared = _prepare_sbf_spectral_region(
        region_mask=region_mask,
        resid=resid,
        model=model,
        premask=premask,
        psf=psf,
        pix_area=pix_area,
        fft_workers=fft_workers,
        n_e_realizations=n_e_realizations,
        kbins_n=kbins_n,
        region_name=region_name,
        region_role=region_role,
        region_origin_x=region_origin_x,
        region_origin_y=region_origin_y,
    )
    if prepared is None:
        return [None for _ in k_windows]
    return [
        _fit_sbf_prepared_spectrum(prepared, kmin, kmax)
        for kmin, kmax in k_windows
    ]


def measure_sbf_spectral_region(
    region_mask,
    resid,
    model,
    premask,
    psf,
    pix_area,
    kmin=None,
    kmax=None,
    fft_workers=None,
    n_e_realizations=None,
    kbins_n=None,
    region_name="region",
    region_role="science",
    region_origin_x=0,
    region_origin_y=0,
):
    """Обратная совместимость для одиночного k-окна."""
    if kmin is None or kmax is None:
        kmin, kmax = FFT_K_RANGE_MAIN
    return measure_sbf_spectral_windows(
        region_mask=region_mask,
        resid=resid,
        model=model,
        premask=premask,
        psf=psf,
        pix_area=pix_area,
        k_windows=[(kmin, kmax)],
        fft_workers=fft_workers,
        n_e_realizations=n_e_realizations,
        kbins_n=kbins_n,
        region_name=region_name,
        region_role=region_role,
        region_origin_x=region_origin_x,
        region_origin_y=region_origin_y,
    )[0]


def measure_sbf_spec_for_row(
    row,
    resid,
    model,
    premask,
    psf,
    pix_area,
    kmin=None,
    kmax=None,
    fft_workers=None,
    n_e_realizations=None,
    kbins_n=None,
):
    """
    Считает spectral SBF для одного isophote-window row.

    Важно: raw P0 сам по себе ещё не является stellar SBF signal, потому что
    unresolved contaminants тоже дают PSF-shaped power. Поэтому итогом считаем
    corrected power P_fluc = P0 - Pr.
    """
    ny_r, nx_r = resid.shape
    yy_r, xx_r = np.indices((ny_r, nx_r), dtype=float)

    sma_in = float(row["sma_in"])
    sma_out = float(row["sma_out"])
    x0_ann = float(row["x0"])
    y0_ann = float(row["y0"])
    pa_ann = float(row["pa"])
    q_ann = max(GEOM_Q_FLOOR, float(row["q"]))

    dx = xx_r - x0_ann
    dy = yy_r - y0_ann
    cosp = np.cos(pa_ann)
    sinp = np.sin(pa_ann)
    xp = dx * cosp + dy * sinp
    yp = -dx * sinp + dy * cosp
    r_ell = np.sqrt(xp * xp + (yp / q_ann) * (yp / q_ann))
    annulus_ell = (r_ell >= sma_in) & (r_ell <= sma_out)

    out = measure_sbf_spectral_region(
        region_mask=annulus_ell,
        resid=resid,
        model=model,
        premask=premask,
        psf=psf,
        pix_area=pix_area,
        kmin=kmin,
        kmax=kmax,
        fft_workers=fft_workers,
        n_e_realizations=n_e_realizations,
        kbins_n=kbins_n,
        region_name=f"plateau_{sma_in:.1f}_{sma_out:.1f}",
        region_role="diagnostic",
        region_origin_x=0,
        region_origin_y=0,
    )
    if out is None:
        return None

    out.update({
        "sma_in": sma_in,
        "sma_out": sma_out,
        "sma_mid": float(row["sma_mid"]),
        "i0": int(row["i0"]),
        "i1": int(row["i1"]),
    })
    return out


## SBF в круговых annuli

Определяем маски круговых annuli и запускаем спектральный SBF-фит в двух принятых областях: 8.2-16.4 arcsec и 16.4-32.8 arcsec. Основное science window задаётся `FFT_K_RANGE_MAIN`; остальные окна оставлены только как лёгкая проверка устойчивости.


In [26]:
def build_region_mask_ellipse(shape, x0, y0, q, pa, sma_in, sma_out):
    ny, nx = shape
    yy, xx = np.ogrid[:ny, :nx]

    dx = xx.astype(float) - float(x0)
    dy = yy.astype(float) - float(y0)

    cosp = np.cos(pa)
    sinp = np.sin(pa)

    xp = dx * cosp + dy * sinp
    yp = -dx * sinp + dy * cosp

    r_ell = np.sqrt(xp * xp + (yp / q) * (yp / q))
    return (r_ell >= sma_in) & (r_ell <= sma_out)

def build_region_mask_circle(shape, x0, y0, r_in, r_out):
    ny, nx = shape
    yy, xx = np.ogrid[:ny, :nx]
    rr2 = (xx.astype(float) - float(x0)) ** 2 + (yy.astype(float) - float(y0)) ** 2
    return (rr2 >= float(r_in) ** 2) & (rr2 <= float(r_out) ** 2)


In [27]:
def measure_sbf_for_mask(
    region_mask,
    resid,
    model,
    premask,
    psf,
    pix_area,
    kmin,
    kmax,
    fft_workers=None,
    n_e_realizations=None,
    kbins_n=None,
    region_name="region",
    region_role="science",
    region_origin_x=0,
    region_origin_y=0,
):
    return measure_sbf_spectral_region(
        region_mask=region_mask,
        resid=resid,
        model=model,
        premask=premask,
        psf=psf,
        pix_area=pix_area,
        kmin=kmin,
        kmax=kmax,
        fft_workers=fft_workers,
        n_e_realizations=n_e_realizations,
        kbins_n=kbins_n,
        region_name=region_name,
        region_role=region_role,
        region_origin_x=region_origin_x,
        region_origin_y=region_origin_y,
    )


def measure_sbf_for_mask_windows(
    region_mask,
    resid,
    model,
    premask,
    psf,
    pix_area,
    k_windows,
    fft_workers=None,
    n_e_realizations=None,
    kbins_n=None,
    region_name="region",
    region_role="science",
    region_origin_x=0,
    region_origin_y=0,
):
    return measure_sbf_spectral_windows(
        region_mask=region_mask,
        resid=resid,
        model=model,
        premask=premask,
        psf=psf,
        pix_area=pix_area,
        k_windows=k_windows,
        fft_workers=fft_workers,
        n_e_realizations=n_e_realizations,
        kbins_n=kbins_n,
        region_name=region_name,
        region_role=region_role,
        region_origin_x=region_origin_x,
        region_origin_y=region_origin_y,
    )


In [28]:
print("measuring SBF in Jensen/TRGB-SBF circular annuli...")

if "science_model" not in globals() or "science_resid" not in globals():
    raise RuntimeError("[SBF-REGION] unified science path is unavailable; rerun the science-path cell")

pix_scale = float(np.sqrt(pix_area))
r1_in, r1_out = SBF_LIT_INNER_ARCSEC[0] / pix_scale, SBF_LIT_INNER_ARCSEC[1] / pix_scale
r2_in, r2_out = SBF_LIT_OUTER_ARCSEC[0] / pix_scale, SBF_LIT_OUTER_ARCSEC[1] / pix_scale

print(
    f"[SBF-REGION] unified science path: residual={science_resid_name}, model={science_model_name}; "
    "fixed circular annuli are the science regions"
)
print(
    f"[SBF-REGION] circular center = ({x0_sbf_circ:.2f}, {y0_sbf_circ:.2f}), "
    f"radii = {r1_in:.1f}-{r1_out:.1f} px and {r2_in:.1f}-{r2_out:.1f} px"
)

regions = [
    {
        "region": "circular_inner_lit",
        "role": "science",
        "shape": "circle",
        "mask": build_region_mask_circle(
            science_resid.shape, x0_sbf_circ, y0_sbf_circ, r1_in, r1_out
        ),
        "rin_px": r1_in,
        "rout_px": r1_out,
        "rin_arcsec": SBF_LIT_INNER_ARCSEC[0],
        "rout_arcsec": SBF_LIT_INNER_ARCSEC[1],
        "resid_data": science_resid,
        "model_data": science_model,
        "resid_source": science_resid_name,
        "model_source": science_model_name,
        "origin_x": 0,
        "origin_y": 0,
    },
    {
        "region": "circular_outer_lit",
        "role": "science",
        "shape": "circle",
        "mask": build_region_mask_circle(
            science_resid.shape, x0_sbf_circ, y0_sbf_circ, r2_in, r2_out
        ),
        "rin_px": r2_in,
        "rout_px": r2_out,
        "rin_arcsec": SBF_LIT_OUTER_ARCSEC[0],
        "rout_arcsec": SBF_LIT_OUTER_ARCSEC[1],
        "resid_data": science_resid,
        "model_data": science_model,
        "resid_source": science_resid_name,
        "model_source": science_model_name,
        "origin_x": 0,
        "origin_y": 0,
    },
]

k_windows = list(SBF_REGION_K_WINDOWS)
rows_df = []

for reg in regions:
    region_mask = reg["mask"]
    reg_resid = reg["resid_data"]
    reg_model = reg["model_data"]
    region_pixels = int(region_mask.sum())
    usable_region = region_mask & (~premask) & np.isfinite(reg_resid) & np.isfinite(reg_model) & (reg_model > 0.0)
    usable_pixels = int(usable_region.sum())
    usable_fraction = float(usable_pixels / region_pixels) if region_pixels > 0 else np.nan

    print(
        f"[SBF-REGION] {reg['region']} ({reg['role']}): "
        f"N_region={region_pixels}, N_usable={usable_pixels}, "
        f"usable_fraction={usable_fraction:.3f}, residual={reg['resid_source']}, model={reg['model_source']}"
    )
    src_reg_summary = summarize_sources_in_region_mask(
        region_mask,
        region_origin_x=reg['origin_x'],
        region_origin_y=reg['origin_y'],
    )
    print(
        f"[SRC-CAT-REGION] {reg['region']}: overlap={src_reg_summary['n_overlap']}, "
        f"valid={src_reg_summary['n_valid']}, nonpositive_flux={src_reg_summary['n_nonpositive_flux']}, "
        f"mag_med={src_reg_summary['mag_med']:.3f}"
    )

    outputs_for_windows = measure_sbf_for_mask_windows(
        region_mask=region_mask,
        resid=reg_resid,
        model=reg_model,
        premask=premask,
        psf=psf,
        pix_area=pix_area,
        k_windows=k_windows,
        fft_workers=FFT_WORKERS,
        n_e_realizations=FFT_E_REALIZATIONS_MAIN,
        kbins_n=FFT_KBINS_N,
        region_name=reg["region"],
        region_role=reg["role"],
        region_origin_x=reg["origin_x"],
        region_origin_y=reg["origin_y"],
    )

    for (kmin, kmax), out in zip(k_windows, outputs_for_windows):
        base_row = {
            "region": reg["region"],
            "role": reg["role"],
            "shape": reg["shape"],
            "rin_px": reg["rin_px"],
            "rout_px": reg["rout_px"],
            "rin_arcsec": reg["rin_arcsec"],
            "rout_arcsec": reg["rout_arcsec"],
            "resid_source": reg["resid_source"],
            "model_source": reg["model_source"],
            "region_pixels": region_pixels,
            "usable_pixels": usable_pixels,
            "usable_fraction": usable_fraction,
            "kmin": kmin,
            "kmax": kmax,
        }

        if out is None:
            rows_df.append({**base_row, "status": "failed_raw"})
            print(f"[SBF-REGION]   k={kmin:.3f}..{kmax:.3f} -> raw spectral fit failed")
            continue

        status = "ok" if out.get("measurement_ok", False) else "invalid_corrected"
        rows_df.append({**base_row, "status": status, **out})

        if status == "ok":
            corr_text = f"{out['Pr_over_P0']:.3%}" if np.isfinite(out.get("Pr_over_P0", np.nan)) else "n/a"
            print(
                f"[SBF-REGION]   k={kmin:.3f}..{kmax:.3f}: raw mbar={out['mbar_spec_raw']:.4f}, "
                f"corrected mbar={out['mbar_spec']:.4f}, P0={out['P0']:.3e}, Pr={out['Pr']:.3e}, "
                f"P_fluc={out['P_fluc']:.3e}, Pr/P0={corr_text}, "
                f"Nsrc_valid={out['n_detected_sources']}, Nsrc_overlap={out.get('n_detected_sources_total', 0)}, "
                f"m_lim={out['m_lim']:.3f}, gamma={out['lf_gamma']:.3f}"
            )
        else:
            print(
                f"[SBF-REGION]   k={kmin:.3f}..{kmax:.3f}: raw mbar={out['mbar_spec_raw']:.4f}, "
                f"corrected measurement invalid ({out.get('failure_reason', '')}), "
                f"P0={out['P0']:.3e}, Pr={out['Pr']:.3e}, P_fluc={out['P_fluc']:.3e}"
            )

df_sbf = pd.DataFrame(rows_df)
display(df_sbf)


[14:53:04] measuring SBF in Jensen/TRGB-SBF circular annuli...
[14:53:04] [SBF-REGION] unified science path: residual=resid_full_clip_3p5sigma_catalog_mcut, model=model_full_measured_isophotes; fixed circular annuli are the science regions
[14:53:04] [SBF-REGION] circular center = (6614.14, 1168.76), radii = 262.6-525.2 px and 525.2-1050.4 px
[14:53:04] [SBF-REGION] circular_inner_lit (science): N_region=649907, N_usable=646642, usable_fraction=0.995, residual=resid_full_clip_3p5sigma_catalog_mcut, model=model_full_measured_isophotes
[14:53:04] [SRC-CAT-REGION] circular_inner_lit: overlap=57, valid=57, nonpositive_flux=0, mag_med=24.355
[14:53:04] [SBF-FFT] circular_inner_lit: full=(4377, 9876), crop=(1308, 1309), bounds=(y:515:1823, x:5960:7269), Nuse=646642; P(k) считается один раз для всех k-окон
[14:53:10] [SBF-FFT] circular_inner_lit: E(k) PSF 1/5 готов, реализаций=64
[14:53:15] [SBF-FFT] circular_inner_lit: E(k) PSF 2/5 готов, реализаций=64
[14:53:21] [SBF-FFT] circular_inner_lit

,region,role,shape,rin_px,rout_px,rin_arcsec,rout_arcsec,resid_source,model_source,region_pixels,...,lf_phi_mlim,lf_fit_method,lf_fit_rmse,lf_n_fit_bins,lf_n_fit_sources,lf_fit_mbright,lf_fit_mfaint,gclf_turnover_mag,gclf_sigma_mag,pr_note
0,circular_inner_lit,science,circle,262.597443,525.194886,8.2,16.4,resid_full_clip_3p5sigma_catalog_mcut,model_full_measured_isophotes,649907,...,2.077448e-05,poisson_weighted_gclf_plus_background,0.000010,16,49,21.608226,26.608226,26.890259,1.2,local GCLF + background-galaxy LF
1,circular_inner_lit,science,circle,262.597443,525.194886,8.2,16.4,resid_full_clip_3p5sigma_catalog_mcut,model_full_measured_isophotes,649907,...,2.077448e-05,poisson_weighted_gclf_plus_background,0.000010,16,49,21.608226,26.608226,26.890259,1.2,local GCLF + background-galaxy LF
2,circular_inner_lit,science,circle,262.597443,525.194886,8.2,16.4,resid_full_clip_3p5sigma_catalog_mcut,model_full_measured_isophotes,649907,...,2.077448e-05,poisson_weighted_gclf_plus_background,0.000010,16,49,21.608226,26.608226,26.890259,1.2,local GCLF + background-galaxy LF
3,circular_outer_lit,science,circle,525.194886,1050.389772,16.4,32.8,resid_full_clip_3p5sigma_catalog_mcut,model_full_measured_isophotes,2599673,...,3.799952e-16,poisson_weighted_gclf_plus_background,0.000363,21,2426,21.608226,26.608226,25.048715,1.2,local GCLF + background-galaxy LF
4,circular_outer_lit,science,circle,525.194886,1050.389772,16.4,32.8,resid_full_clip_3p5sigma_catalog_mcut,model_full_measured_isophotes,2599673,...,3.799952e-16,poisson_weighted_gclf_plus_background,0.000363,21,2426,21.608226,26.608226,25.048715,1.2,local GCLF + background-galaxy LF
5,circular_outer_lit,science,circle,525.194886,1050.389772,16.4,32.8,resid_full_clip_3p5sigma_catalog_mcut,model_full_measured_isophotes,2599673,...,3.799952e-16,poisson_weighted_gclf_plus_background,0.000363,21,2426,21.608226,26.608226,25.048715,1.2,local GCLF + background-galaxy LF


## FITS с реально измеряемыми пикселями

Сохраняем два FITS-кадра, где оставлены только пиксели, реально входящие в SBF-измерение inner и outer annuli. Всё вне usable region записывается как `NaN`, чтобы было легко глазами проверить измеряемые данные.


In [29]:
print("saving usable science residual FITS for circular annuli...")

if "science_resid" not in globals() or "science_model" not in globals():
    raise RuntimeError("[SCIENCE-RESID-SAVE] unified science residual/model are unavailable; rerun the science-path cell")
if "premask" not in globals():
    raise RuntimeError("[SCIENCE-RESID-SAVE] premask is unavailable; rerun the premask cell")
if "x0_sbf_circ" not in globals() or "y0_sbf_circ" not in globals() or "pix_area" not in globals():
    raise RuntimeError("[SCIENCE-RESID-SAVE] circular geometry is unavailable; rerun the circular-annulus measurement cell")

pix_scale = float(np.sqrt(pix_area))
region_specs = [
    ("circular_inner_lit", SBF_LIT_INNER_ARCSEC[0] / pix_scale, SBF_LIT_INNER_ARCSEC[1] / pix_scale),
    ("circular_outer_lit", SBF_LIT_OUTER_ARCSEC[0] / pix_scale, SBF_LIT_OUTER_ARCSEC[1] / pix_scale),
]

for reg_name, rin_px, rout_px in region_specs:
    region_mask = build_region_mask_circle(science_resid.shape, x0_sbf_circ, y0_sbf_circ, rin_px, rout_px)
    usable_region = (
        region_mask
        & (~premask)
        & np.isfinite(science_resid)
        & np.isfinite(science_model)
        & (science_model > 0.0)
    )

    resid_use = np.full(science_resid.shape, np.nan, dtype=np.float32)
    resid_use[usable_region] = np.asarray(science_resid[usable_region], dtype=np.float32)

    fits_path = out_dir / f"{stem}_sbf_resid_science_{reg_name}_usable.fits"
    fits.writeto(fits_path, resid_use, hdr150, overwrite=True)

    print(
        f"[SCIENCE-RESID-SAVE] {reg_name}: residual={science_resid_name}, model={science_model_name}, "
        f"rin={rin_px:.1f}px, rout={rout_px:.1f}px, N_region={int(region_mask.sum())}, "
        f"N_usable={int(usable_region.sum())}, saved -> {fits_path}"
    )


[14:55:19] saving usable science residual FITS for circular annuli...
[14:55:19] [SCIENCE-RESID-SAVE] circular_inner_lit: residual=resid_full_clip_3p5sigma_catalog_mcut, model=model_full_measured_isophotes, rin=262.6px, rout=525.2px, N_region=649907, N_usable=646642, saved -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_resid_science_circular_inner_lit_usable.fits
[14:55:19] [SCIENCE-RESID-SAVE] circular_outer_lit: residual=resid_full_clip_3p5sigma_catalog_mcut, model=model_full_measured_isophotes, rin=525.2px, rout=1050.4px, N_region=2599673, N_usable=1937803, saved -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_resid_science_circular_outer_lit_usable.fits


## Raw и corrected SBF

Показываем таблицу по annuli: raw `P0`, коррекцию `P_r`, corrected fluctuation power и соответствующие magnitudes.


In [30]:
compare_columns = [
    "region", "role", "status", "kmin", "kmax",
    "mbar_spec_raw", "mbar_spec",
    "P0", "Pr", "P_fluc", "Pr_over_P0",
    "Imean", "n_detected_sources", "m_lim", "lf_gamma", "lf_fit_method",
]

compare_columns = [col for col in compare_columns if col in df_sbf.columns]
df_sbf_compare = df_sbf[compare_columns].copy()
display(df_sbf_compare.sort_values(["region", "kmin", "kmax"]).reset_index(drop=True))


,region,role,status,kmin,kmax,mbar_spec_raw,mbar_spec,P0,Pr,P_fluc,Pr_over_P0,Imean,n_detected_sources,m_lim,lf_gamma,lf_fit_method
0,circular_inner_lit,science,ok,0.01,0.25,28.085639,28.085669,0.923792,0.000026,0.923766,0.000028,15.035577,57,26.608226,0.100000,poisson_weighted_gclf_plus_background
1,circular_inner_lit,science,ok,0.03,0.25,28.091384,28.091415,0.918917,0.000026,0.918891,0.000028,15.035577,57,26.608226,0.100000,poisson_weighted_gclf_plus_background
2,circular_inner_lit,science,ok,0.04,0.25,28.093990,28.094021,0.916714,0.000026,0.916688,0.000028,15.035577,57,26.608226,0.100000,poisson_weighted_gclf_plus_background
3,circular_outer_lit,science,ok,0.01,0.25,28.131807,28.131830,0.885334,0.000019,0.885315,0.000021,5.804880,2527,26.608226,0.692639,poisson_weighted_gclf_plus_background
4,circular_outer_lit,science,ok,0.03,0.25,28.136164,28.136187,0.881788,0.000019,0.881769,0.000021,5.804880,2527,26.608226,0.692639,poisson_weighted_gclf_plus_background
5,circular_outer_lit,science,ok,0.04,0.25,28.138928,28.138951,0.879546,0.000019,0.879527,0.000021,5.804880,2527,26.608226,0.692639,poisson_weighted_gclf_plus_background


## Итоговый SBF по фиксированным круговым annuli

Объединяем два fixed circular annuli в science-like result. Если оба annuli успешно измерены, предпочитается configured main k-window; итоговая ошибка берётся как максимум из formal weighted error и scatter между inner/outer annuli.

> **⚠️ ОТЛИЧИЕ ОТ JENSEN — НЕПОЛНЫЙ БЮДЖЕТ ОШИБОК.** Взвешенное среднее двух колец соответствует Jensen (2025), но текущая итоговая ошибка не получена повторными прогонами по фону, PSF, модели и пределу маски. Кроме того, в ноутбуке не применяется явная поправка $A_{150}$ за галактическое поглощение. До калибровки надо хранить наблюдаемую и исправленную `mbar_F150W` раздельно.


In [31]:
print("combining fixed circular SBF annuli with weighted mean (main science result)...")

required_annuli = ["circular_inner_lit", "circular_outer_lit"]
df_sbf_ok = df_sbf[df_sbf["status"].eq("ok")].copy()

def _finite_positive(value):
    try:
        value = float(value)
    except (TypeError, ValueError):
        return False
    return np.isfinite(value) and value > 0.0

def _robust_mag_scatter(values):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]

    if arr.size >= 3:
        med = float(np.nanmedian(arr))
        mad_sigma = float(1.4826 * np.nanmedian(np.abs(arr - med)))
        std_sigma = float(np.nanstd(arr, ddof=1))
        if _finite_positive(mad_sigma):
            return mad_sigma, "k-window MAD"
        if _finite_positive(std_sigma):
            return std_sigma, "k-window std"

    if arr.size >= 2:
        std_sigma = float(np.nanstd(arr, ddof=1))
        half_range = float(0.5 * (np.nanmax(arr) - np.nanmin(arr)))
        if _finite_positive(std_sigma):
            return std_sigma, "k-window std"
        if _finite_positive(half_range):
            return half_range, "k-window half-range"

    return np.nan, "no k-window scatter"

region_sigma_info = {}
for region_name, grp in df_sbf_ok.groupby("region"):
    sigma_k, sigma_method = _robust_mag_scatter(grp["mbar_spec"].values)
    region_sigma_info[region_name] = {
        "sigma_kwindow": sigma_k,
        "sigma_method": sigma_method,
    }

def _fit_resid_proxy_mag(row):
    fit_resid_std = row.get("fit_resid_std", np.nan)
    power_ref = row.get("P_fluc", np.nan)
    if not _finite_positive(power_ref):
        power_ref = row.get("P0", np.nan)
    if _finite_positive(fit_resid_std) and _finite_positive(abs(power_ref)):
        return float((2.5 / np.log(10.0)) * fit_resid_std / abs(power_ref))
    return np.nan

def _measurement_sigma_for_row(row):
    candidates = []
    info = region_sigma_info.get(row["region"], {})

    sigma_k = info.get("sigma_kwindow", np.nan)
    if _finite_positive(sigma_k):
        candidates.append((float(sigma_k), info.get("sigma_method", "k-window scatter")))

    sigma_fit = row.get("mbar_fit_sigma", np.nan)
    if _finite_positive(sigma_fit):
        candidates.append((float(sigma_fit), "fit covariance"))

    if candidates:
        sigma, method = max(candidates, key=lambda item: item[0])
        if len(candidates) > 1:
            method = "max(" + ", ".join(m for _, m in candidates) + ")"
        return sigma, method

    sigma_proxy = _fit_resid_proxy_mag(row)
    if _finite_positive(sigma_proxy):
        return sigma_proxy, "fit_resid_std/P_fluc proxy"

    return np.nan, "sigma unavailable"

def _row_for(region_name, kmin_value, kmax_value):
    if df_sbf_ok.empty:
        return None
    match = df_sbf_ok[
        df_sbf_ok["region"].eq(region_name)
        & np.isclose(df_sbf_ok["kmin"].astype(float), float(kmin_value))
        & np.isclose(df_sbf_ok["kmax"].astype(float), float(kmax_value))
    ]
    if match.empty:
        return None
    return match.iloc[0]

def _annulus_measurement(region_name, kmin_value, kmax_value):
    row = _row_for(region_name, kmin_value, kmax_value)
    if row is None:
        return np.nan, np.nan, "not ok", None

    mbar = float(row["mbar_spec"]) if np.isfinite(row.get("mbar_spec", np.nan)) else np.nan
    sigma, sigma_method = _measurement_sigma_for_row(row)
    return mbar, sigma, sigma_method, row

pairs_from_df = df_sbf[df_sbf["region"].isin(required_annuli)][["kmin", "kmax"]].dropna().drop_duplicates()
k_pairs = sorted({(float(r.kmin), float(r.kmax)) for r in pairs_from_df.itertuples(index=False)})

sigma_pipeline = globals().get("sigma_pipeline", np.nan)
summary_rows = []
for kmin_value, kmax_value in k_pairs:
    mbar_inner, sigma_inner, method_inner, row_inner = _annulus_measurement(
        "circular_inner_lit", kmin_value, kmax_value
    )
    mbar_outer, sigma_outer, method_outer, row_outer = _annulus_measurement(
        "circular_outer_lit", kmin_value, kmax_value
    )

    mbar_inner_raw = float(row_inner["mbar_spec_raw"]) if row_inner is not None and np.isfinite(row_inner.get("mbar_spec_raw", np.nan)) else np.nan
    mbar_outer_raw = float(row_outer["mbar_spec_raw"]) if row_outer is not None and np.isfinite(row_outer.get("mbar_spec_raw", np.nan)) else np.nan
    inner_ok = np.isfinite(mbar_inner) and _finite_positive(sigma_inner)
    outer_ok = np.isfinite(mbar_outer) and _finite_positive(sigma_outer)
    both_measured = np.isfinite(mbar_inner) and np.isfinite(mbar_outer)

    mbar_weighted = np.nan
    mbar_weighted_raw = np.nan
    sigma_weighted_formal = np.nan
    annulus_scatter = float(abs(mbar_inner - mbar_outer) / 2.0) if both_measured else np.nan
    sigma_adopted = np.nan

    if inner_ok and outer_ok:
        mvals = np.array([mbar_inner, mbar_outer], dtype=float)
        sigmas = np.array([sigma_inner, sigma_outer], dtype=float)
        weights = 1.0 / (sigmas * sigmas)
        mbar_weighted = float(np.sum(weights * mvals) / np.sum(weights))
        sigma_weighted_formal = float(np.sqrt(1.0 / np.sum(weights)))
        if np.isfinite(mbar_inner_raw) and np.isfinite(mbar_outer_raw):
            mbar_weighted_raw = float(np.sum(weights * np.array([mbar_inner_raw, mbar_outer_raw])) / np.sum(weights))

        if _finite_positive(sigma_pipeline):
            sigma_adopted = float(
                np.sqrt(sigma_weighted_formal**2 + annulus_scatter**2 + float(sigma_pipeline)**2)
            )
            notes = "inner+outer weighted; adopted includes pipeline systematic"
        else:
            sigma_adopted = float(max(sigma_weighted_formal, annulus_scatter))
            notes = "inner+outer weighted; no pipeline systematic yet"

    elif inner_ok or outer_ok:
        if inner_ok:
            mbar_weighted = float(mbar_inner)
            mbar_weighted_raw = float(mbar_inner_raw) if np.isfinite(mbar_inner_raw) else np.nan
            sigma_weighted_formal = float(sigma_inner)
            sigma_adopted = float(sigma_inner)
            notes = "only circular_inner_lit ok; annulus scatter unavailable"
        else:
            mbar_weighted = float(mbar_outer)
            mbar_weighted_raw = float(mbar_outer_raw) if np.isfinite(mbar_outer_raw) else np.nan
            sigma_weighted_formal = float(sigma_outer)
            sigma_adopted = float(sigma_outer)
            notes = "only circular_outer_lit ok; annulus scatter unavailable"
    else:
        notes = "no circular annulus has both corrected mbar and sigma_meas"

    if method_inner != "not ok" or method_outer != "not ok":
        notes += f"; sigma_inner={method_inner}; sigma_outer={method_outer}"

    summary_rows.append({
        "kmin": float(kmin_value),
        "kmax": float(kmax_value),
        "mbar_inner_raw": mbar_inner_raw,
        "mbar_inner": mbar_inner,
        "sigma_inner": sigma_inner,
        "Pr_inner": float(row_inner.get("Pr", np.nan)) if row_inner is not None else np.nan,
        "Pr_over_P0_inner": float(row_inner.get("Pr_over_P0", np.nan)) if row_inner is not None else np.nan,
        "mbar_outer_raw": mbar_outer_raw,
        "mbar_outer": mbar_outer,
        "sigma_outer": sigma_outer,
        "Pr_outer": float(row_outer.get("Pr", np.nan)) if row_outer is not None else np.nan,
        "Pr_over_P0_outer": float(row_outer.get("Pr_over_P0", np.nan)) if row_outer is not None else np.nan,
        "mbar_weighted_raw": mbar_weighted_raw,
        "mbar_weighted": mbar_weighted,
        "sigma_weighted_formal": sigma_weighted_formal,
        "annulus_scatter": annulus_scatter,
        "sigma_adopted": sigma_adopted,
        "notes": notes,
    })

summary_columns = [
    "kmin",
    "kmax",
    "mbar_inner_raw",
    "mbar_inner",
    "sigma_inner",
    "Pr_inner",
    "Pr_over_P0_inner",
    "mbar_outer_raw",
    "mbar_outer",
    "sigma_outer",
    "Pr_outer",
    "Pr_over_P0_outer",
    "mbar_weighted_raw",
    "mbar_weighted",
    "sigma_weighted_formal",
    "annulus_scatter",
    "sigma_adopted",
    "notes",
]
df_annulus_summary = pd.DataFrame(summary_rows, columns=summary_columns)
display(df_annulus_summary)

recommendable = df_annulus_summary[
    np.isfinite(df_annulus_summary["mbar_weighted"])
    & np.isfinite(df_annulus_summary["sigma_adopted"])
].copy()

recommended_sbf = None
if recommendable.empty:
    print("[SBF-ANNULI] no recommended annulus-combined SBF result available")
else:
    main_kmin, main_kmax = FFT_K_RANGE_MAIN
    recommendable["is_main_window"] = (
        np.isclose(recommendable["kmin"].astype(float), float(main_kmin))
        & np.isclose(recommendable["kmax"].astype(float), float(main_kmax))
    )
    recommendable["uses_two_annuli"] = recommendable["notes"].str.contains(r"inner\+outer weighted", regex=True)
    recommendable = recommendable.sort_values(
        ["uses_two_annuli", "is_main_window", "sigma_adopted", "sigma_weighted_formal"],
        ascending=[False, False, True, True],
    )
    best = recommendable.iloc[0]
    recommended_sbf = best.to_dict()

    print(
        "[SBF-SCIENCE] recommended fixed-circular-annuli k-window "
        f"{best['kmin']:.3f}..{best['kmax']:.3f}"
    )
    print(f"[SBF-SCIENCE] weighted raw mbar       = {best['mbar_weighted_raw']:.4f} mag")
    print(f"[SBF-SCIENCE] weighted corrected mbar = {best['mbar_weighted']:.4f} mag")
    print(
        "[SBF-SCIENCE] formal={:.4f} mag, annulus_scatter={:.4f} mag, adopted={:.4f} mag".format(
            best["sigma_weighted_formal"],
            best["annulus_scatter"] if np.isfinite(best["annulus_scatter"]) else np.nan,
            best["sigma_adopted"],
        )
    )
    print(
        f"[SBF-SCIENCE] Pr/P0 inner={best['Pr_over_P0_inner']:.3%}, "
        f"outer={best['Pr_over_P0_outer']:.3%}"
    )
    print("[SBF-SCIENCE] adopted sigma = max(formal, annulus scatter); pipeline systematic is not measured yet")

pipeline_variants = pd.DataFrame([
    {
        "variant": "raw_current_residual",
        "status": "available",
        "mbar_spec": recommended_sbf["mbar_weighted"] if recommended_sbf is not None else np.nan,
        "notes": "science residual for circular annuli = sigma-capped resid_full_clip built from data - model_full; unresolved-source correction included via P_fluc = P0 - Pr",
    },
    {
        "variant": "sigma_clipped_or_capped_residual",
        "status": "not_run",
        "mbar_spec": np.nan,
        "notes": "future branch: rerun SBF after clipping/capping residual outliers",
    },
    {
        "variant": "spline_cleaned_residual",
        "status": "not_run",
        "mbar_spec": np.nan,
        "notes": "future branch: rerun SBF after spline-cleaned large-scale residuals",
    },
])

print("[SBF-PIPELINE] pipeline systematic placeholder:")
print("[SBF-PIPELINE] fill pipeline_variants with independent branch results, then use their mbar scatter as sigma_pipeline")
display(pipeline_variants)


[14:55:19] combining fixed circular SBF annuli with weighted mean (main science result)...


,kmin,kmax,mbar_inner_raw,mbar_inner,sigma_inner,Pr_inner,Pr_over_P0_inner,mbar_outer_raw,mbar_outer,sigma_outer,Pr_outer,Pr_over_P0_outer,mbar_weighted_raw,mbar_weighted,sigma_weighted_formal,annulus_scatter,sigma_adopted,notes
0,0.01,0.25,28.085639,28.085669,0.021376,0.000026,0.000028,28.131807,28.131830,0.018147,0.000019,0.000021,28.112470,28.112496,0.013834,0.023080,0.023080,inner+outer weighted; no pipeline systematic y...
1,0.03,0.25,28.091384,28.091415,0.020877,0.000026,0.000028,28.136164,28.136187,0.018210,0.000019,0.000021,28.116814,28.116841,0.013723,0.022386,0.022386,inner+outer weighted; no pipeline systematic y...
2,0.04,0.25,28.093990,28.094021,0.021286,0.000026,0.000028,28.138928,28.138951,0.018425,0.000019,0.000021,28.119680,28.119707,0.013931,0.022465,0.022465,inner+outer weighted; no pipeline systematic y...


[14:55:19] [SBF-SCIENCE] recommended fixed-circular-annuli k-window 0.040..0.250
[14:55:19] [SBF-SCIENCE] weighted raw mbar       = 28.1197 mag
[14:55:19] [SBF-SCIENCE] weighted corrected mbar = 28.1197 mag
[14:55:19] [SBF-SCIENCE] formal=0.0139 mag, annulus_scatter=0.0225 mag, adopted=0.0225 mag
[14:55:19] [SBF-SCIENCE] Pr/P0 inner=0.003%, outer=0.002%
[14:55:19] [SBF-SCIENCE] adopted sigma = max(formal, annulus scatter); pipeline systematic is not measured yet
[14:55:19] [SBF-PIPELINE] pipeline systematic placeholder:
[14:55:19] [SBF-PIPELINE] fill pipeline_variants with independent branch results, then use their mbar scatter as sigma_pipeline


,variant,status,mbar_spec,notes
0,raw_current_residual,available,28.119707,science residual for circular annuli = sigma-c...
1,sigma_clipped_or_capped_residual,not_run,NaN,future branch: rerun SBF after clipping/cappin...
2,spline_cleaned_residual,not_run,NaN,future branch: rerun SBF after spline-cleaned ...


## Цвета в тех же annuli

Измеряем F090W−F150W в тех же круговых annuli и по той же маске, что использованы для SBF. F090W перепроецируется на WCS F150W, изображения приводятся к общей PSF, после чего цвет получается из отношения интегральных потоков.

> **✅ ИСПРАВЛЕНО В ПОРЯДКЕ JENSEN:** общие WCS, PSF, геометрия и маска; отношение суммарных потоков; наблюдаемый и исправленный за $A_{090}-A_{150}$ цвета хранятся раздельно. При отсутствии коэффициентов поглощения это явно записывается, а не молча считается нулевой поправкой.


In [32]:
print("computing colors in the same fixed circular annuli used for SBF...")

required_annuli = ["circular_inner_lit", "circular_outer_lit"]
region_lookup = {reg["region"]: reg for reg in regions}
color_rows = []

if img_f090 is None:
    print("[COLOR-ANNULI] F090W image unavailable; annulus-matched colors skipped")
else:
    if wcs090 is None or wcs150 is None:
        raise RuntimeError("[COLOR-ANNULI] both F090W and F150W WCS are required")
    if "color_psf_f090" not in globals() or color_psf_f090 is None:
        raise RuntimeError("[COLOR-ANNULI] run the PSF-library cell to build the F090W matching PSF")

    color_union = np.zeros_like(img, dtype=bool)
    for region_name in required_annuli:
        color_union |= np.asarray(region_lookup[region_name]["mask"], dtype=bool)
    y_union, x_union = np.where(color_union)
    if y_union.size == 0:
        raise RuntimeError("[COLOR-ANNULI] color annuli are empty")
    y0_color, y1_color = int(y_union.min()), int(y_union.max()) + 1
    x0_color, x1_color = int(x_union.min()), int(x_union.max()) + 1

    yy_color, xx_color = np.indices((y1_color - y0_color, x1_color - x0_color), dtype=float)
    xx_full_color = xx_color + x0_color
    yy_full_color = yy_color + y0_color
    ra_color, dec_color = wcs150.pixel_to_world_values(xx_full_color, yy_full_color)
    x090_color, y090_color = wcs090.world_to_pixel_values(ra_color, dec_color)
    inside090 = (
        (x090_color >= 0.0) & (x090_color <= img_f090.shape[1] - 1)
        & (y090_color >= 0.0) & (y090_color <= img_f090.shape[0] - 1)
    )

    # Независимый скалярный F090W sky по четырём углам; F150W sky уже вычтен выше.
    corner_h090 = max(1, int(round(BKG_CHECK_CORNER_FRAC * img_f090.shape[0])))
    corner_w090 = max(1, int(round(BKG_CHECK_CORNER_FRAC * img_f090.shape[1])))
    corner_slices090 = (
        (slice(0, corner_h090), slice(0, corner_w090)),
        (slice(0, corner_h090), slice(-corner_w090, None)),
        (slice(-corner_h090, None), slice(0, corner_w090)),
        (slice(-corner_h090, None), slice(-corner_w090, None)),
    )
    sky090_values = []
    for sy, sx in corner_slices090:
        values = img_f090[sy, sx][valid090[sy, sx] & np.isfinite(img_f090[sy, sx])]
        if values.size >= BKG_CHECK_MIN_PIXELS:
            _, corner_median, _ = sigma_clipped_stats(
                values, sigma=SIGMA_STAT, maxiters=SIGMA_MAXIT
            )
            sky090_values.append(float(corner_median))
    if len(sky090_values) < 2:
        raise RuntimeError("[COLOR-ANNULI] insufficient F090W corners for scalar sky")
    color_sky_f090 = float(np.nanmedian(sky090_values))
    img090_sky_sub = np.asarray(img_f090 - color_sky_f090, dtype=float)

    map_coordinates090 = np.vstack([y090_color.ravel(), x090_color.ravel()])
    img090_reprojected = ndimage.map_coordinates(
        img090_sky_sub,
        map_coordinates090,
        order=1,
        mode="constant",
        cval=np.nan,
        prefilter=False,
    ).reshape(xx_color.shape)
    valid090_reprojected = ndimage.map_coordinates(
        valid090.astype(float),
        map_coordinates090,
        order=0,
        mode="constant",
        cval=0.0,
        prefilter=False,
    ).reshape(xx_color.shape) > 0.5
    valid090_reprojected &= inside090 & np.isfinite(img090_reprojected)

    def _psf_core_sigma_pixels(psf_array, core_radius=8.0):
        psf_array = np.asarray(psf_array, dtype=float)
        yy_psf, xx_psf = np.indices(psf_array.shape, dtype=float)
        total = float(np.nansum(psf_array))
        xcen = float(np.nansum(xx_psf * psf_array) / total)
        ycen = float(np.nansum(yy_psf * psf_array) / total)
        rr2 = (xx_psf - xcen) ** 2 + (yy_psf - ycen) ** 2
        core = (rr2 <= core_radius ** 2) & np.isfinite(psf_array) & (psf_array >= 0.0)
        core_flux = float(np.sum(psf_array[core]))
        return float(np.sqrt(np.sum(psf_array[core] * rr2[core]) / (2.0 * core_flux)))

    from photutils.psf.matching import SplitCosineBellWindow, create_matching_kernel

    sigma_psf090 = _psf_core_sigma_pixels(color_psf_f090)
    sigma_psf150 = _psf_core_sigma_pixels(color_psf_f150)
    if sigma_psf090 >= sigma_psf150:
        raise RuntimeError(
            f"[COLOR-ANNULI] F090W PSF is not narrower than F150W: "
            f"sigma090={sigma_psf090:.3f}, sigma150={sigma_psf150:.3f} pix"
        )
    sigma_match = float(np.sqrt(sigma_psf150 ** 2 - sigma_psf090 ** 2))
    matching_window = SplitCosineBellWindow(alpha=0.35, beta=0.30)
    color_matching_kernel = create_matching_kernel(
        color_psf_f090,
        color_psf_f150,
        window=matching_window,
    )
    finite090_float = valid090_reprojected.astype(float)
    numerator090 = fftconvolve(
        np.nan_to_num(img090_reprojected, nan=0.0) * finite090_float,
        color_matching_kernel,
        mode="same",
    )
    denominator090 = fftconvolve(
        finite090_float,
        color_matching_kernel,
        mode="same",
    )
    img090_c = np.full_like(img090_reprojected, np.nan, dtype=float)
    good_denominator = denominator090 > 0.95
    img090_c[good_denominator] = numerator090[good_denominator] / denominator090[good_denominator]
    valid090_c = good_denominator & np.isfinite(img090_c)

    img150_c = np.asarray(img[y0_color:y1_color, x0_color:x1_color], dtype=float)
    valid150_c = valid150[y0_color:y1_color, x0_color:x1_color]
    premask_c = premask[y0_color:y1_color, x0_color:x1_color]
    A_F090W_COLOR = optional_finite_float(globals().get("A_F090W", np.nan))
    A_F150W_COLOR = optional_finite_float(globals().get("A_F150W", np.nan))

    for region_name in required_annuli:
        reg = region_lookup[region_name]
        reg_mask = reg["mask"][y0_color:y1_color, x0_color:x1_color]

        use_color = (
            reg_mask
            & (~premask_c)
            & valid150_c
            & valid090_c
            & np.isfinite(img150_c)
            & np.isfinite(img090_c)
        )

        n_raw = int(use_color.sum())
        row = {
            "region": region_name,
            "rin_arcsec": reg["rin_arcsec"],
            "rout_arcsec": reg["rout_arcsec"],
            "n_raw": n_raw,
            "n_clip": 0,
            "F150_med": np.nan,
            "F090_med": np.nan,
            "F150_sum": np.nan,
            "F090_sum": np.nan,
            "color_F090W_F150W_observed": np.nan,
            "color_F090W_F150W_extinction_corrected": np.nan,
            "color_F090W_F150W": np.nan,
            "A_F090W": A_F090W_COLOR,
            "A_F150W": A_F150W_COLOR,
            "F090W_scalar_sky": color_sky_f090,
            "psf_match_sigma_pix": sigma_match,
            "color_scatter": np.nan,
            "color_sem_proxy": np.nan,
            "notes": "",
        }

        if n_raw <= MIN_COLOR_PIXELS:
            row["notes"] = "too few usable pixels"
            color_rows.append(row)
            continue

        vals150 = img150_c[use_color]
        vals090 = img090_c[use_color]

        _, med150_clip, std150_clip = sigma_clipped_stats(vals150, sigma=COLOR_CLIP_SIGMA, maxiters=COLOR_CLIP_MAXIT)
        _, med090_clip, std090_clip = sigma_clipped_stats(vals090, sigma=COLOR_CLIP_SIGMA, maxiters=COLOR_CLIP_MAXIT)

        keep = (
            (vals150 >= med150_clip - COLOR_CLIP_SIGMA * std150_clip)
            & (vals150 <= med150_clip + COLOR_CLIP_SIGMA * std150_clip)
            & (vals090 >= med090_clip - COLOR_CLIP_SIGMA * std090_clip)
            & (vals090 <= med090_clip + COLOR_CLIP_SIGMA * std090_clip)
        )

        vals150_rob = vals150[keep]
        vals090_rob = vals090[keep]
        n_clip = int(vals150_rob.size)

        row["n_clip"] = n_clip
        if n_clip <= MIN_COLOR_PIXELS:
            row["notes"] = "too few pixels in the common clipped mask"
            color_rows.append(row)
            continue

        f150_med = float(np.nanmedian(vals150_rob))
        f090_med = float(np.nanmedian(vals090_rob))
        f150_sum = float(np.nansum(vals150_rob))
        f090_sum = float(np.nansum(vals090_rob))
        if f150_sum <= 0.0 or f090_sum <= 0.0:
            row["notes"] = "non-positive integrated flux after common masking"
            color_rows.append(row)
            continue
        color_observed = float(-2.5 * np.log10(f090_sum / f150_sum))
        if np.isfinite(A_F090W_COLOR) and np.isfinite(A_F150W_COLOR):
            color_corrected = float(color_observed - (A_F090W_COLOR - A_F150W_COLOR))
            color_adopted = color_corrected
            extinction_note = "Galactic-extinction corrected"
        else:
            color_corrected = np.nan
            color_adopted = color_observed
            extinction_note = "extinction coefficients unavailable; adopted color is observed"

        positive_for_scatter = (vals150_rob > 0.0) & (vals090_rob > 0.0)
        pixel_colors = -2.5 * np.log10(
            vals090_rob[positive_for_scatter] / vals150_rob[positive_for_scatter]
        )
        color_med = float(np.nanmedian(pixel_colors)) if pixel_colors.size else np.nan
        color_mad = (
            float(1.4826 * np.nanmedian(np.abs(pixel_colors - color_med)))
            if pixel_colors.size else np.nan
        )
        color_sem_proxy = (
            float(color_mad / np.sqrt(pixel_colors.size))
            if np.isfinite(color_mad) and pixel_colors.size else np.nan
        )

        row.update({
            "F150_med": f150_med,
            "F090_med": f090_med,
            "F150_sum": f150_sum,
            "F090_sum": f090_sum,
            "color_F090W_F150W_observed": color_observed,
            "color_F090W_F150W_extinction_corrected": color_corrected,
            "color_F090W_F150W": color_adopted,
            "color_scatter": color_mad,
            "color_sem_proxy": color_sem_proxy,
            "notes": (
                "WCS-reprojected; PSF-matched; common mask; integrated flux ratio; "
                + extinction_note
            ),
        })
        color_rows.append(row)

df_color_annuli = pd.DataFrame(color_rows)
display(df_color_annuli)

color_summary_rows = []
if not df_color_annuli.empty:
    ok_color = df_color_annuli[
        df_color_annuli["region"].isin(required_annuli)
        & np.isfinite(df_color_annuli["color_F090W_F150W"])
    ].copy()

    if len(ok_color) == 2 and recommended_sbf is not None:
        sigma_inner = float(recommended_sbf.get("sigma_inner", np.nan))
        sigma_outer = float(recommended_sbf.get("sigma_outer", np.nan))
        if _finite_positive(sigma_inner) and _finite_positive(sigma_outer):
            weights_by_region = {
                "circular_inner_lit": 1.0 / sigma_inner**2,
                "circular_outer_lit": 1.0 / sigma_outer**2,
            }
            weights = ok_color["region"].map(weights_by_region).to_numpy(dtype=float)
            colors = ok_color["color_F090W_F150W"].to_numpy(dtype=float)
            color_weighted = float(np.sum(weights * colors) / np.sum(weights))
            color_annulus_scatter = float(0.5 * (np.nanmax(colors) - np.nanmin(colors)))
            color_summary_rows.append({
                "summary": "SBF-weighted circular-annuli color",
                "color_F090W_F150W": color_weighted,
                "sigma_proxy": color_annulus_scatter,
                "notes": "uses same annulus weights as recommended SBF; sigma_proxy is half inner-outer color difference, not full calibration error",
            })

    if not ok_color.empty:
        color_summary_rows.append({
            "summary": "mean of available circular-annuli colors",
            "color_F090W_F150W": float(np.nanmean(ok_color["color_F090W_F150W"])),
            "sigma_proxy": float(np.nanstd(ok_color["color_F090W_F150W"], ddof=1)) if len(ok_color) > 1 else np.nan,
            "notes": "simple annulus-matched diagnostic color summary",
        })

df_color_summary = pd.DataFrame(color_summary_rows)
display(df_color_summary)

color_annuli_csv_path = out_dir / f"{stem}_sbf_color_annuli_wcs_psf_matched.csv"
color_summary_csv_path = out_dir / f"{stem}_sbf_color_summary_wcs_psf_matched.csv"
df_color_annuli.to_csv(color_annuli_csv_path, index=False)
df_color_summary.to_csv(color_summary_csv_path, index=False)
print(f"[OUT] annulus-matched colors -> {color_annuli_csv_path}")
print(f"[OUT] color summary         -> {color_summary_csv_path}")


[14:55:20] computing colors in the same fixed circular annuli used for SBF...


,region,rin_arcsec,rout_arcsec,n_raw,n_clip,F150_med,F090_med,F150_sum,F090_sum,color_F090W_F150W_observed,color_F090W_F150W_extinction_corrected,color_F090W_F150W,A_F090W,A_F150W,F090W_scalar_sky,psf_match_sigma_pix,color_scatter,color_sem_proxy,notes
0,circular_inner_lit,8.2,16.4,646642,633705,14.077426,8.611120,9.321584e+06,5.689793e+06,0.535983,NaN,0.535983,NaN,NaN,0.203255,0.405581,0.044178,0.000055,WCS-reprojected; PSF-matched; common mask; int...
1,circular_outer_lit,16.4,32.8,2437352,2382353,4.599873,2.895203,1.185558e+07,7.405435e+06,0.510931,NaN,0.510931,NaN,NaN,0.203255,0.405581,0.074916,0.000049,WCS-reprojected; PSF-matched; common mask; int...


,summary,color_F090W_F150W,sigma_proxy,notes
0,SBF-weighted circular-annuli color,0.521661,0.012526,uses same annulus weights as recommended SBF; ...
1,mean of available circular-annuli colors,0.523457,0.017715,simple annulus-matched diagnostic color summary


[14:55:22] [OUT] annulus-matched colors -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_color_annuli_wcs_psf_matched.csv
[14:55:22] [OUT] color summary         -> data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_color_summary_wcs_psf_matched.csv
